# Nota da Propriedade Intelectual

Em conformidade com os direitos de propriedade intelectual e padrões de confidencialidade, esteja ciente de que os arquivos compartilhados com você por e-mail ou qualquer outro meio estão sujeitos à proteção sob as leis de propriedade intelectual. Esses arquivos podem conter informações proprietárias, segredos comerciais ou material protegido por direitos autorais de propriedade exclusiva da Almo Intellect.

Enfatizamos que esses arquivos são para sua referência e utilizados exclusivamente no contexto das tarefas ou responsabilidades que lhe foram atribuídas. Eles não devem ser distribuídos, transmitidos ou compartilhados com quaisquer outros indivíduos ou entidades sem autorização explícita da Almo Intellect.

A sua cooperação na manutenção da confidencialidade das informações contidas nestes ficheiros é crucial para preservar os nossos direitos de propriedade intelectual e manter a confidencialidade dos dados sensíveis.

Caso tenha alguma dúvida ou necessite de maiores esclarecimentos sobre o manuseio desses arquivos, sinta-se à vontade para contactar-nos.

Obrigado pela sua compreensão e estrita adesão a estas medidas de confidencialidade.

In compliance with intellectual property rights and confidentiality standards, please be aware that files shared with you via email or any other means are subject to protection under intellectual property laws. These files may contain proprietary information, trade secrets or copyrighted material exclusively owned by Almo Intellect.

We emphasize that these files are for your reference and used solely in the context of the tasks or responsibilities assigned to you. They must not be distributed, transmitted or shared with any other individuals or entities without explicit permission from Almo Intellect.

Your cooperation in maintaining the confidentiality of the information contained in these files is crucial to preserving our intellectual property rights and maintaining the confidentiality of sensitive data.

If you have any questions or require further clarification regarding the handling of these files, please feel free to contact us.

Thank you for your understanding and strict adherence to these confidentiality measures.


# VISU Traffic Transformer Model (config.yaml disabled)

## Imports

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Google Drive is already mounted at /content/drive")


Mounted at /content/drive


In [ ]:
"""
Traffic Prediction with Transformers
Changelog:
- Structure with clear separation of concerns
- Type hints
- Mixed precision training for improved performance
- Gradient accumulation for effective larger batch sizes
"""

## @title VISU Traffic Transformer Model #TODO: Uncomment

from IPython import get_ipython
from IPython.display import display
# %%
# Install required libraries
# !pip install torch torchvision torchaudio pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric pytz #pmdarima
!pip install torch pandas numpy scikit-learn matplotlib seaborn holidays statsmodels optuna torch_geometric #torchvision torchaudio  #pmdarima
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu118
# Install Azure packages - run this cell first
#!pip install azure-ai-ml azure-identity

import os
import pandas as pd
import numpy as np
import types
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.nn import TransformerEncoder, TransformerEncoderLayer
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yaml
import math
import torch.optim as _optim
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
import pytz
from datetime import datetime
from scipy import stats
import warnings
import gc
import requests

# Optional imports with error handling
try:
    import holidays
except ImportError:
    warnings.warn("holidays package not found, holiday features will be limited")
    holidays = None

# Try to import necessary mixed precision components
try:
    # import torch.cuda.amp
    from torch.cuda.amp import autocast, GradScaler
    AMP_AVAILABLE = True
    # Check if newer API with device_type is available
    try:
        # Try creating a GradScaler with device_type
        test_scaler = GradScaler(device_type='cuda')
        del test_scaler
        DEVICE_TYPE_SUPPORTED = True
    except TypeError:
        # Older PyTorch version without device_type support
        DEVICE_TYPE_SUPPORTED = False
except ImportError:
    warnings.warn("Mixed precision training not available (torch.cuda.amp not available)")
    AMP_AVAILABLE = False
    DEVICE_TYPE_SUPPORTED = False

# Benchmarking Imports
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    STATSMODELS_AVAILABLE = True
except ImportError:
    warnings.warn("statsmodels not available, some benchmarks will be disabled")
    STATSMODELS_AVAILABLE = False

# Optuna for hyperparameter tuning
try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    warnings.warn("optuna not available, hyperparameter optimization will be disabled")
    OPTUNA_AVAILABLE = False

# PyTorch Geometric Imports
try:
    from torch_geometric.nn import GCNConv, GATConv
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    warnings.warn("PyTorch Geometric not available, GNN functionality will be limited")
    TORCH_GEOMETRIC_AVAILABLE = False

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

<ipython-input-2-6d1f60b915b7>:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  test_scaler = GradScaler(device_type='cuda')


## GPU Memory & Performance Utilities

In [ ]:

# =============================================================================
# GPU Memory & Performance Utilities
# =============================================================================

def get_gpu_memory_info():
    """Get GPU memory usage information"""
    if not torch.cuda.is_available():
        return {"error": "CUDA not available"}

    try:
        # Try to use nvidia-smi via subprocess if needed
        # But for simplicity, we'll use PyTorch's built-in functions
        device_count = torch.cuda.device_count()
        gpu_info = {}

        for i in range(device_count):
            total_memory = torch.cuda.get_device_properties(i).total_memory
            reserved_memory = torch.cuda.memory_reserved(i)
            allocated_memory = torch.cuda.memory_allocated(i)
            free_memory = total_memory - reserved_memory

            gpu_info[f"gpu_{i}"] = {
                "total_memory_GB": total_memory / 1e9,
                "reserved_memory_GB": reserved_memory / 1e9,
                "allocated_memory_GB": allocated_memory / 1e9,
                "free_memory_GB": free_memory / 1e9,
                "utilization_pct": (allocated_memory / total_memory) * 100
            }

        return gpu_info

    except Exception as e:
        return {"error": str(e)}

def find_optimal_batch_size(
    model: nn.Module,
    sample_input: torch.Tensor,
    sample_target: torch.Tensor,
    max_batch_size: int = 2048,
    start_batch: int = 32,
    device: str = 'cuda'
) -> int:
    """
    Find the optimal batch size for the model and GPU memory

    Args:
        model: The model to test
        sample_input: A sample input tensor
        sample_target: A sample target tensor
        max_batch_size: Maximum batch size to test
        start_batch: Starting batch size
        device: Device to test on

    Returns:
        Optimal batch size
    """
    if device == 'cpu' or not torch.cuda.is_available():
        return 64  # Default for CPU

    model = model.to(device)
    optimal_batch_size = start_batch

    # Try to clear some memory first
    torch.cuda.empty_cache()
    gc.collect()

    print("Finding optimal batch size for GPU...")
    try:
        # Get a copy of the sample tensors on the correct device
        sample_input = sample_input.to(device)
        sample_target = sample_target.to(device)

        for batch_size in [2**i for i in range(int(np.log2(start_batch)), int(np.log2(max_batch_size))+1)]:
            try:
                # Try to process a batch of this size
                input_batch = sample_input.repeat(batch_size, 1, 1)
                target_batch = sample_target.repeat(batch_size, 1, 1)

                # Forward and backward pass
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)
                else:
                    with autocast():
                        output = model(input_batch)
                        loss = nn.MSELoss()(output, target_batch)

                loss.backward()

                # If we got here without an OOM error, update optimal batch size
                optimal_batch_size = batch_size

                # Clean up to prevent memory accumulation
                del input_batch, target_batch, output, loss
                torch.cuda.empty_cache()

                print(f"  Successfully tested batch size: {batch_size}")

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"  OOM at batch size: {batch_size}")
                    break
                else:
                    raise

    except Exception as e:
        print(f"Error while finding optimal batch size: {e}")
        return 64  # Fallback to default

    finally:
        # Clean up
        torch.cuda.empty_cache()
        gc.collect()

    # Return a slightly smaller batch size to be safe
    return max(int(optimal_batch_size * 0.8), start_batch)

# Simplified GPU monitoring - no longer using contextmanager to avoid import issues
def start_gpu_memory_monitor(config, interval=10):
    """Start GPU memory monitoring - returns a flag to control monitoring"""
    if not config.monitor_gpu_usage or not torch.cuda.is_available():
        return None

    # Import time here to ensure it's available
    import time

    # Flag to control monitoring
    stop_monitoring = [False]

    # Function that will run in a separate thread
    def monitor_memory_usage():
        print("Starting GPU memory monitoring...")
        start_time = time.time()

        while not stop_monitoring[0]:
            try:
                memory_info = get_gpu_memory_info()

                # Print memory usage for each GPU
                for gpu_id, gpu_data in memory_info.items():
                    if isinstance(gpu_data, dict) and 'error' not in gpu_data:
                        print(f"{gpu_id.upper()}: "
                              f"Used {gpu_data['allocated_memory_GB']:.2f}/{gpu_data['total_memory_GB']:.2f} GB "
                              f"({gpu_data['utilization_pct']:.1f}%)")

                # Also monitor CPU memory if psutil is available
                try:
                    import psutil
                    process = psutil.Process(os.getpid())
                    print(f"CPU Memory: {process.memory_info().rss / 1e9:.2f} GB")
                except ImportError:
                    pass

                time.sleep(interval)
            except Exception as e:
                print(f"Error in GPU monitoring: {e}")
                time.sleep(interval)

    try:
        # Start monitoring in a separate thread
        import threading
        monitor_thread = threading.Thread(target=monitor_memory_usage)
        monitor_thread.daemon = True  # Daemon thread will exit when main thread exits
        monitor_thread.start()

        return stop_monitoring
    except Exception as e:
        print(f"Failed to start GPU monitoring: {e}")
        return None

def stop_gpu_memory_monitor(stop_flag):
    """Stop GPU memory monitoring by setting the stop flag"""
    if stop_flag is not None:
        stop_flag[0] = True

def transfer_compatible_weights(source_model, target_model):
    """Transfer weights for layers with matching shapes between models"""
    with torch.no_grad():
        # Skip embedding and decoder layers as they have different dimensions

        # 1. Positional encoding - copy if shapes match
        if hasattr(source_model.pos_encoder, 'pe') and hasattr(target_model.pos_encoder, 'pe'):
            if source_model.pos_encoder.pe.shape == target_model.pos_encoder.pe.shape:
                target_model.pos_encoder.pe.copy_(source_model.pos_encoder.pe)
                print("  Transferred positional encoding weights")

        # 2. Transformer layers - copy layers with matching dimensions
        for i, (source_layer, target_layer) in enumerate(zip(source_model.transformer, target_model.transformer)):
            # This is a nested function to handle potential shape mismatches
            def safe_copy_if_shapes_match(target_param, source_param):
                if target_param.shape == source_param.shape:
                    target_param.copy_(source_param)
                    return True
                return False

            # Try to copy self-attention weights
            attn_keys = [
                'self_attn.in_proj_weight', 'self_attn.in_proj_bias',
                'self_attn.out_proj.weight', 'self_attn.out_proj.bias'
            ]

            # Try to copy FFN weights
            ffn_keys = [
                'linear1.weight', 'linear1.bias',
                'linear2.weight', 'linear2.bias'
            ]

            # Try to copy layer norm weights
            norm_keys = [
                'norm1.weight', 'norm1.bias',
                'norm2.weight', 'norm2.bias'
            ]

            # Attempt to copy parameters that match in shape
            copied_params = 0
            total_params = 0

            for attr_list in [attn_keys, ffn_keys, norm_keys]:
                for attr in attr_list:
                    total_params += 1
                    # Navigate the nested attributes
                    parts = attr.split('.')
                    source_param = source_layer
                    target_param = target_layer

                    try:
                        for part in parts:
                            source_param = getattr(source_param, part)
                            target_param = getattr(target_param, part)

                        if safe_copy_if_shapes_match(target_param, source_param):
                            copied_params += 1
                    except AttributeError:
                        # Skip if attribute doesn't exist
                        pass

            print(f"  Transferred {copied_params}/{total_params} parameter tensors for transformer layer {i+1}")

## Config Module

In [ ]:
# =============================================================================
# Config Module
# =============================================================================

@dataclass
class TrainingConfig:
    """Configuration class for training parameters with enhanced validation and transfer learning support"""
    base_output_dir: str
    batch_size: int = 16
    seq_length: int = 12
    pred_length: int = 1
    num_epochs: int = 1000
    patience: int = 25
    learning_rate: float = 0.0001 #0.001
    hidden_dim: int = 704
    num_layers: int = 2
    num_heads: int = 8
    dropout: float = 0.1
    ff_dim_multiplier: int = 4
    activation: str = 'relu'
    data_scaler_type: str = 'minmax'
    optimizer_type: str = 'adamw'
    loss_function: str = 'mse'
    use_time_features: bool = True
    use_holiday_feature: bool = False
    holiday_country_code: str = 'US'
    use_weather_feature: bool = True
    weather_feature_type: str = 'all_features'
    weather_data_file: str = 'paste.txt'
    gradient_clip: Optional[float] = 1.0
    scheduler_type: Optional[str] = 'plateau'
    scheduler_patience: int = 25
    scheduler_factor: float = 0.5
    step_scheduler_step_size: int = 10
    step_scheduler_gamma: float = 0.1
    use_lagged_features: bool = False
    num_lags: int = 24
    decoder_type: str = 'mlp'
    use_spatial_features: bool = True
    spatial_feature_dim: int = 704
    use_gnn_pre_transformer: bool = True
    gnn_type: str = 'gcn'
    use_quantile_regression: bool = False
    quantiles: List[float] = None
    optuna_trials: Optional[int] = 0
    warmup_epochs: int = 10
    use_mixed_precision: bool = True
    accumulation_steps: int = 16
    num_workers: int = 2
    pin_memory: bool = True
    find_optimal_batch_size: bool = True
    monitor_gpu_usage: bool = True

    # Transfer Learning Support Fields - Added to main config but optional
    enable_transfer_learning: bool = True
    run_only_transfer_learning: bool = True
    source_model_path: Optional[str] = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14/Transformers_Output_20250421_111818/Models_20250421_111818/best_model_20250421_132601.pth'
    # source_model_path: Optional[str] = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14/Transformers_Output_20250502_205903 [Best Run]/Models_20250502_205903/best_model_20250502_210140.pth'
    # source_model_path: Optional[str] = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V14/Transformers_Output_20250502_174139/Models_20250502_174139/best_model_20250502_174249.pth'
    # target_dataset_name: Optional[str] = 'clean_616_traffic station_data.csv'
    target_dataset_name: Optional[str] = 'METR-CPT-Rolling-Mean-Expanded-v2-NoAugmentation.csv'
    # target_data_path: Optional[str] = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/clean_616_traffic station_data.csv'
    target_data_path: Optional[str] = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/METR-CPT-Rolling-Mean-Expanded-v2-NoAugmentation.csv'
    freeze_encoder: bool = True
    freeze_layers: int = 2
    adapter_dim: int = 256
    transfer_learning_rate: float = 1e-5

    # Directories (to be set after initialization)
    input_dir: Optional[str] = None
    output_dir: Optional[str] = None
    model_dir: Optional[str] = None
    results_dir: Optional[str] = None

    def __post_init__(self):
        if self.quantiles is None:
            self.quantiles = [0.1, 0.5, 0.9]

        # Enhanced check for divisibility with better warning messages
        if self.num_heads > 0:
            if self.hidden_dim % self.num_heads != 0:
                original_hidden_dim = self.hidden_dim
                # Adjust hidden_dim to be divisible by num_heads
                self.hidden_dim = (self.hidden_dim // self.num_heads) * self.num_heads

                # If adjustment resulted in zero, set it to num_heads
                if self.hidden_dim == 0:
                    self.hidden_dim = self.num_heads

                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure divisibility by num_heads={self.num_heads}")

            # Also ensure hidden_dim is even for positional encoding
            if self.hidden_dim % 2 != 0:
                original_hidden_dim = self.hidden_dim
                self.hidden_dim += 1  # Make it even by adding 1
                warnings.warn(f"Adjusted hidden_dim from {original_hidden_dim} to {self.hidden_dim} to ensure it's even for positional encoding")

        # Check for mixed precision availability
        if self.use_mixed_precision and not AMP_AVAILABLE:
            self.use_mixed_precision = False
            warnings.warn("Mixed precision requested but not available, disabled")

        # Check for GNN availability
        if self.use_gnn_pre_transformer and not TORCH_GEOMETRIC_AVAILABLE:
            self.use_gnn_pre_transformer = False
            warnings.warn("GNN pre-transformer requested but PyTorch Geometric not available, disabled")

        # Adjust workers based on system capabilities
        import multiprocessing
        max_workers = multiprocessing.cpu_count()
        if self.num_workers > max_workers:
            warnings.warn(f"Reducing num_workers from {self.num_workers} to {max_workers} based on system capabilities")
            self.num_workers = max_workers

        # Validate weather feature type
        valid_weather_types = ['all_features', 'temperature', 'weather_condition_code',
                               'visibility', 'wind_speed', 'wind_direction_code',
                               'wind', 'humidity', 'dew_point', 'cloud_cover_code']
        if self.weather_feature_type not in valid_weather_types:
            warnings.warn(f"Invalid weather_feature_type: {self.weather_feature_type}, using 'all_features'")
            self.weather_feature_type = 'all_features'

        # Validate transfer learning settings if enabled
        if self.enable_transfer_learning:
            if self.target_dataset_name is None:
                warnings.warn("Transfer learning enabled but target_dataset_name not set. Using 'mozambique'")
                self.target_dataset_name = 'mozambique'

            if self.target_data_path is None:
                self.target_data_path = f"{self.target_dataset_name}_traffic.csv"
                warnings.warn(f"Transfer learning enabled but target_data_path not set. Using '{self.target_data_path}'")


def ensure_compatible_dimensions(model_params, config):
    """
    Ensures that hidden_dim is divisible by num_heads and is even
    for compatibility with the MultiheadAttention module

    Args:
        model_params: Dictionary of model parameters
        config: Training configuration object

    Returns:
        Updated model_params and config
    """
    # Check if hidden_dim is divisible by num_heads
    if model_params['hidden_dim'] % model_params['num_heads'] != 0:
        # Adjust to nearest value divisible by num_heads
        adjusted_hidden_dim = (model_params['hidden_dim'] // model_params['num_heads']) * model_params['num_heads']
        # Ensure it's not zero
        if adjusted_hidden_dim == 0:
            adjusted_hidden_dim = model_params['num_heads']

        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure divisibility by num_heads={model_params['num_heads']}")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    # Also ensure hidden_dim is even for positional encoding
    if model_params['hidden_dim'] % 2 != 0:
        adjusted_hidden_dim = model_params['hidden_dim'] + 1
        print(f"Warning: Adjusting hidden_dim from {model_params['hidden_dim']} to {adjusted_hidden_dim} to ensure it's even for positional encoding")
        model_params['hidden_dim'] = adjusted_hidden_dim
        config.hidden_dim = adjusted_hidden_dim

    return model_params, config


def load_config(config_path: str = 'config.yaml') -> TrainingConfig:
    """Load configuration from YAML file and create TrainingConfig object"""
    try:
        with open(config_path, 'r') as f:
            config_dict = yaml.safe_load(f)
        return TrainingConfig(**config_dict)
    except (FileNotFoundError, yaml.YAMLError) as e:
        warnings.warn(f"Error loading config from {config_path}: {e}. Using default configuration.")
        return TrainingConfig(base_output_dir='/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15')


def setup_directories(config: TrainingConfig) -> Tuple[str, str, str, str]:
    """Sets up input, output, model, and results directories with timestamped folders."""
    timestamp = get_maputo_timestamp()

    # Use local directories instead of Google Drive paths
    output_dir = os.path.join(config.base_output_dir, f"Transformers_Output_{timestamp}")
    input_dir = os.path.join(config.base_output_dir, f"Transformers_Input")
    model_dir = os.path.join(output_dir, f"Models_{timestamp}")
    results_dir = os.path.join(output_dir, f"Results_{timestamp}")

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # Update config with directory paths
    config.input_dir = input_dir
    config.output_dir = output_dir
    config.model_dir = model_dir
    config.results_dir = results_dir

    # Check for weather data file and copy if needed
    weather_src = config.weather_data_file  # Source path
    weather_dest = os.path.join(input_dir, os.path.basename(config.weather_data_file))

    if os.path.exists(weather_src) and not os.path.exists(weather_dest):
        try:
            import shutil
            shutil.copyfile(weather_src, weather_dest)
            print(f"Weather data copied to {weather_dest}")
        except Exception as e:
            warnings.warn(f"Could not copy weather data file: {e}")

    # If transfer learning is enabled, check for target dataset
    if config.enable_transfer_learning and config.target_data_path:
        target_src = config.target_data_path
        target_dest = os.path.join(input_dir, os.path.basename(config.target_data_path))

        if os.path.exists(target_src) and not os.path.exists(target_dest):
            try:
                import shutil
                shutil.copyfile(target_src, target_dest)
                print(f"Target dataset copied to {target_dest}")
                # Update path to point to the copied file
                config.target_data_path = os.path.basename(config.target_data_path)
            except Exception as e:
                warnings.warn(f"Could not copy target dataset file: {e}")

    return input_dir, output_dir, model_dir, results_dir

def get_maputo_timestamp() -> str:
    """Returns current timestamp in Maputo timezone."""
    maputo_tz = pytz.timezone('Africa/Maputo')
    return datetime.now(maputo_tz).strftime("%Y%m%d_%H%M%S")


def apply_transfer_config(main_config: TrainingConfig, transfer_config) -> TrainingConfig:
    """
    Apply transfer learning configuration to the main config

    Args:
        main_config: Main TrainingConfig object
        transfer_config: TransferLearningConfig object

    Returns:
        Updated TrainingConfig with transfer learning settings
    """
    # Enable transfer learning
    main_config.enable_transfer_learning = True

    # Copy basic transfer settings
    main_config.source_model_path = transfer_config.pretrained_model_path
    main_config.target_dataset_name = transfer_config.target_dataset_name
    main_config.target_data_path = transfer_config.target_data_path
    main_config.freeze_encoder = transfer_config.freeze_encoder
    main_config.freeze_layers = transfer_config.freeze_layers

    # Apply adapter settings if enabled
    if transfer_config.use_adapters:
        main_config.adapter_dim = transfer_config.adapter_dim
    else:
        main_config.adapter_dim = 0

    # Use the transfer-specific learning rate
    main_config.transfer_learning_rate = transfer_config.learning_rate

    # Adjust training parameters for fine-tuning
    main_config.num_epochs = min(main_config.num_epochs, transfer_config.num_epochs)
    main_config.patience = min(main_config.patience, transfer_config.patience)
    main_config.batch_size = min(main_config.batch_size, transfer_config.batch_size)
    main_config.gradient_clip = transfer_config.gradient_clip

    print(f"Applied transfer learning configuration for {main_config.target_dataset_name} dataset")
    return main_config

## Transfer Learning Config Module

In [ ]:
# =============================================================================
# Transfer Learning Configuration Module
# =============================================================================

class TransferLearningConfig:
    """Configuration class for transfer learning parameters"""

    def __init__(self):
        # Source model (pre-trained) settings
        self.pretrained_model_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V16/Transformers_Output_20250514_212434/Models_20250514_212434/best_model_20250514_224227.pth'  # Path to pre-trained model
        self.source_dataset_name = 'PEMS-Bay'                  # Name of source dataset

        # Target dataset settings
        self.target_dataset_name = 'mozambique'               # 'mozambique' or 'south_africa'
        # self.target_data_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/clean_616_traffic station_data.csv'                       # Path to target dataset
        self.target_data_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/METR-CPT-Rolling-Mean-Expanded-v2-NoAugmentation.csv'                       # Path to target dataset
        # self.target_data_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/METR-MPT-Parallel-Sensors-Fill-Zero-v2-Extented.csv'      # Path to target dataset
        #self.target_data_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/METR-CPT-Mean-Zero-Padding.csv'      # Path to target dataset
        self.test_split = 0.2                                 # Split ratio for test set

        # Transfer learning strategy
        self.freeze_encoder = True                            # Whether to freeze encoder layers
        self.freeze_layers = 3                                # Number of transformer layers to freeze
        self.adapter_dim = 256                                 # Dimension for adapter layers (0 to disable)
        self.use_adapters = True                              # Whether to use adapter layers

        # Fine-tuning hyperparameters
        self.learning_rate = 1e-5                             # Learning rate for fine-tuning (usually smaller than initial training)
        self.num_epochs = 100                                  # Maximum number of epochs for fine-tuning
        self.patience = 10                                     # Early stopping patience
        self.batch_size = 16                                  # Batch size for fine-tuning
        self.gradient_clip = 0.05                              # Gradient clipping value

        # Scheduler settings
        self.scheduler_type = 'cosine'                       # 'plateau', 'cosine', 'step', or None
        self.scheduler_patience = 10                           # Patience for ReduceLROnPlateau
        self.scheduler_factor = 0.6                           # Factor for ReduceLROnPlateau

        # Evaluation settings
        self.evaluate_on_source = True                        # Whether to evaluate on source dataset after fine-tuning
        self.save_comparison_report = True                    # Whether to save a comparison report

        # Visualization settings
        self.visualize_attention = True                       # Whether to visualize attention patterns
        self.visualize_training_curve = True                  # Whether to visualize training curve
        self.visualize_metrics_comparison = True              # Whether to visualize metrics comparison

        # Output settings
        self.save_fine_tuned_model = True                     # Whether to save the fine-tuned model
        self.fine_tuned_model_prefix = 'fine_tuned'           # Prefix for fine-tuned model filename

    def update(self, **kwargs):
        """Update config attributes from keyword arguments"""
        for key, value in kwargs.items():
            if hasattr(self, key):
                setattr(self, key, value)
            else:
                print(f"Warning: TransferLearningConfig has no attribute '{key}'")
        return self

    def print_summary(self):
        """Print a summary of the transfer learning configuration"""
        print("\n==== Transfer Learning Configuration ====")
        print(f"Target Dataset: {self.target_dataset_name}")
        print(f"Pre-trained Model: {self.pretrained_model_path}")
        print(f"Strategy: {'Partial fine-tuning' if self.freeze_encoder else 'Full fine-tuning'}")
        if self.freeze_encoder:
            print(f"  - Freezing {self.freeze_layers} transformer layers")
        print(f"  - Adapters: {'Enabled' if self.use_adapters else 'Disabled'}")
        if self.use_adapters:
            print(f"    - Adapter dimension: {self.adapter_dim}")
        print(f"Learning Rate: {self.learning_rate}")
        print(f"Batch Size: {self.batch_size}")
        print(f"Max Epochs: {self.num_epochs}")
        print("========================================\n")

    def to_dict(self):
        """Convert configuration to dictionary"""
        return {k: v for k, v in self.__dict__.items()}


# Create transfer learning configuration instance
transfer_config = TransferLearningConfig()

# Example of how to modify settings
# transfer_config.update(
#     target_dataset_name='south_africa',
#     target_data_path='south_africa_traffic.csv',
#     freeze_layers=2,
#     learning_rate=1e-5
# )

# Print configuration summary
transfer_config.print_summary()

# To apply these settings to main config, use in your workflow:
# config = apply_transfer_config(config, transfer_config)


==== Transfer Learning Configuration ====
Target Dataset: mozambique
Pre-trained Model: /content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V16/Transformers_Output_20250514_212434/Models_20250514_212434/best_model_20250514_224227.pth
Strategy: Partial fine-tuning
  - Freezing 3 transformer layers
  - Adapters: Enabled
    - Adapter dimension: 256
Learning Rate: 1e-05
Batch Size: 16
Max Epochs: 100



## Logging Module

In [ ]:
import sys
import os
from datetime import datetime
from dataclasses import asdict
from enum import Enum
import pytz

# Define the timestamp function
def generate_timestamp():
    """Returns current timestamp in Maputo timezone."""
    maputo_tz = pytz.timezone('Africa/Maputo')
    return datetime.now(maputo_tz).strftime("%Y%m%d_%H%M%S")

# Create timestamp and log filename
timestamp = generate_timestamp()
log_filename = f"training_log_{timestamp}.txt"

# Specify your output directory directly
base_output_dir = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction'
log_path = os.path.join(f'{base_output_dir}/Transformer_Versions/COLAB_NOBASELINE_V15', log_filename)

# Make sure the directory exists
os.makedirs(base_output_dir, exist_ok=True)

def save_predictions_and_actuals(
    predictions: np.ndarray,
    actuals: np.ndarray,
    results_dir: str,
    filename: str = "predictions_and_actuals",
    fold: Optional[int] = None
) -> None:
    """
    Save model predictions and actual values to CSV files for later analysis.

    Args:
        predictions: numpy array of model predictions
        actuals: numpy array of actual values
        results_dir: Directory to save results
        filename: Directory to save results
        fold: Optional fold number for cross-validation experiments
    """
    # Create timestamp for unique filenames
    timestamp = generate_timestamp()

    # Create fold suffix if fold is provided
    fold_suffix = f'_fold_{fold}' if fold is not None else ''

    # Convert arrays to DataFrames
    df_predictions = pd.DataFrame(predictions.ravel(), columns=['predictions'])
    df_actuals = pd.DataFrame(actuals.ravel(), columns=['actuals'])

    # Combine predictions and actuals into one DataFrame
    df_combined = pd.concat([df_predictions, df_actuals], axis=1)

    # Define CSV path
    csv_path = os.path.join(results_dir, f'{filename}{fold_suffix}_{timestamp}.csv')

    # Save to CSV
    df_combined.to_csv(csv_path, index=False)

    print(f"Predictions and actuals saved to {csv_path}")


def save_experiment_results(
    config: TrainingConfig,
    model_params: Dict,
    metrics: Tuple[float, float, float, float],
    results_dir: str,
    fold: Optional[int] = None
) -> None:
    """
    Save experiment configuration, model parameters and metrics to a CSV file.
    Each attribute becomes its own column for better analysis.

    Args:
        config: Training configuration object
        model_params: Dictionary of model parameters
        metrics: Tuple of (mae, rmse, r2, mape)
        results_dir: Directory to save results
        fold: Optional fold number for cross-validation experiments
    """

    # Extract metrics
    mae, rmse, r2, mape = metrics

    # Create base dictionary with metrics and basic info
    row_data = {
        'timestamp': datetime.now().strftime('%Y-%m-%d_%H-%M-%S'),
        'fold': fold if fold is not None else 'NA',
        'mae': float(mae),
        'rmse': float(rmse),
        'r2': float(r2),
        'mape': float(mape),
    }

    # Add all config attributes with 'config_' prefix
    config_dict = asdict(config)
    for key, value in config_dict.items():
        # Handle Enum values
        if isinstance(value, Enum):
            value = value.value
        row_data[f'config_{key}'] = value

    # Add all model parameters with 'model_' prefix
    for key, value in model_params.items():
        row_data[f'model_{key}'] = value

    # Convert to DataFrame
    df_row = pd.DataFrame([row_data])

    # Define CSV path
    csv_path = os.path.join(results_dir, f'experiment_results{generate_timestamp()}.csv')

    # If file exists, append; if not, create new
    if os.path.exists(csv_path):
        df_row.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df_row.to_csv(csv_path, index=False)

    print(f"Experiment results appended to {csv_path}")

# Create a logger that writes to both console and file
class TeeLogger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, 'w')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()  # Ensure immediate writing

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Redirect stdout to our custom logger
sys.stdout = TeeLogger(log_path)

print(f"Log file created at: {log_path}")
print(f"All console output will now be saved to this file")

Log file created at: /content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/training_log_20250515_105321.txt
All console output will now be saved to this file


## Data Module

In [ ]:
# =============================================================================
# Data Module
# =============================================================================

class TrafficDataset(Dataset):
    """Enhanced Traffic Dataset with proper typing and improved feature handling"""

    def __init__(
        self,
        data: np.ndarray,
        timestamps: Optional[pd.DatetimeIndex] = None,
        sequence_length: int = 24,
        prediction_window: int = 1,
        config: Optional[TrainingConfig] = None
    ):
        """
        Traffic Dataset with configurable features including lagged, spatial, and weather.

        Args:
            data: Sensor data array
            timestamps: Timestamps for time-based features
            sequence_length: Input sequence length
            prediction_window: Prediction window
            config: Configuration object with feature flags
        """
        self.data = data
        self.seq_length = sequence_length
        self.pred_window = prediction_window
        self.timestamps = timestamps
        self.config = config or TrainingConfig(base_output_dir="./output")

        self.features_list = []

        if self.config.use_time_features and timestamps is not None:
            self.create_time_features(timestamps)
            self.features_list.extend(['hour', 'dayofweek', 'weekofyear', 'month'])

        if self.config.use_holiday_feature and timestamps is not None:
            self.create_holiday_feature(timestamps)
            self.features_list.append('is_holiday')

        if self.config.use_weather_feature and timestamps is not None:
            self.create_weather_feature(timestamps)
            self.features_list.append('weather_condition')

        if self.config.use_lagged_features:
            self.create_lagged_features()
            self.features_list.extend([f'lag_{i+1}' for i in range(self.config.num_lags)])

        if self.config.use_spatial_features:
            self.create_spatial_features()
            self.features_list.extend([f'spatial_feature_{i+1}' for i in range(self.config.spatial_feature_dim)])


    def __len__(self):
        return len(self.data) - self.seq_length - self.pred_window + 1


    def create_time_features(self, timestamps: pd.DatetimeIndex) -> None:
        """Create normalized time-based features"""
        times = pd.to_datetime(timestamps)
        self.time_features = np.stack([
            times.hour.values / 23.0,
            times.dayofweek.values / 6.0,
            (times.dayofyear // 7).values / 51.0,
            times.month.values / 11.0
        ], axis=1)

    def create_holiday_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Create holiday indicator feature"""
        if holidays is None:
            # Create dummy holiday feature if holidays package is not available
            self.holiday_feature = np.zeros((len(timestamps), 1))
            warnings.warn("holidays package not available, using dummy holiday feature")
            return

        dates = pd.to_datetime(timestamps).date
        country_code = self.config.holiday_country_code
        try:
            country_holidays = holidays.CountryHoliday(country_code, years=set(d.year for d in dates))
        except KeyError:
            warnings.warn(f"Country code '{country_code}' not recognized. Using US holidays instead.")
            country_holidays = holidays.CountryHoliday('US', years=set(d.year for d in dates))

        self.holiday_feature = np.array([(1 if d in country_holidays else 0) for d in dates], dtype=float).reshape(-1, 1)

    def create_weather_feature(self, timestamps: pd.DatetimeIndex) -> None:
        """Create weather feature using actual weather data"""
        try:
            # Path to weather data file
            weather_file = os.path.join(self.config.input_dir, os.path.basename(self.config.weather_data_file))
            print(f"Attempting to load weather data from: {weather_file}")

            # Check if file exists
            if not os.path.exists(weather_file):
                print(f"Weather file not found at: {weather_file}")
                # Try direct path
                if os.path.exists(self.config.weather_data_file):
                    weather_file = self.config.weather_data_file
                    print(f"Using direct path instead: {weather_file}")
                else:
                    raise FileNotFoundError(f"Weather file not found at either {weather_file} or {self.config.weather_data_file}")

            # Create weather integration object
            weather_integration = WeatherIntegration()

            # Load weather data
            weather_integration.load_weather_data(weather_file)

            # Match weather to traffic timestamps
            weather_features = weather_integration.match_weather_to_traffic(timestamps)
            print(f"Raw weather features shape: {weather_features.shape}")

            # Ensure correct reshaping if needed
            self.weather_feature = weather_features
            print(f"Final weather feature shape: {self.weather_feature.shape}")

        except Exception as e:
            # Fallback to simulated weather with matching dimensions
            print(f"Error loading weather data: {str(e)}. Using simulated data instead.")
            num_samples = len(timestamps)
            # Create 8-dimensional simulated data to match expected dimension
            weather_features = np.random.rand(num_samples, 8)
            self.weather_feature = weather_features
            print(f"Created simulated weather features with shape: {self.weather_feature.shape}")
            warnings.warn("Using simulated weather features. Replace with actual weather data integration.")

    def _load_weather_data(self, filepath: str) -> pd.DataFrame:
        """
        Load and preprocess weather data from CSV file

        Args:
            filepath: Path to the weather data CSV file

        Returns:
            Preprocessed weather DataFrame with datetime index
        """
        # Load the data
        weather_df = pd.read_csv(filepath)

        # Convert time column to datetime and set as index
        weather_df['valid_time'] = pd.to_datetime(weather_df['valid_time'])
        weather_df.set_index('valid_time', inplace=True)

        # Handle missing values
        weather_df = weather_df.replace('', np.nan)

        # Convert categorical weather conditions to numerical values
        weather_condition_map = {
            'Fair': 0,
            'Partly Cloudy': 1,
            'Mostly Cloudy': 2,
            'Cloudy': 3,
            'Rain': 4,
            'Snow': 5,
            'Thunderstorm': 6,
            'Fog': 7
        }

        weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
            lambda x: weather_condition_map.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Convert cloud cover to numerical
        cloud_cover_map = {
            'CLR': 0,  # Clear
            'FEW': 1,  # Few clouds
            'SCT': 2,  # Scattered clouds
            'BKN': 3,  # Broken clouds
            'OVC': 4   # Overcast
        }

        weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
            lambda x: cloud_cover_map.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Handle wind direction
        wind_dir_map = {
            'CALM': 0,
            'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
            'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
            'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
            'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16,
            'VAR': 17  # Variable
        }

        weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
            lambda x: wind_dir_map.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Fill missing values with appropriate method
        weather_df.fillna(method='ffill', inplace=True)  # Forward fill
        weather_df.fillna(method='bfill', inplace=True)  # Backward fill for any remaining NaNs

        return weather_df

    def _create_weather_features(self, weather_df: pd.DataFrame) -> np.ndarray:
        """
        Create normalized weather features for the model

        Args:
            weather_df: Preprocessed weather DataFrame

        Returns:
            Array of normalized weather features
        """
        # Select relevant weather features
        selected_features = [
            'temperature',
            'weather_condition_code',
            'visibility',
            'wind_speed',
            'wind_direction_code',
            'relative_humidity',
            'dew_point',
            'cloud_cover_code'
        ]

        # Subset the dataframe to only include selected features
        weather_features_df = weather_df[selected_features].copy()

        # Normalize numerical features between 0 and 1
        scaler = MinMaxScaler()
        numerical_features = ['temperature', 'visibility', 'wind_speed',
                              'relative_humidity', 'dew_point']

        # Check if there are any non-numeric values
        for col in numerical_features:
            weather_features_df[col] = pd.to_numeric(weather_features_df[col], errors='coerce')
            weather_features_df[col].fillna(weather_features_df[col].mean(), inplace=True)

        weather_features_df[numerical_features] = scaler.fit_transform(
            weather_features_df[numerical_features]
        )

        # Categorical features are already normalized between 0 and N
        # Further normalize to 0-1 range for consistency
        categorical_features = ['weather_condition_code', 'wind_direction_code', 'cloud_cover_code']
        max_vals = {
            'weather_condition_code': 7,  # Based on weather_condition_map
            'wind_direction_code': 17,    # Based on wind_dir_map
            'cloud_cover_code': 4         # Based on cloud_cover_map
        }

        for feat in categorical_features:
            weather_features_df[feat] = weather_features_df[feat] / max_vals[feat]

        # Create a combined feature array
        all_features = np.column_stack([
            weather_features_df['temperature'].values,
            weather_features_df['weather_condition_code'].values,
            weather_features_df['visibility'].values,
            weather_features_df['wind_speed'].values,
            weather_features_df['wind_direction_code'].values,
            weather_features_df['relative_humidity'].values,
            weather_features_df['dew_point'].values,
            weather_features_df['cloud_cover_code'].values
        ])

        return all_features

    def _match_weather_to_traffic(
        self,
        weather_df: pd.DataFrame,
        traffic_timestamps: pd.DatetimeIndex
    ) -> np.ndarray:
        """
        Match weather data to traffic timestamps using closest time approach

        Args:
            weather_df: Weather DataFrame with datetime index
            traffic_timestamps: DatetimeIndex of traffic data timestamps

        Returns:
            Numpy array of weather features matching traffic timestamps
        """
        # Create features from weather data
        weather_features = self._create_weather_features(weather_df)

        # Match each traffic timestamp to nearest weather timestamp
        matched_indices = []
        weather_timestamps = weather_df.index

        for traffic_time in traffic_timestamps:
            # Find the closest weather timestamp
            closest_idx = weather_timestamps.get_indexer([traffic_time], method='nearest')[0]
            matched_indices.append(closest_idx)

        # Get the weather features at the matched indices
        matched_features = weather_features[matched_indices]

        # Reshape to have appropriate dimensions for the model
        return matched_features.reshape(-1, matched_features.shape[1])

    def create_lagged_features(self) -> None:
        """Creates lagged features from the sensor data itself."""
        num_lags = self.config.num_lags
        lagged_features = []

        for i in range(1, num_lags + 1):
            lagged_data = np.roll(self.data, shift=i, axis=0)
            lagged_data[:i] = np.nan  # Fill first 'i' rows with NaN
            lagged_features.append(lagged_data)

        self.lagged_features = np.concatenate(lagged_features, axis=1)
        # Handle NaN values with zero filling
        self.lagged_features = np.nan_to_num(self.lagged_features, nan=0.0)

    def create_spatial_features(self) -> None:
        """
        Placeholder for spatial feature creation using random embeddings.
        """
        num_sensors = self.data.shape[1]
        spatial_feature_dim = self.config.spatial_feature_dim
        # Simulate spatial features (random embeddings)
        spatial_features = np.random.rand(num_sensors, spatial_feature_dim)
        self.spatial_features = np.tile(spatial_features, (len(self.data), 1))

    def get_adjacency_matrix(self) -> torch.Tensor:
        """
        Get adjacency matrix for graph neural network with comprehensive error handling.

        Returns:
            Normalized adjacency matrix as a PyTorch tensor
        """
        # Check if we've already created and cached the adjacency matrix
        if hasattr(self, '_cached_adjacency_matrix') and self._cached_adjacency_matrix is not None:
            return self._cached_adjacency_matrix

        try:
            # Try direct loading first (original method)
            adj_matrix_path = os.path.join(self.config.input_dir, 'adj_METR-LA.pkl')

            # If spatial features module is available, use it
            try:
                from spatial_features_module import create_spatial_integration_from_config, load_adjacency_matrix, normalize_adj

                # Create the spatial integration instance
                spatial_integration = create_spatial_integration_from_config(self.config)

                # Get the normalized adjacency matrix
                adj_norm_tensor = spatial_integration.get_normalized_adjacency_matrix()

                print(f"Successfully loaded adjacency matrix with shape {adj_norm_tensor.shape}")

                # Cache for future use
                self._cached_adjacency_matrix = adj_norm_tensor

                return adj_norm_tensor

            except ImportError:
                # Fall back to direct loading if module not available
                print("Spatial features module not found, using direct loading method")

                if os.path.exists(adj_matrix_path):
                    # Use the original load and normalize functions
                    adj_matrix = load_adjacency_matrix(adj_matrix_path)
                    adj_norm = normalize_adj(adj_matrix)
                    adj_norm_tensor = torch.tensor(adj_norm, dtype=torch.float32)

                    # Cache for future use
                    self._cached_adjacency_matrix = adj_norm_tensor

                    return adj_norm_tensor

        except Exception as e:
            # Handle any errors in loading
            print(f"Error loading adjacency matrix: {str(e)}")
            print("Falling back to fully connected adjacency matrix")

        # Create a fallback fully connected adjacency matrix
        num_sensors = self.data.shape[1]
        print(f"Creating default adjacency matrix with {num_sensors} nodes")

        # Create adjacency matrix (all 1s except diagonal)
        adjacency_matrix = torch.ones(num_sensors, num_sensors) - torch.eye(num_sensors)

        # Normalize the adjacency matrix for GCN
        D = torch.sum(adjacency_matrix, dim=1)
        D_inv_sqrt = torch.pow(D, -0.5)
        D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
        D_mat_inv_sqrt = torch.diag(D_inv_sqrt)

        normalized_adjacency = D_mat_inv_sqrt @ adjacency_matrix @ D_mat_inv_sqrt

        # Cache for future use
        self._cached_adjacency_matrix = normalized_adjacency

        return normalized_adjacency

    def __len__(self) -> int:
        return len(self.data) - self.seq_length - self.pred_window + 1

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        # feature dimensions
        x_data = self.data[idx:idx+self.seq_length]
        feature_sizes = {'base_data': x_data.shape[1]}
        x_features = []

        if hasattr(self, 'time_features') and self.config.use_time_features:
            time_feat = self.time_features[idx:idx+self.seq_length]
            feature_sizes['time'] = time_feat.shape[1]
            x_features.append(time_feat)

        if hasattr(self, 'weather_feature') and self.config.use_weather_feature:
            weather_feat = self.weather_feature[idx:idx+self.seq_length]
            feature_sizes['weather'] = weather_feat.shape[1]
            x_features.append(weather_feat)

        # Combine features and log actual dimensions
        if x_features:
            x = np.concatenate([x_data] + x_features, axis=1)
        else:
            x = x_data

        y = self.data[idx+self.seq_length:idx+self.seq_length+self.pred_window]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def prepare_data(
    df: pd.DataFrame,
    config: TrainingConfig
) -> Tuple[np.ndarray, pd.DatetimeIndex, Any, int]:
    """
    Prepare and preprocess data for training

    Args:
        df: Input dataframe with timestamp index
        config: Training configuration

    Returns:
        Tuple of:
        - Normalized data
        - Timestamps
        - Scaler object
        - Number of features
    """
    # Handle null values
    df.replace(0.0, np.nan, inplace=True)
    df.ffill(inplace=True)
    df.bfill(inplace=True)

    timestamps = df.index
    sensor_data = df.values
    num_features = sensor_data.shape[1]

    # Data Scaling
    if config.data_scaler_type == 'minmax':
        data_scaler = MinMaxScaler()
    elif config.data_scaler_type == 'standard':
        data_scaler = StandardScaler()
    elif config.data_scaler_type == 'robust':
        data_scaler = RobustScaler()
    else:
        warnings.warn(f"Invalid scaler type: {config.data_scaler_type}, using MinMaxScaler")
        data_scaler = MinMaxScaler()

    data_normalized = data_scaler.fit_transform(sensor_data)

    return data_normalized, timestamps, data_scaler, num_features

## Weather Features Module



In [ ]:
"""
Weather Integration Module for Traffic Transformer

This module provides functions to load, preprocess, and integrate weather data
with traffic data for use in the Traffic Transformer model.

Usage:
    from weather_integration import load_weather_data, create_weather_features, match_weather_to_traffic

    # Load weather data
    weather_df = load_weather_data('weather_data.csv')

    # Create weather features
    weather_features = create_weather_features(weather_df)

    # Match weather data to traffic timestamps
    matched_features = match_weather_to_traffic(weather_df, traffic_timestamps)
"""

# =============================================================================
# Weather Integration Module
# =============================================================================

import os
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Union, Any
from sklearn.preprocessing import MinMaxScaler

# Dictionary mappings for categorical weather data
WEATHER_CONDITION_MAP = {
    'Fair': 0,
    'Partly Cloudy': 1,
    'Mostly Cloudy': 2,
    'Cloudy': 3,
    'Rain': 4,
    'Snow': 5,
    'Thunderstorm': 6,
    'Fog': 7
}

CLOUD_COVER_MAP = {
    'CLR': 0,  # Clear
    'FEW': 1,  # Few clouds
    'SCT': 2,  # Scattered clouds
    'BKN': 3,  # Broken clouds
    'OVC': 4   # Overcast
}

WIND_DIRECTION_MAP = {
    'CALM': 0,
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16,
    'VAR': 17  # Variable
}

class WeatherIntegration:
    """
    Weather data integration for traffic prediction models.
    Handles loading, preprocessing, and aligning weather data with traffic timestamps.
    """

    def __init__(self, weather_file_path: str = None):
        """
        Initialize the weather integration module.

        Args:
            weather_file_path: Path to weather CSV file (optional)
        """
        self.weather_df = None
        self.feature_arrays = None

        if weather_file_path is not None:
            self.load_weather_data(weather_file_path)

    def load_weather_data(self, filepath: str) -> pd.DataFrame:
        """
        Load and preprocess weather data from CSV file

        Args:
            filepath: Path to the weather data CSV file

        Returns:
            Preprocessed weather DataFrame with datetime index
        """
        print(f"Loading weather data from {filepath}")

        # Load the data
        weather_df = pd.read_csv(filepath)

        # Convert time column to datetime and set as index
        weather_df['valid_time'] = pd.to_datetime(weather_df['valid_time'])
        weather_df.set_index('valid_time', inplace=True)

        # Handle missing values
        weather_df = weather_df.replace('', np.nan)

        # Convert categorical weather conditions to numerical values
        weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
            lambda x: WEATHER_CONDITION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Convert cloud cover to numerical
        weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
            lambda x: CLOUD_COVER_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Handle wind direction
        weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
            lambda x: WIND_DIRECTION_MAP.get(x, 0) if pd.notnull(x) else np.nan
        )

        # Fill missing values with appropriate method
        weather_df = weather_df.ffill().bfill()  # Forward fill then backward fill

        # Store the dataframe
        self.weather_df = weather_df

        # Create feature arrays
        self._create_feature_arrays()

        print(f"Weather data loaded: {len(weather_df)} records from {weather_df.index.min()} to {weather_df.index.max()}")
        return weather_df

    def _create_feature_arrays(self) -> None:
        """
        Create normalized feature arrays from the weather dataframe
        """
        if self.weather_df is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        weather_df = self.weather_df

        # Select relevant weather features
        selected_features = [
            'temperature',
            'weather_condition_code',
            'visibility',
            'wind_speed',
            'wind_direction_code',
            'relative_humidity',
            'dew_point',
            'cloud_cover_code'
        ]

        # Ensure all selected features exist, with fallbacks
        for feature in selected_features:
            if feature not in weather_df.columns:
                if feature == 'weather_condition_code' and 'weather_condition' in weather_df.columns:
                    # Create from text field if codes haven't been created
                    weather_df['weather_condition_code'] = weather_df['weather_condition'].map(
                        lambda x: WEATHER_CONDITION_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                elif feature == 'cloud_cover_code' and 'cloud_cover' in weather_df.columns:
                    weather_df['cloud_cover_code'] = weather_df['cloud_cover'].map(
                        lambda x: CLOUD_COVER_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                elif feature == 'wind_direction_code' and 'wind_direction' in weather_df.columns:
                    weather_df['wind_direction_code'] = weather_df['wind_direction'].map(
                        lambda x: WIND_DIRECTION_MAP.get(x, 0) if pd.notnull(x) else 0
                    )
                else:
                    # Create dummy column with zeros
                    print(f"Warning: Feature {feature} not found in weather data. Using zeros.")
                    weather_df[feature] = 0

        # Create a subset with only selected features
        weather_features_df = weather_df[selected_features].copy()

        # Ensure all columns are numeric
        for col in weather_features_df.columns:
            weather_features_df[col] = pd.to_numeric(weather_features_df[col], errors='coerce')
            # Fill any NaNs with column mean or 0
            if weather_features_df[col].isna().any():
                if weather_features_df[col].count() > 0:
                    weather_features_df[col].fillna(weather_features_df[col].mean(), inplace=True)
                else:
                    weather_features_df[col].fillna(0, inplace=True)

        # Normalize numerical features between 0 and 1
        scaler = MinMaxScaler()
        numerical_features = ['temperature', 'visibility', 'wind_speed',
                              'relative_humidity', 'dew_point']

        try:
            weather_features_df[numerical_features] = scaler.fit_transform(
                weather_features_df[numerical_features]
            )
        except Exception as e:
            print(f"Warning: Error in normalizing numerical features: {str(e)}")
            print("Trying column-by-column normalization...")

            # Try normalization column by column
            for col in numerical_features:
                try:
                    min_val = weather_features_df[col].min()
                    max_val = weather_features_df[col].max()
                    if max_val > min_val:
                        weather_features_df[col] = (weather_features_df[col] - min_val) / (max_val - min_val)
                    else:
                        weather_features_df[col] = 0  # If all values are the same
                except Exception as e2:
                    print(f"Warning: Could not normalize {col}: {str(e2)}")
                    weather_features_df[col] = 0

        # Normalize categorical features to 0-1 range
        categorical_features = ['weather_condition_code', 'wind_direction_code', 'cloud_cover_code']
        max_vals = {
            'weather_condition_code': max(WEATHER_CONDITION_MAP.values()),
            'wind_direction_code': max(WIND_DIRECTION_MAP.values()),
            'cloud_cover_code': max(CLOUD_COVER_MAP.values())
        }

        for feat in categorical_features:
            if max_vals[feat] > 0:  # Avoid division by zero
                weather_features_df[feat] = weather_features_df[feat] / max_vals[feat]

        # Create feature arrays
        self.feature_arrays = {
            'temperature': weather_features_df['temperature'].values.reshape(-1, 1),
            'weather_condition': weather_features_df['weather_condition_code'].values.reshape(-1, 1),
            'visibility': weather_features_df['visibility'].values.reshape(-1, 1),
            'wind': np.column_stack([
                weather_features_df['wind_speed'].values,
                weather_features_df['wind_direction_code'].values
            ]),
            'humidity': weather_features_df['relative_humidity'].values.reshape(-1, 1),
            'dew_point': weather_features_df['dew_point'].values.reshape(-1, 1),
            'cloud_cover': weather_features_df['cloud_cover_code'].values.reshape(-1, 1),
            'all_features': np.column_stack([
                weather_features_df['temperature'].values,
                weather_features_df['weather_condition_code'].values,
                weather_features_df['visibility'].values,
                weather_features_df['wind_speed'].values,
                weather_features_df['wind_direction_code'].values,
                weather_features_df['relative_humidity'].values,
                weather_features_df['dew_point'].values,
                weather_features_df['cloud_cover_code'].values
            ])
        }

    def match_weather_to_traffic(
        self,
        traffic_timestamps: pd.DatetimeIndex,
        feature_name: str = 'all_features'
    ) -> np.ndarray:
        """
        Match weather data to traffic timestamps using closest time approach

        Args:
            traffic_timestamps: DatetimeIndex of traffic data timestamps
            feature_name: Which weather feature to use ('all_features' or specific feature)

        Returns:
            Numpy array of weather features matching traffic timestamps
        """
        if self.weather_df is None or self.feature_arrays is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        if feature_name not in self.feature_arrays:
            raise ValueError(f"Feature '{feature_name}' not found. Available features: {list(self.feature_arrays.keys())}")

        # Get the requested feature
        selected_feature = self.feature_arrays[feature_name]

        # Match each traffic timestamp to nearest weather timestamp
        matched_indices = []
        weather_timestamps = self.weather_df.index

        for traffic_time in traffic_timestamps:
            # Find the closest weather timestamp
            closest_idx = weather_timestamps.get_indexer([traffic_time], method='nearest')[0]
            matched_indices.append(closest_idx)

        # Get the weather features at the matched indices
        matched_features = selected_feature[matched_indices]

        print(f"Weather features matched to {len(traffic_timestamps)} traffic timestamps")
        return matched_features

    def get_feature_dimension(self, feature_name: str = 'all_features') -> int:
        """
        Get the dimension (number of columns) for a specific feature type

        Args:
            feature_name: Name of the feature

        Returns:
            Number of columns in the feature
        """
        if self.feature_arrays is None:
            raise ValueError("Weather data not loaded. Call load_weather_data first.")

        if feature_name not in self.feature_arrays:
            raise ValueError(f"Feature '{feature_name}' not found. Available features: {list(self.feature_arrays.keys())}")

        return self.feature_arrays[feature_name].shape[1]

# Function to create a weather feature for the TrafficDataset class
def create_weather_feature_for_dataset(
    dataset,
    weather_file_path: str,
    feature_name: str = 'all_features'
) -> np.ndarray:
    """
    Create weather feature array for a traffic dataset

    Args:
        dataset: Instance of TrafficDataset class with timestamps attribute
        weather_file_path: Path to weather CSV file
        feature_name: Name of the weather feature to use

    Returns:
        Numpy array of weather features
    """
    if not hasattr(dataset, 'timestamps') or dataset.timestamps is None:
        raise ValueError("Dataset must have timestamps attribute")

    # Create weather integration object
    weather_integration = WeatherIntegration()

    # Load weather data
    weather_integration.load_weather_data(weather_file_path)

    # Match weather to traffic timestamps
    weather_features = weather_integration.match_weather_to_traffic(
        dataset.timestamps,
        feature_name=feature_name
    )

    return weather_features

## Spatial Features Module


In [ ]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from typing import Dict, List, Tuple, Optional, Union, Any
import matplotlib.pyplot as plt
import warnings
import requests
from sklearn.preprocessing import StandardScaler
import math


def load_adjacency_matrix(
    adjacency_matrix_path: str,
    fallback_size: int = 207
) -> Tuple[np.ndarray, List[str], List[int]]:
    """
    Load adjacency matrix from pickle file with comprehensive error handling.

    Args:
        adjacency_matrix_path: Path to the pickle file containing the adjacency matrix
        fallback_size: Size to use for a default adjacency matrix if loading fails

    Returns:
        Tuple of (adjacency_matrix, sensor_ids, node_ids)
    """
    try:
        # Check if file exists
        if not os.path.exists(adjacency_matrix_path):
            raise FileNotFoundError(f"Adjacency matrix file not found at: {adjacency_matrix_path}")

        # Load pickle file
        with open(adjacency_matrix_path, 'rb') as f:
            # Handle different pickle formats/versions
            try:
                graph_data = pickle.load(f, encoding='latin1')
            except:
                # Fall back to default encoding if latin1 fails
                f.seek(0)  # Reset file pointer
                graph_data = pickle.load(f)

        # Validate graph data structure
        if isinstance(graph_data, list) and len(graph_data) >= 3:
            sensor_ids = graph_data[0]
            node_ids = graph_data[1]
            adj_matrix = graph_data[2]

            # Sanity check the adjacency matrix dimensions
            if adj_matrix.shape[0] != adj_matrix.shape[1]:
                warnings.warn(f"Adjacency matrix is not square: {adj_matrix.shape}")

            print(f"Successfully loaded adjacency matrix with shape {adj_matrix.shape}")
            print(f"Contains {len(sensor_ids)} sensors")

            return adj_matrix, sensor_ids, node_ids
        else:
            # Handle unexpected data structure
            raise ValueError(f"Unexpected structure in adjacency matrix file: {type(graph_data)}")

    except Exception as e:
        # Generate a fallback adjacency matrix if loading fails
        warnings.warn(f"Error loading adjacency matrix from {adjacency_matrix_path}: {str(e)}")
        warnings.warn(f"Creating fallback adjacency matrix with {fallback_size} nodes")

        # Create a default fully connected adjacency matrix
        adj_matrix = np.ones((fallback_size, fallback_size)) - np.eye(fallback_size)
        sensor_ids = [f"sensor_{i}" for i in range(fallback_size)]
        node_ids = list(range(fallback_size))

        return adj_matrix, sensor_ids, node_ids


def normalize_adj(adj: np.ndarray) -> np.ndarray:
    """
    Symmetric normalization of the adjacency matrix for GCN.
    Formula: D^(-1/2) * (A + I) * D^(-1/2) where A is adjacency matrix,
    I is identity matrix, and D is diagonal degree matrix.

    Args:
        adj: The adjacency matrix with shape (num_nodes, num_nodes)

    Returns:
        Normalized adjacency matrix
    """
    # Add self-connections (A + I)
    adj = adj + np.eye(adj.shape[0])

    # Calculate degree matrix D
    d = np.array(adj.sum(1))

    # Calculate D^(-1/2)
    d_inv_sqrt = np.power(d, -0.5).flatten()

    # Handle any division by zero or infinity
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.

    # Create diagonal matrix
    d_mat_inv_sqrt = np.diag(d_inv_sqrt)

    # Calculate D^(-1/2) * (A + I) * D^(-1/2)
    normalized = d_mat_inv_sqrt @ adj @ d_mat_inv_sqrt

    return normalized


def create_distance_adj_matrix(
    coordinates: np.ndarray,
    threshold: float = 0.1,
    sigma: float = 0.1
) -> np.ndarray:
    """
    Create an adjacency matrix based on spatial distances between nodes.
    Uses Gaussian kernel: exp(-d²/σ²) where d is normalized distance.

    Args:
        coordinates: Array of node coordinates with shape (num_nodes, 2)
        threshold: Distance threshold for keeping edges (0-1)
        sigma: Parameter for Gaussian kernel

    Returns:
        Distance-based adjacency matrix
    """
    num_nodes = coordinates.shape[0]

    # Initialize distance matrix
    distances = np.zeros((num_nodes, num_nodes))

    # Compute pairwise Euclidean distances
    for i in range(num_nodes):
        for j in range(num_nodes):
            if i != j:
                # Calculate Euclidean distance between nodes
                dist = np.sqrt(np.sum((coordinates[i] - coordinates[j])**2))
                distances[i, j] = dist

    # Normalize distances to 0-1 range
    if distances.max() > 0:
        distances = distances / distances.max()

    # Apply Gaussian kernel
    adjacency = np.exp(- (distances**2) / (sigma**2))

    # Apply threshold
    adjacency[adjacency < threshold] = 0

    # Remove self-loops (will be added during normalization)
    np.fill_diagonal(adjacency, 0)

    return adjacency


class SpatialIntegration:
    """
    Handles loading, processing and transforming spatial features.
    """

    def __init__(
        self,
        adjacency_matrix_path: Optional[str] = None,
        coordinates_path: Optional[str] = None,
        num_sensors: int = 207,
        spatial_dim: int = 16,
        embedding_dim: int = 64,
        device: str = 'cpu'
    ):
        """
        Initialize spatial feature integration.

        Args:
            adjacency_matrix_path: Path to adjacency matrix pickle file
            coordinates_path: Path to sensor coordinates file (CSV)
            num_sensors: Number of sensors/nodes in the graph
            spatial_dim: Dimension of spatial features per node
            embedding_dim: Dimension for node embeddings
            device: Computation device ('cpu' or 'cuda')
        """
        self.num_sensors = num_sensors
        self.spatial_dim = spatial_dim
        self.embedding_dim = embedding_dim
        self.device = device

        # Load adjacency matrix if path provided
        if adjacency_matrix_path and os.path.exists(adjacency_matrix_path):
            self.adjacency_matrix, self.sensor_ids, self.node_ids = load_adjacency_matrix(
                adjacency_matrix_path, fallback_size=num_sensors
            )
            self.num_sensors = self.adjacency_matrix.shape[0]
        else:
            # Create a default adjacency matrix
            warnings.warn(f"No adjacency matrix provided, creating default with {num_sensors} nodes")
            self.adjacency_matrix = np.ones((num_sensors, num_sensors)) - np.eye(num_sensors)
            self.sensor_ids = [f"sensor_{i}" for i in range(num_sensors)]
            self.node_ids = list(range(num_sensors))

        # Load sensor coordinates if available
        self.coordinates = self._load_coordinates(coordinates_path)

        # Create normalized adjacency matrix for GCN
        self.normalized_adjacency = normalize_adj(self.adjacency_matrix)

        # Convert to PyTorch tensors
        self.adjacency_tensor = torch.tensor(self.adjacency_matrix, dtype=torch.float32).to(device)
        self.normalized_adjacency_tensor = torch.tensor(self.normalized_adjacency, dtype=torch.float32).to(device)

        # Create learned node embeddings
        self.node_embeddings = self._create_node_embeddings()

    def _load_coordinates(self, coordinates_path: Optional[str]) -> Optional[np.ndarray]:
        """
        Load sensor coordinates from file if available.

        Args:
            coordinates_path: Path to coordinates CSV file

        Returns:
            Numpy array of coordinates or None if not available
        """
        if not coordinates_path or not os.path.exists(coordinates_path):
            warnings.warn(f"Coordinates file not found at: {coordinates_path}")
            return None

        try:
            # Load coordinates CSV (expected format: sensor_id, latitude, longitude)
            df = pd.read_csv(coordinates_path)

            # Check required columns
            required_cols = ['sensor_id', 'latitude', 'longitude']
            if not all(col in df.columns for col in required_cols):
                alt_cols = ['id', 'lat', 'lon']  # Alternative column names
                if all(col in df.columns for col in alt_cols):
                    # Rename to expected format
                    df = df.rename(columns={
                        'id': 'sensor_id',
                        'lat': 'latitude',
                        'lon': 'longitude'
                    })
                else:
                    raise ValueError(f"Coordinates file must contain columns: {required_cols}")

            # Extract coordinates in correct order
            coordinates = np.zeros((self.num_sensors, 2))
            for i, sensor_id in enumerate(self.sensor_ids):
                if sensor_id in df['sensor_id'].values:
                    sensor_data = df[df['sensor_id'] == sensor_id].iloc[0]
                    coordinates[i, 0] = sensor_data['latitude']
                    coordinates[i, 1] = sensor_data['longitude']
                else:
                    # If sensor not found, use fallback coordinates
                    coordinates[i, 0] = i / self.num_sensors  # Normalized position
                    coordinates[i, 1] = i / self.num_sensors

            # Normalize coordinates
            scaler = StandardScaler()
            coordinates = scaler.fit_transform(coordinates)

            print(f"Loaded coordinates for {self.num_sensors} sensors")
            return coordinates

        except Exception as e:
            warnings.warn(f"Error loading coordinates: {str(e)}")
            return None

    def _create_node_embeddings(self) -> torch.Tensor:
        """
        Create learnable node embeddings or positional spatial features.

        Returns:
            Tensor of node embeddings with shape [num_sensors, embedding_dim]
        """
        # If we have real coordinates, create embeddings based on them
        if self.coordinates is not None:
            # Use a positional encoding similar to Transformer's approach
            # but applied to 2D spatial coordinates

            coordinate_embedding = np.zeros((self.num_sensors, self.embedding_dim))

            # Use the normalized coordinates to create positional embeddings
            for i in range(self.num_sensors):
                for j in range(0, self.embedding_dim, 4):
                    if j + 3 < self.embedding_dim:
                        # Latitude encoding
                        coordinate_embedding[i, j] = np.sin(self.coordinates[i, 0] * (1.0 / np.power(10000, j / self.embedding_dim)))
                        coordinate_embedding[i, j + 1] = np.cos(self.coordinates[i, 0] * (1.0 / np.power(10000, j / self.embedding_dim)))

                        # Longitude encoding
                        coordinate_embedding[i, j + 2] = np.sin(self.coordinates[i, 1] * (1.0 / np.power(10000, j / self.embedding_dim)))
                        coordinate_embedding[i, j + 3] = np.cos(self.coordinates[i, 1] * (1.0 / np.power(10000, j / self.embedding_dim)))

            # Convert to tensor
            return torch.tensor(coordinate_embedding, dtype=torch.float32).to(self.device)

        else:
            # If no coordinates available, use learnable embeddings
            # Initialize with Xavier normal to improve convergence
            embeddings = torch.empty(self.num_sensors, self.embedding_dim).to(self.device)
            nn.init.xavier_normal_(embeddings)
            return nn.Parameter(embeddings)

    def get_adjacency_matrix(self) -> torch.Tensor:
        """Get the adjacency matrix as a PyTorch tensor."""
        return self.adjacency_tensor

    def get_normalized_adjacency_matrix(self) -> torch.Tensor:
        """Get the normalized adjacency matrix for GCN as a PyTorch tensor."""
        return self.normalized_adjacency_tensor

    def get_node_embeddings(self) -> torch.Tensor:
        """Get node embeddings tensor."""
        return self.node_embeddings

    def create_spatial_features(self, batch_size: int) -> torch.Tensor:
        """
        Create spatial features tensor ready for model input.

        Args:
            batch_size: Batch size for the features

        Returns:
            Spatial features tensor with shape [batch_size, seq_len, num_sensors * spatial_dim]
        """
        # Start with embeddings [num_sensors, embedding_dim]
        embeddings = self.get_node_embeddings()

        # Project to desired spatial dimension if needed
        if self.embedding_dim != self.spatial_dim and hasattr(self, 'projection'):
            # Use a projection if available
            spatial_features = self.projection(embeddings)
        elif self.embedding_dim != self.spatial_dim:
            # Create a simple projection
            projection = nn.Linear(self.embedding_dim, self.spatial_dim).to(self.device)
            spatial_features = projection(embeddings)
            self.projection = projection
        else:
            spatial_features = embeddings

        # Repeat for batch size
        # Shape becomes [batch_size, num_sensors, spatial_dim]
        batched_features = spatial_features.unsqueeze(0).repeat(batch_size, 1, 1)

        return batched_features

    def visualize_adjacency_matrix(self, save_path: Optional[str] = None) -> None:
        """
        Visualize the adjacency matrix as a heatmap.

        Args:
            save_path: Path to save the visualization image
        """
        plt.figure(figsize=(10, 8))
        plt.title("Adjacency Matrix Heatmap")
        plt.imshow(self.adjacency_matrix, cmap='viridis')
        plt.colorbar(label="Connection Strength")
        plt.xlabel("Node Index")
        plt.ylabel("Node Index")

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Adjacency matrix visualization saved to {save_path}")

        plt.show()

    def visualize_node_embeddings(self, save_path: Optional[str] = None) -> None:
        """
        Visualize node embeddings using PCA or t-SNE reduction.

        Args:
            save_path: Path to save the visualization image
        """
        try:
            from sklearn.decomposition import PCA

            # Get embeddings as numpy array
            embeddings = self.node_embeddings.detach().cpu().numpy()

            # Apply PCA for dimensionality reduction
            pca = PCA(n_components=2)
            embeddings_2d = pca.fit_transform(embeddings)

            # Visualize
            plt.figure(figsize=(10, 8))
            plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.7)

            # Add node indices as labels
            for i, (x, y) in enumerate(embeddings_2d):
                plt.annotate(str(i), (x, y), alpha=0.7, fontsize=8)

            plt.title("Node Embeddings Visualization (PCA)")
            plt.xlabel("PC1")
            plt.ylabel("PC2")

            if save_path:
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                print(f"Node embeddings visualization saved to {save_path}")

            plt.show()

        except ImportError:
            warnings.warn("sklearn.decomposition.PCA not available, skipping visualization")


def create_spatial_integration_from_config(config, device='cpu'):
    """
    Create a SpatialIntegration instance from configuration.
    """
    # Get adjacency matrix path from config
    adjacency_path = None
    if hasattr(config, 'input_dir'):
        adjacency_path = os.path.join(config.input_dir, 'adj_METR-LA.pkl')

    # Get coordinates file path from input_dir
    coordinates_path = None
    if hasattr(config, 'coordinates_file') and config.coordinates_file:
        coordinates_path = config.coordinates_file
        if os.path.exists(coordinates_path):
            print(f"Found coordinates file at: {coordinates_path}")
        else:
            print(f"Warning: Coordinates file not found at {coordinates_path}")
            coordinates_path = None

    # If adjacency path doesn't exist, try standard locations
    if not adjacency_path or not os.path.exists(adjacency_path):
        potential_paths = [
            './adj_METR-LA.pkl',
            './data/adj_METR-LA.pkl',
            os.path.join(config.input_dir, 'adj_matrix.pkl') if hasattr(config, 'input_dir') else None,
            '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/ADJACENCY_MATRIX_METR_LA/adj_METR-LA.pkl'
        ]

        for path in potential_paths:
            if path and os.path.exists(path):
                adjacency_path = path
                print(f"Found adjacency matrix at: {adjacency_path}")
                break

    # Get spatial integration parameters from config
    num_sensors = 207  # Default for METR-LA
    spatial_dim = getattr(config, 'spatial_feature_dim', 16)

    # Create spatial integration instance with coordinates
    spatial_integration = SpatialIntegration(
        adjacency_matrix_path=adjacency_path,
        coordinates_path=coordinates_path,  # Now passing the coordinates path from config
        num_sensors=num_sensors,
        spatial_dim=spatial_dim,
        embedding_dim=64,  # Default embedding dimension
        device=device
    )

    return spatial_integration

## Model Module

In [ ]:

# =============================================================================
# Model Module
# =============================================================================

class CustomTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """Modified Transformer Encoder Layer to capture attention weights."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_weights = None

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False):
        x, weights = self.self_attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            is_causal=is_causal
        )
        self.attn_weights = weights.detach()
        return self.dropout1(x)


class PositionalEncoding(nn.Module):
    """
    Positional Encoding module with enhanced dimension checking and error handling.
    Ensures d_model is even and properly applies positional encoding to the input.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Ensure d_model is even
        if d_model % 2 != 0:
            raise ValueError(f"d_model must be even for positional encoding, got {d_model}")

        # Explicitly initialize buffer with proper shape
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # Compute div_term with precise checks
        div_term_indices = torch.arange(0, d_model, 2).float()
        try:
            div_term = torch.exp(div_term_indices * (-math.log(10000.0) / d_model))
        except Exception as e:
            raise ValueError(f"Error calculating div_term: {e}, d_model={d_model}")

        # Do additional validation
        if len(div_term) * 2 > d_model:
            raise ValueError(f"div_term length ({len(div_term)}) too large for d_model={d_model}")

        # Apply sin and cos with explicit shape checking
        try:
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
        except Exception as e:
            raise RuntimeError(f"Error in positional encoding calculation: {e}\n"
                              f"Shapes - position: {position.shape}, div_term: {div_term.shape}, "
                              f"pe: {pe.shape}, d_model: {d_model}")

        # Register as buffer (not a parameter)
        pe = pe.unsqueeze(0).transpose(0, 1)  # Shape: [max_len, 1, d_model]
        self.register_buffer('pe', pe)

        # For debugging
        self.d_model = d_model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [seq_len, batch_size, d_model]

        Returns:
            Tensor with positional encoding added
        """
        # Validate input shape
        if x.size(-1) != self.d_model:
            raise ValueError(f"Input feature dimension {x.size(-1)} doesn't match "
                           f"positional encoding dimension {self.d_model}")

        # Add positional encoding with proper shape handling
        try:
            max_len = min(x.size(0), self.pe.size(0))
            x = x + self.pe[:max_len, :]
        except Exception as e:
            raise RuntimeError(f"Error adding positional encoding: {e}\n"
                              f"Shapes - x: {x.shape}, pe: {self.pe.shape}, "
                              f"x size(0): {x.size(0)}, pe size(0): {self.pe.size(0)}")

        return self.dropout(x)


class GCNEncoder(nn.Module):
    """Graph Convolutional Network Encoder with dense to sparse conversion"""
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int = 2,
        dropout: float = 0.1,
        gnn_type: str = 'gcn'
    ):
        super(GCNEncoder, self).__init__()

        if not TORCH_GEOMETRIC_AVAILABLE:
            raise ImportError("PyTorch Geometric is required for GCNEncoder")

        self.layers = nn.ModuleList()

        if gnn_type == 'gcn':
            gnn_layer = GCNConv
        elif gnn_type == 'gat':
            gnn_layer = GATConv
        else:
            raise ValueError(f"Invalid GNN type: {gnn_type}. Choose 'gcn' or 'gat'.")

        self.layers.append(gnn_layer(input_dim, hidden_dim))  # First GNN layer

        for _ in range(num_layers - 1):
            self.layers.append(gnn_layer(hidden_dim, hidden_dim))  # Subsequent GNN layers

        self.dropout = nn.Dropout(dropout)
        self.gnn_type = gnn_type

    def _dense_to_sparse(self, adj_matrix):
        """
        Convert dense adjacency matrix to sparse edge_index format

        Args:
            adj_matrix: Dense adjacency matrix tensor of shape [num_nodes, num_nodes]

        Returns:
            edge_index: Sparse adjacency in COO format with shape [2, num_edges]
        """
        # Get indices where values are non-zero (connections exist)
        indices = torch.nonzero(adj_matrix, as_tuple=True)

        # Stack to create edge_index format [2, num_edges]
        edge_index = torch.stack(indices)

        return edge_index

    def forward(self, x: torch.Tensor, adjacency_matrix: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through GCN layers

        Args:
            x: Node features tensor [num_nodes, input_dim]
            adjacency_matrix: Adjacency matrix [num_nodes, num_nodes]

        Returns:
            Updated node features [num_nodes, hidden_dim]
        """
        # Convert dense adjacency matrix to edge_index format
        edge_index = self._dense_to_sparse(adjacency_matrix)

        for i, layer in enumerate(self.layers):
            if self.gnn_type == 'gat' and i == 0:  # GAT needs explicit head specification in first layer
                x = layer(x, edge_index, heads=8)  # Example heads for GAT
            else:
                x = layer(x, edge_index)

            if i < len(self.layers) - 1:
                x = F.relu(x)  # Activation after each layer except last
                x = self.dropout(x)

        return x


class TrafficTransformer(nn.Module):
    """
    Traffic Transformer Model with configurable components
    """
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int,
        num_heads: int,
        num_features: int,  # New: explicitly passing number of features
        dropout: float = 0.1,
        ff_dim_multiplier: int = 4,
        activation: str = 'relu',
        decoder_type: str = 'linear',
        use_gnn_pre_transformer: bool = False,
        spatial_feature_dim: int = 0,
        gnn_type: str = 'gcn',
        pred_len: int = 1
    ):
        """
        Initialize Traffic Transformer Model

        Args:
            input_dim: Input feature dimension
            hidden_dim: Hidden dimension for transformer
            num_layers: Number of transformer layers
            num_heads: Number of attention heads
            num_features: Number of features in output (spatial dimension)
            dropout: Dropout rate
            ff_dim_multiplier: Multiplier for feedforward dimension
            activation: Activation function ('relu' or 'gelu')
            decoder_type: Type of decoder ('linear' or 'mlp')
            use_gnn_pre_transformer: Whether to use GNN before transformer
            spatial_feature_dim: Dimension of spatial features (for GNN)
            gnn_type: Type of GNN ('gcn' or 'gat')
            pred_len: Prediction length (temporal dimension)
        """
        super(TrafficTransformer, self).__init__()

        # Validate dimensions before initialization
        if hidden_dim % num_heads != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be divisible by number of attention heads {num_heads}")

        if hidden_dim % 2 != 0:
            raise ValueError(f"Hidden dimension {hidden_dim} must be even for positional encoding, got {hidden_dim}")

        # Store instance attributes
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_features = num_features
        self.pred_len = pred_len
        self.use_gnn_pre_transformer = use_gnn_pre_transformer
        self.input_dim = input_dim

        self.attention_weights = None

        # Always create embedding layer regardless of GNN usage
        self.embedding = nn.Linear(input_dim, hidden_dim)
        transformer_input_dim = hidden_dim

        # Create GNN encoder if needed
        if use_gnn_pre_transformer:
            if not TORCH_GEOMETRIC_AVAILABLE:
                raise ImportError("PyTorch Geometric is required for GNN pre-transformer")

            self.gnn_encoder = GCNEncoder(
                spatial_feature_dim,
                hidden_dim,
                dropout=dropout,
                gnn_type=gnn_type
            )

        self.pos_encoder = PositionalEncoding(hidden_dim, dropout)

        # Encoder Layers with configurable activation
        encoder_layers = [
            CustomTransformerEncoderLayer(
                hidden_dim,
                num_heads,
                hidden_dim * ff_dim_multiplier,
                dropout,
                batch_first=True,
                activation=activation
            ) for _ in range(num_layers)
        ]
        self.transformer = nn.ModuleList(encoder_layers)

        # Configurable Decoder
        if decoder_type == 'linear':
            self.decoder = nn.Linear(hidden_dim, num_features * pred_len)
        elif decoder_type == 'mlp':
            self.decoder = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, num_features * pred_len)
            )
        else:
            raise ValueError(f"Invalid decoder type: {decoder_type}. Choose 'linear' or 'mlp'.")

    def forward(self, src: torch.Tensor, adjacency_matrix: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass through the transformer model with improved spatial feature handling

        Args:
            src: Input tensor [batch_size, seq_len, feature_dim]
            adjacency_matrix: Optional adjacency matrix for GNN

        Returns:
            Output tensor [batch_size, pred_len, num_features]
        """
        # Ensure input has proper dimensions
        if src.dim() == 2:
            src = src.unsqueeze(1)

        batch_size, seq_len, feature_dim = src.shape

        if self.use_gnn_pre_transformer:
            if adjacency_matrix is None:
                raise ValueError("adjacency_matrix is required when use_gnn_pre_transformer=True")

            # Create spatial integration instance if not already created
            if not hasattr(self, '_spatial_integration'):
                # MODIFICATION: Use the function directly instead of importing it
                # from spatial_features_module import create_spatial_integration_from_config

                # Create config-based spatial integration on the same device as input
                self._spatial_integration = create_spatial_integration_from_config(
                    config=getattr(self, 'config', None),  # Pass config if available
                    device=src.device
                )

            # Get node embeddings for all sensors/nodes (shape: [num_sensors, embedding_dim])
            node_embeddings = self._spatial_integration.get_node_embeddings()

            # Project node embeddings to required dimension
            node_embedding_projection = nn.Linear(node_embeddings.shape[1], self.hidden_dim).to(node_embeddings.device)
            projected_embeddings = node_embedding_projection(node_embeddings)

            # Use projected embeddings instead
            gcn_encoded_features = self.gnn_encoder(projected_embeddings, adjacency_matrix)

            # Apply temporal embedding to input features
            # This produces [batch_size, seq_len, hidden_dim]
            temporal_features = self.embedding(src)

            # Combine temporal features with GCN-encoded spatial features
            # 1. Expand GCN features to match batch and sequence dimensions
            expanded_gcn = gcn_encoded_features.unsqueeze(0).unsqueeze(0)
            expanded_gcn = expanded_gcn.expand(batch_size, seq_len, -1, -1)

            # 2. Average across nodes to get [batch_size, seq_len, hidden_dim]
            spatial_context = expanded_gcn.mean(dim=2)

            # 3. Add spatial context to temporal features (residual connection)
            src = temporal_features + spatial_context
        else:
            # Standard embedding without GCN
            src = self.embedding(src)

        # Apply positional encoding
        src = self.pos_encoder(src)

        # Apply transformer layers
        for i, layer in enumerate(self.transformer):
            src = layer(src)
            # Capture attention weights from the last layer
            if i == len(self.transformer) - 1:  # Last layer
                if hasattr(layer, 'attn_weights') and layer.attn_weights is not None:
                    self.attention_weights = layer.attn_weights.mean(dim=0) if layer.attn_weights.dim() > 2 else layer.attn_weights

        # Apply decoder to last timestep
        output = self.decoder(src[:, -1, :])

        # Reshape to [batch_size, pred_len, num_features]
        return output.view(-1, self.pred_len, self.num_features)

    def freeze_layers(self, freeze_encoder: bool = True, num_layers: int = 1):
        """Freeze specified parts of the model for transfer learning"""
        if freeze_encoder:
            # Freeze embedding layer
            for param in self.embedding.parameters():
                param.requires_grad = False

            # Freeze positional encoding
            if hasattr(self.pos_encoder, 'pe'):
                self.pos_encoder.pe.requires_grad = False

            # Freeze specified transformer layers
            for i, layer in enumerate(self.transformer):
                if i < num_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

            print(f"Froze embedding layer and {num_layers} transformer layers")
        else:
            print("No parameter freezing applied - full fine-tuning")

        # Report number of trainable parameters
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,} of {total_params:,} total ({trainable_params/total_params:.1%})")

        # Return self to allow method chaining
        return self

    def add_transfer_adapters(self, source_input_dim: int, source_output_dim: int, target_input_dim: int, target_output_dim: int):
        """
        Add input and output adapters for transfer learning across different dimensions

        Args:
            source_input_dim: Input dimension of the source model
            source_output_dim: Output dimension of the source model
            target_input_dim: Actual input dimension from target dataset
            target_output_dim: Actual output dimension for target dataset
        """
        import types

        # Store dimensions for reference
        self.source_input_dim = source_input_dim
        self.source_output_dim = source_output_dim
        self.target_input_dim = target_input_dim
        self.target_output_dim = target_output_dim
        self.has_transfer_adapters = True

        # CRITICAL FIX: Update num_features to match target output dimension
        self.num_features = target_output_dim

        # Get the device where the model is located
        device = next(self.parameters()).device
        print(f"Adding adapters on device: {device}")

        # Debug dimensions
        print(f"\n=== Adapter Dimension Setup ===")
        print(f"Source model expects input_dim: {source_input_dim}")
        print(f"Source model produces output_dim: {source_output_dim}")
        print(f"Target data actual input_dim: {target_input_dim}")
        print(f"Target data requires output_dim: {target_output_dim}")
        print(f"Updated model.num_features from {source_output_dim} to {target_output_dim}")
        print(f"Model prediction length (pred_len): {self.pred_len}")

        # CRITICAL CHANGE: Create input adapter FROM actual target data dimension TO source model dimension
        print(f"Creating input adapter: {target_input_dim} → {source_input_dim}")
        self.input_adapter = nn.Linear(target_input_dim, source_input_dim).to(device)

        # Initialize with identity-like mapping for shared dimensions
        with torch.no_grad():
            # Zero initialization
            nn.init.zeros_(self.input_adapter.weight)
            nn.init.zeros_(self.input_adapter.bias)

            # Set identity mapping for shared dimensions
            min_dim = min(target_input_dim, source_input_dim)
            print(f"Setting identity mapping for first {min_dim} dimensions")

            # Set diagonal elements to 1 for the overlapping dimensions
            for i in range(min_dim):
                self.input_adapter.weight.data[i, i] = 1.0

        # Store original decoder
        self.original_decoder = self.decoder

        # Create output adapter FROM source output TO target output, with proper handling of prediction length
        source_output_total = source_output_dim * self.pred_len
        target_output_total = target_output_dim * self.pred_len

        print(f"Creating output adapter: {source_output_total} → {target_output_total} (accounting for pred_len={self.pred_len})")

        # Create output adapter
        output_adapter = nn.Linear(source_output_total, target_output_total).to(device)

        # Initialize output adapter
        with torch.no_grad():
            # Zero initialization
            nn.init.zeros_(output_adapter.weight)
            nn.init.zeros_(output_adapter.bias)

            # Set identity mapping for shared dimensions with proper handling of prediction length
            # For each prediction step, map corresponding features
            if self.pred_len == 1:
                # Simpler case for single-step prediction
                min_out_dim = min(source_output_dim, target_output_dim)
                print(f"Setting identity mapping for first {min_out_dim} output dimensions")
                for i in range(min_out_dim):
                    output_adapter.weight.data[i, i] = 1.0
            else:
                # Handle multi-step prediction by mapping each time step's features separately
                min_out_dim = min(source_output_dim, target_output_dim)
                print(f"Setting identity mapping for first {min_out_dim} output dimensions across {self.pred_len} prediction steps")
                for step in range(self.pred_len):
                    for i in range(min_out_dim):
                        src_idx = step * source_output_dim + i
                        tgt_idx = step * target_output_dim + i
                        if src_idx < source_output_total and tgt_idx < target_output_total:
                            output_adapter.weight.data[tgt_idx, src_idx] = 1.0

        # Create a sequential model with the original decoder followed by output adapter
        self.decoder = nn.Sequential(
            self.original_decoder,  # Original decoder (source dimensions)
            output_adapter          # Adapter to target dimensions
        ).to(device)

        print(f"Added transfer adapters: Input {target_input_dim} → {source_input_dim}, "
              f"Output {source_output_dim} → {target_output_dim} (with pred_len={self.pred_len})")

        # Save original forward method
        self._original_forward = self.forward

        # Create adapter forward method with debugging
        def _adapter_forward(self, src, adjacency_matrix=None):
            """Modified forward method that applies input and output adapters"""
            # Debug shapes on first call
            if not hasattr(self, '_printed_shapes'):
                batch_size, seq_len, feat_dim = src.shape
                print(f"\n=== First Forward Pass Shapes ===")
                print(f"Input batch shape: {src.shape}")
                print(f"Input adapter expects: {self.input_adapter.in_features} features")
                print(f"Input adapter transforms to: {self.input_adapter.out_features} features")
                print(f"Model will reshape output to use {self.num_features} features for {self.pred_len} steps")
                self._printed_shapes = True

            # Ensure on correct device
            if src.device != device:
                src = src.to(device)

            # Reshape for linear adapter if needed
            orig_shape = src.shape
            if len(orig_shape) == 3:
                # Reshape from [batch, seq_len, features] to [batch*seq_len, features]
                # This is needed because linear layer expects 2D input
                batch_size, seq_len, _ = orig_shape
                src_reshaped = src.reshape(batch_size * seq_len, -1)

                # Apply input adapter
                adapted = self.input_adapter(src_reshaped)

                # Reshape back to [batch, seq_len, adapted_features]
                adapted = adapted.reshape(batch_size, seq_len, -1)
            else:
                # Direct application if already 2D
                adapted = self.input_adapter(src)

            # Forward through original model
            if adjacency_matrix is not None:
                if adjacency_matrix.device != device:
                    adjacency_matrix = adjacency_matrix.to(device)
                return self._original_forward(adapted, adjacency_matrix)
            else:
                return self._original_forward(adapted)

        # Replace forward method
        self.forward = types.MethodType(_adapter_forward, self)

        # Add utility methods for adapter management
        def remove_adapters(self):
            """Temporarily remove adapters to evaluate on source data"""
            print("Removing adapters for source data evaluation")
            self.forward = self._original_forward
            self.decoder = self.original_decoder
            # Restore original num_features too
            self.num_features = self.source_output_dim
            return self

        def restore_adapters(self):
            """Restore adapters after source data evaluation"""
            print("Restoring adapters")
            self.forward = types.MethodType(_adapter_forward, self)
            self.decoder = nn.Sequential(
                self.original_decoder,
                output_adapter
            ).to(device)
            # Restore target num_features
            self.num_features = self.target_output_dim
            return self

        # Add these utility methods to the model
        self.remove_adapters = types.MethodType(remove_adapters, self)
        self.restore_adapters = types.MethodType(restore_adapters, self)

        return self  # Allow method chaining

class PyTorchLSTMForecaster(nn.Module):
    """Improved LSTM-based model for time series forecasting"""
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int,
        seq_length: int,  # Added sequence length parameter
        pred_length: int,  # Added prediction length parameter
        dropout: float = 0.1,
        epochs: int = 100,
        batch_size: int = 32,
        learning_rate: float = 0.001,
        device: str = 'cpu'
    ):
        super(PyTorchLSTMForecaster, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.seq_length = seq_length
        self.pred_length = pred_length
        self.input_size = input_size
        self.output_size = output_size

        # LSTM with dropout
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Decoder to map from hidden state to output
        self.decoder = nn.Linear(hidden_size, output_size * pred_length)

        # Training parameters
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = device
        self.history = {'loss': [], 'val_loss': []}  # Store training history

    def forward(self, x):
        """
        Forward pass through LSTM model

        Args:
            x: Input tensor of shape [batch_size, seq_length, input_size]

        Returns:
            Output tensor of shape [batch_size, pred_length, output_size]
        """
        # Initialize hidden state and cell state
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Decode the hidden state of the last time step
        out = self.decoder(out[:, -1, :])

        # Reshape to [batch_size, pred_length, output_size]
        out = out.view(-1, self.pred_length, self.output_size)

        return out

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """
        Train the LSTM model on input data with proper sequence handling

        Args:
            X_train: Training input of shape [num_samples, seq_length, input_size]
            y_train: Training target of shape [num_samples, pred_length, output_size]
            X_val: Optional validation input
            y_val: Optional validation target
        """
        self.to(self.device)
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        criterion = nn.MSELoss()

        # Convert numpy arrays to tensors if needed
        if not isinstance(X_train, torch.Tensor):
            X_train = torch.tensor(X_train, dtype=torch.float32)
        if not isinstance(y_train, torch.Tensor):
            y_train = torch.tensor(y_train, dtype=torch.float32)

        # Move to device
        X_train = X_train.to(self.device)
        y_train = y_train.to(self.device)

        # Prepare validation data if provided
        if X_val is not None and y_val is not None:
            if not isinstance(X_val, torch.Tensor):
                X_val = torch.tensor(X_val, dtype=torch.float32)
            if not isinstance(y_val, torch.Tensor):
                y_val = torch.tensor(y_val, dtype=torch.float32)

            X_val = X_val.to(self.device)
            y_val = y_val.to(self.device)

        # Create data loaders
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

        # Early stopping
        best_loss = float('inf')
        patience = 25
        no_improve = 0

        for epoch in range(self.epochs):
            self.train()  # Set model to training mode
            total_loss = 0

            for X_batch, y_batch in train_loader:
                # Forward pass
                outputs = self(X_batch)
                loss = criterion(outputs, y_batch)

                # Backward and optimize
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_train_loss = total_loss / len(train_loader)
            self.history['loss'].append(avg_train_loss)

            # Validation if data provided
            if X_val is not None and y_val is not None:
                self.eval()  # Set model to evaluation mode
                with torch.no_grad():
                    val_outputs = self(X_val)
                    val_loss = criterion(val_outputs, y_val)

                val_loss = val_loss.item()
                self.history['val_loss'].append(val_loss)

                # Early stopping check
                if val_loss < best_loss:
                    best_loss = val_loss
                    no_improve = 0
                else:
                    no_improve += 1

                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

                print(f'Epoch [{epoch+1}/{self.epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')
            else:
                print(f'Epoch [{epoch+1}/{self.epochs}], Loss: {avg_train_loss:.4f}')

    def predict(self, X):
        """
        Generate predictions with the trained model

        Args:
            X: Input data of shape [num_samples, seq_length, input_size]

        Returns:
            Predictions of shape [num_samples, pred_length, output_size]
        """
        self.eval()  # Set model to evaluation mode

        # Convert to tensor if needed
        if not isinstance(X, torch.Tensor):
            X = torch.tensor(X, dtype=torch.float32)

        # Move to device
        X = X.to(self.device)

        with torch.no_grad():
            predictions = self(X)

        return predictions.cpu().numpy()

# Custom learning rate scheduler with warmup
class CosineWarmupLR(_optim.lr_scheduler.LRScheduler):
    """Cosine annealing with warmup learning rate scheduler"""
    def __init__(
        self,
        optimizer: torch.optim.Optimizer,
        warmup_epochs: int,
        total_epochs: int,
        base_lr: float,
        warmup_lr: float = 0.0,
        last_epoch: int = -1
    ):
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.warmup_lr = warmup_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self) -> List[float]:
        if self.last_epoch < self.warmup_epochs:
            # Linear warmup phase
            alpha = self.last_epoch / self.warmup_epochs
            return [self.warmup_lr + (self.base_lr - self.warmup_lr) * alpha] * len(self.optimizer.param_groups)
        else:
            # Cosine annealing phase
            progress = float(self.last_epoch - self.warmup_epochs) / float(max(1, self.total_epochs - self.warmup_epochs))
            return [max(0.0, self.base_lr * 0.5 * (1.0 + math.cos(math.pi * progress)))] * len(self.optimizer.param_groups)



## Training Module

In [ ]:

# =============================================================================
# Training Module
# =============================================================================

import contextlib

def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler],
    criterion: Callable,
    config: TrainingConfig,
    data_scaler: MinMaxScaler | StandardScaler | RobustScaler,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[nn.Module, List[float], List[float]]:
    """
    Train the traffic forecasting model with support for mixed precision and gradient accumulation

    Args:
        model: The model to train
        train_loader: DataLoader with training data
        val_loader: DataLoader with validation data
        optimizer: Optimizer for training
        scheduler: Learning rate scheduler
        criterion: Loss function
        config: Training configuration
        device: Device to train on ('cpu' or 'cuda')
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (trained model, training losses, validation losses)
    """
    num_epochs = config.num_epochs
    patience = config.patience
    best_loss = float('inf')
    no_improve = 0
    train_losses = []
    val_losses = []
    model_dir = getattr(config, 'model_dir', None)
    use_quantile_regression = config.use_quantile_regression
    loss_function_type = config.loss_function
    accumulation_steps = config.accumulation_steps

    '''
    # Setup for mixed precision training
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        if DEVICE_TYPE_SUPPORTED:
            scaler = GradScaler(device_type='cuda')
        else:
            # Older PyTorch versions don't support device_type
            scaler = GradScaler()
    else:
        scaler = None '''

    # --- MODIFICATION START: Define scaler and autocast_context ---
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    torch_device = torch.device(device) # Get device object

    if use_amp:
        try:
            # scaler = torch.amp.GradScaler(device=torch_device.type, enabled=use_amp) # Newer API
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp) # Stick with cuda.amp GradScaler
            print("Train: Using torch.cuda.amp.GradScaler")
            autocast_context = lambda: torch.amp.autocast(device_type=torch_device.type, enabled=use_amp) # Use torch.amp.autocast
            print(f"Train: Using torch.amp.autocast with device_type='{torch_device.type}'")
        except (TypeError, AttributeError):
             try:
                scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
                print("Train: Using torch.cuda.amp.GradScaler (fallback)")
                autocast_context = lambda: torch.cuda.amp.autocast(enabled=use_amp)
                print("Train: Using torch.cuda.amp.autocast (fallback)")
             except Exception as e_amp:
                 print(f"Train: Failed to initialize AMP: {e_amp}. Disabling AMP.")
                 use_amp = False
                 scaler = None
                 autocast_context = lambda: contextlib.nullcontext()
    else:
        scaler = None
        autocast_context = lambda: contextlib.nullcontext()
    # --- MODIFICATION END ---

    # Log memory usage initially
    if torch.cuda.is_available() and device == 'cuda':
        print(f"Initial GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"Initial GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()  # Zero gradients at the start of each epoch

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Mixed precision forward pass
            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps
                else:
                    # Older PyTorch versions
                    with autocast():
                        if config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles) / accumulation_steps
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target) / accumulation_steps
                        else:
                            loss = criterion(output, target) / accumulation_steps

                # Mixed precision backward pass
                scaler.scale(loss).backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
            else:
                # Standard precision training
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles) / accumulation_steps
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target) / accumulation_steps
                else:
                    loss = criterion(output, target) / accumulation_steps

                # Standard backward pass
                loss.backward()

                # Gradient accumulation - only step every accumulation_steps
                if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                    if config.gradient_clip is not None:
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)

                    optimizer.step()
                    optimizer.zero_grad()

            # Track loss (use full loss for logging)
            train_loss += loss.item() * accumulation_steps

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Report GPU memory usage after training epoch
        if torch.cuda.is_available() and device == 'cuda':
            print(f"GPU memory after training: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB / "
                  f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB "
                  f"({torch.cuda.memory_allocated(0) / torch.cuda.get_device_properties(0).total_memory * 100:.1f}%)")

        # Validation
        avg_val_loss, _ = evaluate_model(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            device=device,
            config=config,
            adjacency_matrix=adjacency_matrix,
            data_scaler=data_scaler
        )
        val_losses.append(avg_val_loss)

        # Early stopping check
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            no_improve = 0

            # Save best model if model_dir is available
            if model_dir is not None:
                try:
                    timestamp = get_maputo_timestamp()
                    model_path = os.path.join(model_dir, f'best_model_{timestamp}.pth')
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_loss,
                        'config': {k: v for k, v in vars(config).items() if not k.startswith('_')}
                    }, model_path)
                except Exception as e:
                    print(f"Warning: Could not save model: {str(e)}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

        # Update learning rate
        if config.scheduler_type == 'cosine_warmup':
            scheduler.step()
        elif scheduler and config.scheduler_type == 'plateau':
            scheduler.step(avg_val_loss)
        elif scheduler:  # Other scheduler types
            scheduler.step()

        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

    # Plot training history if results_dir is available
    if hasattr(config, 'results_dir') and config.results_dir is not None:
        try:
            plot_training_history(train_losses, val_losses, config.results_dir)
        except Exception as e:
            print(f"Warning: Could not plot training history: {str(e)}")

    return model, train_losses, val_losses


def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: Callable,
    data_scaler: MinMaxScaler | StandardScaler | RobustScaler,
    device: str = 'cpu',
    config: Optional[TrainingConfig] = None,
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[float, Tuple[float, float, float, float]]:
    """
    Evaluate the model and calculate metrics

    Args:
        model: Model to evaluate
        dataloader: DataLoader with evaluation data
        criterion: Loss function
        device: Device to evaluate on
        config: Training configuration
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (average loss, (MAE, RMSE, R², MAPE))
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    use_quantile_regression = config.use_quantile_regression if config else False
    loss_function_type = config.loss_function if config else 'mse'

    # Use mixed precision for evaluation if enabled
    use_amp = config and config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        if use_quantile_regression:
                            quantiles = config.quantiles
                            loss = quantile_loss(output, target, quantiles)
                        elif loss_function_type == 'hybrid':
                            loss = hybrid_loss(output, target)
                        else:
                            loss = criterion(output, target)
            else:
                if config and config.use_spatial_features and config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

                if use_quantile_regression:
                    quantiles = config.quantiles
                    loss = quantile_loss(output, target, quantiles)
                elif loss_function_type == 'hybrid':
                    loss = hybrid_loss(output, target)
                else:
                    loss = criterion(output, target)

            total_loss += loss.item()
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    save_predictions_and_actuals(
        predictions=predictions,
        actuals=actuals,
        results_dir=config.results_dir,
        filename="scaled_predictions"
    )

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)


    # Inverse transform
    predictions_inv = data_scaler.inverse_transform(predictions_2d)
    actuals_inv = data_scaler.inverse_transform(actuals_2d)

    save_predictions_and_actuals(
        predictions=predictions_inv,
        actuals=actuals_inv,
        results_dir=config.results_dir,
        filename="inverse_scale_predictions"
    )


    # Calculate metrics
    mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
    rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
    r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())
    mape = robust_mape(actuals_inv.ravel(), predictions_inv.ravel())

    print(f'MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.2f}, MAPE: {mape:.2f}%')

    return total_loss / len(dataloader), (mae, rmse, r2, mape)


def predict(
    model: nn.Module,
    dataloader: DataLoader,
    scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None,
    config: Optional[TrainingConfig] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Make predictions using the trained model and inverse transform the results

    Args:
        model: Trained model
        dataloader: DataLoader with test data
        scaler: Scaler used for normalization
        device: Device to use for prediction
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (predictions, actuals) in original scale
    """
    model.eval()
    all_preds = []
    all_targets = []

    # Use AMP for prediction if available on GPU
    use_amp = hasattr(torch.cuda, 'amp') and device != 'cpu'

    with torch.no_grad():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            if use_amp:
                if DEVICE_TYPE_SUPPORTED:
                    with autocast(device_type='cuda'):
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
                else:
                    # Older PyTorch versions
                    with autocast():
                        if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)
            else:
                if hasattr(dataloader.dataset, 'config') and dataloader.dataset.config.use_spatial_features and dataloader.dataset.config.use_gnn_pre_transformer:
                    output = model(data, adjacency_matrix.to(device))
                else:
                    output = model(data)

            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    predictions = np.concatenate(all_preds)
    actuals = np.concatenate(all_targets)

    # Reshape for inverse transformation
    num_samples, pred_window, num_features = predictions.shape
    predictions_2d = predictions.reshape(-1, num_features)
    actuals_2d = actuals.reshape(-1, num_features)

    # Inverse transform
    predictions_inv = scaler.inverse_transform(predictions_2d)
    actuals_inv = scaler.inverse_transform(actuals_2d)

    # save_predictions_and_actuals(
    #     predictions=predictions_inv,
    #     actuals=actuals_inv,
    #     results_dir=config.output_dir,
    #     filename="original_scale_predictions"
    # )

    return predictions_inv, actuals_inv


def evaluate_baseline_model(predictions: np.ndarray, actuals: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Evaluate baseline model predictions against actuals

    Args:
        predictions: Predicted values
        actuals: Actual values

    Returns:
        Tuple of (MAE, RMSE, R², MAPE)
    """
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())
    return mae, rmse, r2, mape

def train_baseline_models(
    train_data: np.ndarray,
    test_data: np.ndarray,
    config: TrainingConfig,
    device: str,
    timestamps_train: Optional[pd.DatetimeIndex] = None,
    timestamps_test: Optional[pd.DatetimeIndex] = None
) -> Dict[str, List[Tuple[float, float, float, float]]]:
    """Return empty baseline metrics - ARIMA, Exp Smoothing and LSTM baselines removed"""
    print("Baseline models (ARIMA, ExpSmoothing, LSTM) have been disabled")

    # Return empty dictionary of metrics
    return {'naive_forecast': []}

# --- Loss Functions ---
def quantile_loss(output: torch.Tensor, target: torch.Tensor, quantiles: List[float]) -> torch.Tensor:
    """
    Quantile Loss function for prediction intervals

    Args:
        output: Model output with shape [batch, pred_len, num_quantiles]
        target: Target values
        quantiles: List of quantiles

    Returns:
        Quantile loss value
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - output[:, :, i]
        losses.append(torch.max((q-1) * errors, q * errors).mean())
    loss = torch.sum(torch.stack(losses))
    return loss


def hybrid_loss(output: torch.Tensor, target: torch.Tensor, alpha: float = 0.5) -> torch.Tensor:
    """
    Hybrid Loss function: Weighted combination of MSE and MAE

    Args:
        output: Model output
        target: Target values
        alpha: Weight for MSE component (1-alpha for MAE)

    Returns:
        Hybrid loss value
    """
    mse_loss = nn.MSELoss()(output, target)
    mae_loss = nn.L1Loss()(output, target)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    return loss


def robust_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-8) -> float:
    """
    Robust MAPE to handle division by zero and near-zero values

    Args:
        y_true: True values
        y_pred: Predicted values
        epsilon: Small value to prevent division by zero

    Returns:
        MAPE value as percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan  # Or 0, or another appropriate value if all y_true are zero
    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]
    return np.mean(np.abs((y_true_masked - y_pred_masked) / (y_true_masked + epsilon))) * 100


##Transfer Learning Training Module

In [ ]:

# =============================================================================
# Transfer Learning Training Module
# =============================================================================

import contextlib

def train_transfer_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    source_loader: Optional[DataLoader] = None,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,
    criterion: Optional[Callable] = None,
    config: TrainingConfig = None,
    data_scaler: Optional[Union[MinMaxScaler, StandardScaler, RobustScaler]] = None,
    source_scaler: Optional[Union[MinMaxScaler, StandardScaler, RobustScaler]] = None,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None
) -> Tuple[nn.Module, Dict[str, List[float]], Dict[str, Dict[str, float]]]:
    """
    Train a model using transfer learning from a pre-trained model with improved
    adapter-based architecture and progressive unfreezing.

    Args:
        model: Pre-trained model to fine-tune
        train_loader: DataLoader with training data from target dataset
        val_loader: DataLoader with validation data from target dataset
        source_loader: Optional DataLoader with test data from source dataset (for comparison)
        optimizer: Optional optimizer (will be created if None)
        scheduler: Optional learning rate scheduler (will be created if None)
        criterion: Optional loss function (will be created if None)
        config: Training configuration
        data_scaler: Scaler used for the target dataset
        source_scaler: Scaler used for the source dataset (required if source_loader is provided)
        device: Device to train on ('cpu' or 'cuda')
        adjacency_matrix: Optional adjacency matrix for GNN

    Returns:
        Tuple of (fine-tuned model, training history, evaluation metrics)
    """
    if config is None:
        raise ValueError("Configuration object is required for transfer learning")

    # Setup directories for saving results
    output_dir = config.output_dir
    results_dir = config.results_dir
    model_dir = config.model_dir

    # Get training parameters specific to transfer learning
    num_epochs = min(config.num_epochs, 150)  # Reasonable limit for fine-tuning
    patience = config.patience
    learning_rate = config.transfer_learning_rate if hasattr(config, 'transfer_learning_rate') else 5e-5
    target_dataset_name = config.target_dataset_name if hasattr(config, 'target_dataset_name') else "target"

    # Print transfer learning setup
    print(f"\n=== Transfer Learning Setup ({target_dataset_name}) ===")
    print(f"Learning Rate: {learning_rate}")
    print(f"Max Epochs: {num_epochs}")
    print(f"Patience: {patience}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Has adapters: {hasattr(model, 'has_transfer_adapters') and model.has_transfer_adapters}")

    # Create optimizer if not provided (only optimize parameters that require gradients)
    if optimizer is None:
        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate,
            weight_decay=0.01  # Slightly stronger regularization for fine-tuning
        )

    # Create scheduler if not provided
    if scheduler is None:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=patience//2, factor=0.5, verbose=True
        )

    # Create criterion if not provided
    if criterion is None:
        if config.loss_function == 'mse':
            criterion = nn.MSELoss()
        elif config.loss_function == 'mae':
            criterion = nn.L1Loss()
        elif config.loss_function == 'huber':
            criterion = nn.SmoothL1Loss()
        else:
            criterion = nn.MSELoss()

    # Initialize tracking variables
    best_loss = float('inf')
    no_improve = 0
    history = {
        'train_loss': [],
        'val_loss': []
    }
    metrics = {
        'source': None,
        'target': None
    }

    # Best model state dictionary
    best_model_state = None

    # Setup for mixed precision training
    use_amp = config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        try:
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

            # Fix: Use device.type if device is torch.device
            if isinstance(device, torch.device):
                device_type = device.type
            else:
                # Handle string device
                device_type = device.split(':')[0] if ':' in device else device

            autocast_context = lambda: torch.amp.autocast(device_type=device_type, enabled=use_amp)
            print(f"Using mixed precision with device type: {device_type}")
        except Exception as amp_err:
            print(f"Error setting up mixed precision: {amp_err}")
            scaler = None
            autocast_context = lambda: contextlib.nullcontext()
            use_amp = False
    else:
        scaler = None
        autocast_context = lambda: contextlib.nullcontext()

    try:
        # Check if model has adapters
        has_adapters = hasattr(model, 'has_transfer_adapters') and model.has_transfer_adapters

        # === STAGE 1: First train adapters or input/output layers only ===
        print("\n=== Transfer Learning Stage 1: Adapter/IO Layer Training ===")

        # Store original requires_grad status
        original_requires_grad = {}
        for name, param in model.named_parameters():
            original_requires_grad[name] = param.requires_grad

        # Freeze everything first
        for param in model.parameters():
            param.requires_grad = False

        if has_adapters:
            # Unfreeze input adapter
            if hasattr(model, 'input_adapter'):
                for param in model.input_adapter.parameters():
                    param.requires_grad = True
                print("  Unfroze input adapter")

            # Unfreeze output adapter in decoder if it's a Sequential
            if isinstance(model.decoder, nn.Sequential) and len(list(model.decoder.children())) > 1:
                # Assuming last layer is the adapter
                output_adapter = list(model.decoder.children())[-1]
                for param in output_adapter.parameters():
                    param.requires_grad = True
                print("  Unfroze output adapter")
        else:
            # If no adapters, train input embedding and output decoder
            for param in model.embedding.parameters():
                param.requires_grad = True

            for param in model.decoder.parameters():
                param.requires_grad = True
            print("  Unfroze input embedding and decoder")

        # Create Stage 1 optimizer with higher learning rate
        stage1_optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate * 5.0,  # Higher LR for adapters/IO
            weight_decay=0.01
        )

        # Stage 1 training - just a few epochs
        stage1_epochs = min(20, num_epochs // 3)

        for epoch in range(stage1_epochs):
            # Training phase
            model.train()
            train_loss = 0

            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)

                # Zero gradients
                stage1_optimizer.zero_grad()

                # Forward pass with mixed precision if enabled
                with autocast_context():
                    if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                        output = model(data, adjacency_matrix.to(device))
                    else:
                        output = model(data)

                    loss = criterion(output, target)

                # Backward pass
                if use_amp:
                    scaler.scale(loss).backward()
                    if config.gradient_clip:
                        scaler.unscale_(stage1_optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                    scaler.step(stage1_optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    if config.gradient_clip:
                        nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                    stage1_optimizer.step()

                train_loss += loss.item()

            # Calculate average loss
            avg_train_loss = train_loss / len(train_loader)
            history['train_loss'].append(avg_train_loss)

            # Validation phase
            model.eval()
            val_loss = 0

            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(device), target.to(device)

                    with autocast_context():
                        if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        loss = criterion(output, target)

                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            history['val_loss'].append(avg_val_loss)

            # Early stopping check
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                no_improve = 0
                best_model_state = model.state_dict()
            else:
                no_improve += 1
                if no_improve >= patience // 2:  # Use shorter patience for Stage 1
                    print(f"  Early stopping Stage 1 at epoch {epoch+1}/{stage1_epochs}")
                    break

            print(f"Stage 1 - Epoch {epoch+1}/{stage1_epochs} | "
                  f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

        # === STAGE 2: Progressive Unfreezing of Transformer Layers ===
        print("\n=== Transfer Learning Stage 2: Progressive Unfreezing ===")

        # Determine number of transformer layers
        transformer_layers = [layer for layer in model.transformer]
        num_layers = len(transformer_layers)

        # Calculate epochs for progressive unfreezing
        remaining_epochs = num_epochs - (epoch + 1)  # Account for epochs already spent
        epochs_per_layer = max(5, remaining_epochs // (num_layers + 1))  # +1 for embedding

        # Unfreeze layers from last to first (closest to output first)
        for layer_idx in range(num_layers - 1, -1, -1):
            print(f"\n  --- Unfreezing Transformer Layer {layer_idx + 1}/{num_layers} ---")

            # Unfreeze this layer
            for param in transformer_layers[layer_idx].parameters():
                param.requires_grad = True

            # Create optimizer for this stage with declining learning rate
            # The deeper we go, the smaller the learning rate should be
            layer_lr = learning_rate * (0.7 ** (num_layers - layer_idx))
            stage2_optimizer = optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=layer_lr,
                weight_decay=0.01
            )

            # Create scheduler
            stage2_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                stage2_optimizer, mode='min', patience=patience//2, factor=0.5
            )

            # Train for this layer
            layer_best_loss = float('inf')
            layer_no_improve = 0

            for epoch in range(epochs_per_layer):
                # Training phase
                model.train()
                train_loss = 0

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(device), target.to(device)

                    # Zero gradients
                    stage2_optimizer.zero_grad()

                    # Forward pass with mixed precision if enabled
                    with autocast_context():
                        if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        loss = criterion(output, target)

                    # Backward pass
                    if use_amp:
                        scaler.scale(loss).backward()
                        if config.gradient_clip:
                            scaler.unscale_(stage2_optimizer)
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                        scaler.step(stage2_optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        if config.gradient_clip:
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                        stage2_optimizer.step()

                    train_loss += loss.item()

                # Calculate average loss
                avg_train_loss = train_loss / len(train_loader)
                history['train_loss'].append(avg_train_loss)

                # Validation phase
                model.eval()
                val_loss = 0

                with torch.no_grad():
                    for data, target in val_loader:
                        data, target = data.to(device), target.to(device)

                        with autocast_context():
                            if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                                output = model(data, adjacency_matrix.to(device))
                            else:
                                output = model(data)

                            loss = criterion(output, target)

                        val_loss += loss.item()

                avg_val_loss = val_loss / len(val_loader)
                history['val_loss'].append(avg_val_loss)

                # Update layer scheduler
                stage2_scheduler.step(avg_val_loss)

                # Early stopping check
                if avg_val_loss < layer_best_loss:
                    layer_best_loss = avg_val_loss
                    layer_no_improve = 0

                    # Update overall best if this is better
                    if avg_val_loss < best_loss:
                        best_loss = avg_val_loss
                        best_model_state = model.state_dict()
                else:
                    layer_no_improve += 1
                    if layer_no_improve >= patience // 2:  # Shorter patience for each layer
                        print(f"    Early stopping layer training at epoch {epoch+1}/{epochs_per_layer}")
                        break

                print(f"    Layer {layer_idx+1} - Epoch {epoch+1}/{epochs_per_layer} | "
                      f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

        # === STAGE 3: Optional Final Embedding Layer Unfreezing ===
        # Only if we have remaining epochs and patience
        remaining_epochs = num_epochs - len(history['train_loss'])

        if remaining_epochs > 5 and no_improve < patience:
            print("\n=== Transfer Learning Stage 3: Embedding Layer Unfreezing ===")

            # Unfreeze embedding layer
            for param in model.embedding.parameters():
                param.requires_grad = True

            print("  Unfroze embedding layer for final fine-tuning")

            # Use very small learning rate for full model
            final_optimizer = optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=learning_rate * 0.1,  # Very small LR for full model
                weight_decay=0.01
            )

            # Create scheduler
            final_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                final_optimizer, mode='min', patience=patience//2, factor=0.5
            )

            # Reset early stopping for this stage
            no_improve = 0

            # Train final stage
            for epoch in range(remaining_epochs):
                # Training phase
                model.train()
                train_loss = 0

                for batch_idx, (data, target) in enumerate(train_loader):
                    data, target = data.to(device), target.to(device)

                    # Zero gradients
                    final_optimizer.zero_grad()

                    # Forward pass with mixed precision if enabled
                    with autocast_context():
                        if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                            output = model(data, adjacency_matrix.to(device))
                        else:
                            output = model(data)

                        loss = criterion(output, target)

                    # Backward pass
                    if use_amp:
                        scaler.scale(loss).backward()
                        if config.gradient_clip:
                            scaler.unscale_(final_optimizer)
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                        scaler.step(final_optimizer)
                        scaler.update()
                    else:
                        loss.backward()
                        if config.gradient_clip:
                            nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                        final_optimizer.step()

                    train_loss += loss.item()

                # Calculate average loss
                avg_train_loss = train_loss / len(train_loader)
                history['train_loss'].append(avg_train_loss)

                # Validation phase
                model.eval()
                val_loss = 0

                with torch.no_grad():
                    for data, target in val_loader:
                        data, target = data.to(device), target.to(device)

                        with autocast_context():
                            if config.use_spatial_features and config.use_gnn_pre_transformer and adjacency_matrix is not None:
                                output = model(data, adjacency_matrix.to(device))
                            else:
                                output = model(data)

                            loss = criterion(output, target)

                        val_loss += loss.item()

                avg_val_loss = val_loss / len(val_loader)
                history['val_loss'].append(avg_val_loss)

                # Update scheduler
                final_scheduler.step(avg_val_loss)

                # Early stopping check
                if avg_val_loss < best_loss:
                    best_loss = avg_val_loss
                    no_improve = 0
                    best_model_state = model.state_dict()
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"  Early stopping at epoch {epoch+1}/{remaining_epochs}")
                        break

                print(f"Stage 3 - Epoch {epoch+1}/{remaining_epochs} | "
                      f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

        # Load best model state if available
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            print("Loaded best model state based on validation loss")

            # Save best model to file
            if model_dir:
                try:
                    timestamp = get_maputo_timestamp()
                    model_path = os.path.join(model_dir, f'fine_tuned_{target_dataset_name}_{timestamp}.pth')
                    torch.save({
                        'model_state_dict': best_model_state,
                        'target_dataset': target_dataset_name,
                        'loss': best_loss,
                        'config': {k: v for k, v in vars(config).items() if not k.startswith('_')}
                    }, model_path)
                    print(f"Saved fine-tuned model to {model_path}")
                except Exception as e:
                    print(f"Warning: Could not save model: {str(e)}")

        # Plot training history
        if results_dir:
            try:
                plt.figure(figsize=(10, 6))
                plt.plot(history['train_loss'], label='Training Loss')
                plt.plot(history['val_loss'], label='Validation Loss')
                plt.title(f'Transfer Learning to {target_dataset_name.title()} Dataset')
                plt.xlabel('Epochs')
                plt.ylabel('Loss')
                plt.legend()
                plt.grid(True, alpha=0.3)

                # Add adapter info if applicable
                has_adapters_str = "Uses adapters" if has_adapters else "No adapters"
                plt.figtext(0.02, 0.02, has_adapters_str, fontsize=10)

                plt.tight_layout()
                plt.savefig(os.path.join(results_dir, f'transfer_learning_curve_{target_dataset_name}.png'), dpi=300)
                plt.close()
                print(f"Saved training curve to {os.path.join(results_dir, f'transfer_learning_curve_{target_dataset_name}.png')}")
            except Exception as e:
                print(f"Warning: Could not plot training history: {str(e)}")

        # Evaluate on target dataset
        print(f"\n=== Evaluating Transfer Learning on {target_dataset_name} ===")
        target_metrics = evaluate_model(
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            data_scaler=data_scaler,
            device=device,
            config=config,
            adjacency_matrix=adjacency_matrix
        )[1]  # Get only the metrics tuple

        metrics['target'] = {
            'mae': target_metrics[0],
            'rmse': target_metrics[1],
            'r2': target_metrics[2],
            'mape': target_metrics[3]
        }

        # Evaluate on source dataset using the proper method
        if source_loader is not None and source_scaler is not None:
            print(f"\n=== Evaluating Transfer Learning on Source Dataset (METR-LA) ===")
            _, source_metrics = evaluate_on_source_data(
                model=model,
                source_loader=source_loader,
                source_scaler=source_scaler,
                criterion=criterion,
                device=device,
                config=config,
                source_adjacency_matrix=adjacency_matrix  # Use same adjacency matrix for now
            )

            metrics['source'] = {
                'mae': source_metrics[0],
                'rmse': source_metrics[1],
                'r2': source_metrics[2],
                'mape': source_metrics[3]
            }

        # Create comparative visualization if both metrics are available
        if metrics['source'] and metrics['target'] and results_dir:
            try:
                plt.figure(figsize=(12, 8))
                metric_names = ['mae', 'rmse', 'r2', 'mape']
                source_values = [metrics['source'][m] for m in metric_names]
                target_values = [metrics['target'][m] for m in metric_names]

                # Special handling for R² (higher is better)
                r2_idx = metric_names.index('r2')
                if source_values[r2_idx] < 0:  # If negative, use absolute value
                    display_source_r2 = abs(source_values[r2_idx])
                    display_target_r2 = abs(target_values[r2_idx])
                    r2_label = 'r2 (absolute)'
                else:
                    display_source_r2 = 1 - source_values[r2_idx]
                    display_target_r2 = 1 - target_values[r2_idx]
                    r2_label = 'r2 (inverted)'

                source_values[r2_idx] = display_source_r2
                target_values[r2_idx] = display_target_r2
                metric_names[r2_idx] = r2_label

                x = range(len(metric_names))
                width = 0.35

                plt.bar([i - width/2 for i in x], source_values, width, label='METR-LA (Source)')
                plt.bar([i + width/2 for i in x], target_values, width, label=f'{target_dataset_name.title()} (Target)')

                plt.xlabel('Metrics')
                plt.ylabel('Value (Lower is Better)')
                plt.title('Transfer Learning Performance Comparison')
                plt.xticks(x, metric_names)
                plt.legend()
                plt.grid(True, alpha=0.3)

                # Add text showing improvement/degradation percentages
                for i, (source, target) in enumerate(zip(source_values, target_values)):
                    if metric_names[i] != r2_label and source > 0:
                        change_pct = (target - source) / source * 100
                        color = 'green' if change_pct < 0 else 'red'
                        plt.annotate(
                            f"{change_pct:.1f}%",
                            xy=(i, max(source, target) * 1.05),
                            ha='center',
                            color=color
                        )

                plt.tight_layout()
                plt.savefig(os.path.join(results_dir, f'transfer_comparison_{target_dataset_name}.png'), dpi=300)
                plt.close()
                print(f"Saved transfer comparison to {os.path.join(results_dir, f'transfer_comparison_{target_dataset_name}.png')}")
            except Exception as e:
                print(f"Warning: Could not create comparison visualization: {str(e)}")

        return model, history, metrics

    except Exception as e:
        print(f"\n=== Error during transfer learning: {str(e)} ===")
        import traceback
        traceback.print_exc()

        # Return the model in its current state along with partial history/metrics
        return model, history, metrics

def evaluate_on_source_data(model, source_loader, source_scaler,
                            criterion, device, config, source_adjacency_matrix=None):
    """
    Properly evaluate a transfer-learned model on source data
    by temporarily removing adapters
    """
    model.eval()

    # Store whether model has adapters
    has_adapters = hasattr(model, 'has_transfer_adapters') and model.has_transfer_adapters

    # If model has adapters, temporarily remove them
    if has_adapters:
        print("Temporarily removing adapters for source evaluation...")
        # Store original methods and attributes
        original_forward = model.forward
        original_decoder = model.decoder
        original_num_features = model.num_features

        # IMPORTANT: Store the source output dimension to use during evaluation
        source_output_dim = model.source_output_dim if hasattr(model, 'source_output_dim') else 207

        # Restore original methods (bypassing adapters)
        if hasattr(model, '_original_forward'):
            model.forward = model._original_forward
            # CRITICAL FIX: Update num_features to source dimension
            model.num_features = source_output_dim
            print(f"Restored original forward method with num_features={model.num_features}")

        if hasattr(model, 'original_decoder'):
            model.decoder = model.original_decoder
            print("Restored original decoder")

    # Evaluate normally
    try:
        total_loss = 0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in source_loader:
                data, target = data.to(device), target.to(device)

                # Debug input data shape
                if len(all_preds) == 0:
                    print(f"Source evaluation - Input shape: {data.shape}, Target shape: {target.shape}")
                    print(f"Current model.num_features: {model.num_features}, model.pred_len: {model.pred_len}")

                # Forward pass (using original architecture)
                if config.use_spatial_features and config.use_gnn_pre_transformer and source_adjacency_matrix is not None:
                    output = model(data, source_adjacency_matrix.to(device))
                else:
                    output = model(data)

                # Debug output shape
                if len(all_preds) == 0:
                    print(f"Model output shape before view/reshape: {output.shape}")

                loss = criterion(output, target)
                total_loss += loss.item()

                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        # Calculate metrics
        predictions = np.concatenate(all_preds)
        actuals = np.concatenate(all_targets)

        # Debug shapes after collection
        print(f"Source evaluation - Collected predictions shape: {predictions.shape}, actuals shape: {actuals.shape}")

        # Reshape and inverse transform
        predictions_2d = predictions.reshape(-1, predictions.shape[-1])
        actuals_2d = actuals.reshape(-1, actuals.shape[-1])

        # Inverse transform
        predictions_inv = source_scaler.inverse_transform(predictions_2d)
        actuals_inv = source_scaler.inverse_transform(actuals_2d)

        # Calculate metrics
        mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
        rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
        r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())
        mape = robust_mape(actuals_inv.ravel(), predictions_inv.ravel())

        # Return the same format as evaluate_model
        metrics = (mae, rmse, r2, mape)
        avg_loss = total_loss / len(source_loader)

    finally:
        # Restore adapters if they were removed
        if has_adapters:
            model.forward = original_forward
            model.decoder = original_decoder
            model.num_features = original_num_features
            print(f"Restored adapters after source evaluation (num_features={model.num_features})")

    print(f"Source Data Metrics - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}, MAPE: {mape:.2f}%")
    return avg_loss, metrics

def predict(
    model: nn.Module,
    dataloader: DataLoader,
    scaler: Any,
    device: str = 'cpu',
    adjacency_matrix: Optional[torch.Tensor] = None,
    config: Optional[TrainingConfig] = None,
    is_source_dataset: bool = False  # New parameter to identify dataset type
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Make predictions using the trained model and inverse transform the results

    Args:
        model: Trained model
        dataloader: DataLoader with test data
        scaler: Scaler used for normalization
        device: Device to use for prediction
        adjacency_matrix: Optional adjacency matrix for GNN
        config: Optional configuration object
        is_source_dataset: Whether this is the source dataset (for adapter handling)

    Returns:
        Tuple of (predictions, actuals) in original scale
    """
    import contextlib

    model.eval()
    all_preds = []
    all_targets = []

    torch_device = torch.device(device)

    # Check if we need to temporarily remove adapters for source dataset prediction
    has_adapters = hasattr(model, 'has_transfer_adapters') and model.has_transfer_adapters
    original_forward = None
    original_decoder = None
    original_num_features = None

    if has_adapters and is_source_dataset:
        print("Temporarily removing adapters for source dataset prediction...")
        # Store original methods and attributes
        original_forward = model.forward
        original_decoder = model.decoder
        original_num_features = model.num_features

        # Set source output dimension
        source_output_dim = model.source_output_dim if hasattr(model, 'source_output_dim') else 207

        # Restore original methods
        if hasattr(model, '_original_forward'):
            model.forward = model._original_forward
            model.num_features = source_output_dim
            print(f"Using original forward method with num_features={model.num_features}")

        if hasattr(model, 'original_decoder'):
            model.decoder = model.original_decoder
            print("Using original decoder")

    # Setup autocast context
    use_amp = config and config.use_mixed_precision and AMP_AVAILABLE and device != 'cpu'
    if use_amp:
        try:
            autocast_context = lambda: torch.amp.autocast(device_type=torch_device.type, enabled=use_amp)
            print(f"Predict: Using torch.amp.autocast with device_type='{torch_device.type}'")
        except TypeError:
            autocast_context = lambda: torch.cuda.amp.autocast(enabled=use_amp)
            print("Predict: Using torch.cuda.amp.autocast (fallback)")
        except Exception as e_amp:
            print(f"Predict: Failed to initialize AMP context: {e_amp}. Disabling AMP for prediction.")
            use_amp = False
            autocast_context = lambda: contextlib.nullcontext()
    else:
        autocast_context = lambda: contextlib.nullcontext()

    try:
        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(device), target.to(device)

                # Debug shape on first batch
                if len(all_preds) == 0:
                    print(f"Prediction input shape: {data.shape}, target shape: {target.shape}")
                    print(f"Model using num_features: {model.num_features}, pred_len: {model.pred_len}")
                    if has_adapters:
                        print(f"Adapter mode: {'source (original forward)' if is_source_dataset else 'target (with adapters)'}")

                with autocast_context():
                    # Determine if GNN should be used
                    use_gnn = False
                    current_config = getattr(dataloader.dataset, 'config', config)
                    if current_config and current_config.use_spatial_features and current_config.use_gnn_pre_transformer:
                         use_gnn = True

                    if use_gnn and adjacency_matrix is not None:
                        output = model(data, adjacency_matrix.to(device))
                    else:
                        output = model(data)

                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        predictions = np.concatenate(all_preds)
        actuals = np.concatenate(all_targets)

        # Debug final shapes
        print(f"Collected predictions shape: {predictions.shape}, actuals shape: {actuals.shape}")

        # Reshape for inverse transformation
        num_samples, pred_window, num_features = predictions.shape
        predictions_2d = predictions.reshape(-1, num_features)
        actuals_2d = actuals.reshape(-1, num_features)

        # Inverse transform
        predictions_inv = scaler.inverse_transform(predictions_2d)
        actuals_inv = scaler.inverse_transform(actuals_2d)

        return predictions_inv, actuals_inv

    finally:
        # Restore adapters if they were temporarily removed
        if has_adapters and is_source_dataset and original_forward is not None:
            model.forward = original_forward
            model.decoder = original_decoder
            model.num_features = original_num_features
            print(f"Restored adapters after source prediction (num_features={model.num_features})")


def evaluate_baseline_model(predictions: np.ndarray, actuals: np.ndarray) -> Tuple[float, float, float, float]:
    """
    Evaluate baseline model predictions against actuals

    Args:
        predictions: Predicted values
        actuals: Actual values

    Returns:
        Tuple of (MAE, RMSE, R², MAPE)
    """
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())
    mape = robust_mape(actuals.ravel(), predictions.ravel())
    return mae, rmse, r2, mape

def train_baseline_models(
    train_data: np.ndarray,
    test_data: np.ndarray,
    config: TrainingConfig,
    device: str,
    timestamps_train: Optional[pd.DatetimeIndex] = None,
    timestamps_test: Optional[pd.DatetimeIndex] = None
) -> Dict[str, List[Tuple[float, float, float, float]]]:
    """Return empty baseline metrics - ARIMA, Exp Smoothing and LSTM baselines removed"""
    print("Baseline models (ARIMA, ExpSmoothing, LSTM) have been disabled")

    # Return empty dictionary of metrics
    return {'naive_forecast': []}

# --- Loss Functions ---
def quantile_loss(output: torch.Tensor, target: torch.Tensor, quantiles: List[float]) -> torch.Tensor:
    """
    Quantile Loss function for prediction intervals

    Args:
        output: Model output with shape [batch, pred_len, num_quantiles]
        target: Target values
        quantiles: List of quantiles

    Returns:
        Quantile loss value
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - output[:, :, i]
        losses.append(torch.max((q-1) * errors, q * errors).mean())
    loss = torch.sum(torch.stack(losses))
    return loss


def hybrid_loss(output: torch.Tensor, target: torch.Tensor, alpha: float = 0.5) -> torch.Tensor:
    """
    Hybrid Loss function: Weighted combination of MSE and MAE

    Args:
        output: Model output
        target: Target values
        alpha: Weight for MSE component (1-alpha for MAE)

    Returns:
        Hybrid loss value
    """
    mse_loss = nn.MSELoss()(output, target)
    mae_loss = nn.L1Loss()(output, target)
    loss = alpha * mse_loss + (1 - alpha) * mae_loss
    return loss


def robust_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-8) -> float:
    """
    Robust MAPE to handle division by zero and near-zero values

    Args:
        y_true: True values
        y_pred: Predicted values
        epsilon: Small value to prevent division by zero

    Returns:
        MAPE value as percentage
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan  # Or 0, or another appropriate value if all y_true are zero
    y_true_masked = y_true[mask]
    y_pred_masked = y_pred[mask]
    return np.mean(np.abs((y_true_masked - y_pred_masked) / (y_true_masked + epsilon))) * 100


## Visualization Module

In [ ]:


# =============================================================================
# Visualization Module
# =============================================================================

def plot_attention_weights(
    model: nn.Module,
    seq_length: int,
    results_dir: str,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots attention weights with timestamped filename and model details
    """
    plt.figure(figsize=(12, 10))  # Larger figure to accommodate details

    # Use 4/5 of the figure for the attention plot
    plt.subplot(5, 1, (1, 4))

    if not hasattr(model, 'attention_weights') or model.attention_weights is None:
        plt.text(0.5, 0.5, "No attention weights captured", ha='center', va='center', fontsize=14)
        plt.title('Attention Weights - Not Available')
    else:
        try:
            # Get attention weights and ensure proper shape
            weights = model.attention_weights

            # Convert to numpy if it's a tensor
            if isinstance(weights, torch.Tensor):
                weights = weights.cpu().detach().numpy()

            # Handle different possible shapes
            if len(weights.shape) == 3:  # [batch, seq, seq]
                weights = weights[0] if weights.shape[0] == 1 else np.mean(weights, axis=0)
            elif len(weights.shape) == 2 and weights.shape[1] == 1:  # [seq, 1] shape
                weights = np.tile(weights, (1, seq_length))

            # Create heatmap
            sns.heatmap(weights, cmap='viridis',
                        xticklabels=range(1, weights.shape[1] + 1),
                        yticklabels=range(1, weights.shape[0] + 1))
            plt.title('Attention Weights - Last Layer', fontsize=16)
            plt.xlabel('Key Positions')
            plt.ylabel('Query Positions')

        except Exception as e:
            # Provide error information in the plot
            plt.clf()
            plt.text(0.5, 0.5, f"Error plotting attention weights: {str(e)}\n"
                               f"Shape: {getattr(model.attention_weights, 'shape', 'unknown')}",
                     ha='center', va='center', wrap=True)
            plt.title('Attention Visualization Error')

    # Add model details at the bottom 1/5 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Seq Length: {seq_length} | "
            f"Prediction Length: {config.pred_length} | Dropout: {config.dropout} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(5, 1, 5)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'attention_heatmap_{timestamp}.png'), dpi=300)
    plt.close()


def plot_predictions_vs_actual(
    actuals: np.ndarray,
    predictions: np.ndarray,
    sensor_index: int,
    sensor_id: str,  # Parameter for actual sensor ID
    fold: int,
    results_dir: str,
    pred_len: int,
    config: Optional['TrainingConfig'] = None, # Use string hint if TrainingConfig defined later
) -> None:
    """
    Plots actual vs predicted traffic flow with enhanced details using actual sensor ID.
    Compares the first 288 (24 hours) time steps or the available length if shorter.

    Args:
        actuals: Numpy array of actual values [num_samples, num_sensors].
        predictions: Numpy array of predicted values [num_samples, num_sensors].
        sensor_index: The column index of the sensor to plot.
        sensor_id: The actual string identifier of the sensor (e.g., '773869').
        fold: The cross-validation fold number (0-based).
        results_dir: The directory to save the plot image.
        pred_len: The prediction length (horizon) used by the model.
        config: The TrainingConfig object containing model hyperparameters.
    """
    # Ensure sensor_index is valid before accessing data
    if sensor_index >= actuals.shape[1] or sensor_index >= predictions.shape[1]:
        print(f"Warning: sensor_index {sensor_index} is out of bounds for data shapes "
               f"actuals:{actuals.shape}, predictions:{predictions.shape}. Skipping plot for sensor ID {sensor_id}.")
        return
    if actuals.shape[0] == 0 or predictions.shape[0] == 0:
        print(f"Warning: Actuals or predictions array is empty. Skipping plot for sensor ID {sensor_id}.")
        return

    # Determine the number of time steps to plot (up to 288)
    plot_length = min(288, actuals.shape[0], predictions.shape[0])

    # Calculate R-squared specifically for this plot (subset of data)
    # Ensure there are enough points to calculate R2 score
    if plot_length <= 1:
        print(f"Warning: Not enough data points ({plot_length}) to calculate R-squared for sensor {sensor_id}. Setting R2 to N/A.")
        r2 = np.nan
    else:
        try:
            # Slice data up to plot_length for R2 calculation
            r2 = r2_score(actuals[:plot_length, sensor_index], predictions[:plot_length, sensor_index])
        except ValueError as e:
            # Catch potential errors during R2 calculation (e.g., constant input)
            print(f"Warning: Could not calculate R-squared for sensor {sensor_id}. Error: {e}. Setting R2 to NaN.")
            r2 = np.nan

    plt.figure(figsize=(14, 8))

    # --- Main Plot ---
    plt.subplot(4, 1, (1, 3))  # Use top 3/4 of the figure for the main plot

    plt.plot(actuals[:plot_length, sensor_index], label='Actual', linewidth=2)
    plt.plot(predictions[:plot_length, sensor_index], label='Predicted', linewidth=2, alpha=0.8)

    # Format R-squared part of the title, handling potential NaN
    title_r2_part = f'R² = {r2:.4f}' if not np.isnan(r2) else 'R² = N/A'
    # Use actual sensor_id in the title
    plt.title(f'Fold {fold+1} - Actual vs Predicted Traffic Flow (Sensor {sensor_id})\n'
              f'{title_r2_part}', fontsize=14, fontweight='bold')

    # plt.xlabel(f'Time Steps (First {plot_length})')
    plt.ylabel('Speed (mph)')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim(0, plot_length) # Ensure x-axis limit matches plotted data

    # --- Model Details Section ---
    plt.subplot(4, 1, 4) # Use the bottom 1/4 of the figure for model details
    plt.axis('off')     # Turn off axes for the text area
    model_details = "Configuration details not available." # Default text
    if config:
        # Format model details string if config is provided
        model_details = (
            f"Model: Transformer | Layers: {config.num_layers} | Heads: {config.num_heads} | "
            f"Hidden Dim: {config.hidden_dim} | Prediction Length: {pred_len} | \n" # Added newline for better wrapping
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Loss: {config.loss_function} | Time Features: {config.use_time_features} | \n" # Added newline
            f"Batch Size: {config.batch_size} | Generated: {get_maputo_timestamp()}"
        )
    # Display the model details text
    plt.text(0.01, 0.9, model_details, wrap=True, fontsize=9, va='top') # Adjust vertical alignment

    # --- Save the Plot ---
    timestamp = get_maputo_timestamp()
    # Sanitize sensor_id for use in filename (replace non-alphanumeric chars with '_')
    safe_sensor_id = "".join(c if c.isalnum() else "_" for c in str(sensor_id)) # Ensure sensor_id is string
    # Construct filename using the sanitized sensor ID
    filename = f'predictions_fold{fold+1}_sensor_{safe_sensor_id}_step{pred_len}_{timestamp}.png'
    save_path = os.path.join(results_dir, filename)

    try:
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout slightly to prevent title overlap
        plt.savefig(save_path, dpi=300)
        # print(f"Saved plot: {save_path}") # Optional: uncomment for verbose output
    except Exception as e:
        print(f"Error saving plot {save_path}: {e}")
    finally:
        plt.close() # Ensure the figure is closed to free memory


def plot_training_history(
    train_losses: List[float],
    val_losses: List[float],
    results_dir: str,
    config: Optional[TrainingConfig] = None  # Add config parameter
) -> None:
    """
    Plots training and validation loss history with enhanced details
    """
    plt.figure(figsize=(12, 8))

    # Use 3/4 of the figure for the main plot
    plt.subplot(4, 1, (1, 3))
    plt.plot(train_losses, label='Training Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.title('Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Calculate improvement metrics
    if len(train_losses) > 0:
        initial_loss = train_losses[0]
        final_loss = train_losses[-1]
        best_val_loss = min(val_losses) if val_losses else 0

        improvement = 100 * (initial_loss - final_loss) / initial_loss if initial_loss > 0 else 0

        # Add text annotation for improvement
        plt.annotate(
            f'Training loss reduced by {improvement:.2f}%',
            xy=(len(train_losses) * 0.6, (initial_loss + final_loss) / 2),
            xytext=(len(train_losses) * 0.4, final_loss + (initial_loss - final_loss) * 0.6),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
            fontsize=10
        )

    # Add model details at the bottom 1/4 of the figure
    model_details = ""
    if config:
        model_details = (
            f"Model: Transformer | Epochs: {len(train_losses)} | Batch Size: {config.batch_size} | "
            f"Learning Rate: {config.learning_rate} | Optimizer: {config.optimizer_type} | "
            f"Layers: {config.num_layers} | Heads: {config.num_heads} | Hidden Dim: {config.hidden_dim} | "
            f"Loss Function: {config.loss_function} | Scheduler: {config.scheduler_type or 'None'} | "
            f"Generated: {get_maputo_timestamp()}"
        )

    plt.subplot(4, 1, 4)
    plt.axis('off')
    plt.text(0.01, 0.5, model_details, wrap=True, fontsize=9)

    # Save with timestamp
    timestamp = get_maputo_timestamp()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, f'training_history_{timestamp}.png'), dpi=300)
    plt.close()

def create_summary_comparison_plot(
    transformer_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    config: TrainingConfig
) -> None:
    """Creates a simplified comparison plot without the removed baselines"""
    plt.figure(figsize=(12, 8))

    # Calculate average metrics for transformer
    avg_transformer = np.mean(transformer_metrics, axis=0)

    # Plot only transformer metrics
    plt.subplot(2, 1, 1)
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']
    metrics_values = avg_transformer

    x = range(len(metrics_names))
    bars = plt.bar(x, metrics_values)
    plt.xticks(x, metrics_names)
    plt.title('Transformer Model Performance Metrics')

    # Add values on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)

    # Add model configuration info
    plt.subplot(2, 1, 2)
    plt.axis('off')
    model_info = (
        f"MODEL CONFIGURATION\n\n"
        f"Model: Transformer\n"
        f"Layers: {config.num_layers}\n"
        f"Attention Heads: {config.num_heads}\n"
        f"Hidden Dimension: {config.hidden_dim}\n"
        f"Prediction Length: {config.pred_length}\n"
        f"Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}\n"
        f"Loss Function: {config.loss_function}\n"
        f"Using {config.data_scaler_type} scaling"
        f"{', time features' if config.use_time_features else ''}"
    )
    plt.text(0.1, 0.9, model_info, va='top', fontsize=12)

    # Add run timestamp
    timestamp = get_maputo_timestamp()
    plt.figtext(0.5, 0.01, f"Generated: {timestamp}", ha='center', fontsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.suptitle('Model Performance', fontsize=18, fontweight='bold')

    # Save the figure
    plt.savefig(os.path.join(results_dir, f'model_performance_{timestamp}.png'), dpi=300)
    plt.close()



## Report Generation Module

In [ ]:

# =============================================================================
# Report Generation Module
# =============================================================================
def _create_title_page(model, config, timestamp, pdf):
    """
    Creates title page for the traffic prediction report.

    Args:
        model: The trained model
        config: Training configuration
        timestamp: Timestamp for the report
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Title
    plt.text(0.5, 0.85, "Traffic Prediction Analysis Report",
             fontsize=24, fontweight='bold', ha='center')

    # Subtitle with timestamp
    plt.text(0.5, 0.75, f"Generated on: {timestamp}",
             fontsize=14, ha='center')

    # Model information
    model_info = f"Model: TrafficTransformer"
    if hasattr(model, 'num_layers'):
        model_info += f"\nLayers: {model.num_layers}, Heads: {model.num_heads}"
    if hasattr(model, 'hidden_dim'):
        model_info += f", Hidden Dim: {model.hidden_dim}"
    plt.text(0.5, 0.65, model_info, fontsize=12, ha='center')

    # Configuration highlights
    config_highlights = (
        f"Sequence Length: {config.seq_length}, Prediction Window: {config.pred_length}\n"
        f"Batch Size: {config.batch_size}, Learning Rate: {config.learning_rate}\n"
        f"Optimizer: {config.optimizer_type}, Loss: {config.loss_function}\n"
        f"Using Time Features: {config.use_time_features}, "
        f"Using Holiday Features: {config.use_holiday_feature}"
    )
    plt.text(0.5, 0.55, config_highlights, fontsize=12, ha='center')

    # Footer
    plt.text(0.5, 0.2, "Transformer-Based Traffic Flow Prediction",
             fontsize=16, ha='center', fontstyle='italic')

    pdf.savefig()
    plt.close()

def _create_training_analysis(train_losses, val_losses, pdf):
    """
    Creates training analysis page showing loss curves and convergence patterns.

    Args:
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        pdf: PDF object to save the page
    """
    plt.figure(figsize=(12, 8))

    # Plot training and validation loss curves
    epochs = range(1, len(train_losses) + 1)
    plt.subplot(2, 1, 1)
    plt.plot(epochs, train_losses, 'b-', label='Training Loss')
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot convergence patterns (loss improvement rate)
    plt.subplot(2, 1, 2)
    if len(train_losses) > 5:  # Need enough epochs for moving average
        # Calculate moving average of loss improvement
        window_size = min(5, len(train_losses) // 4)
        train_improvements = [train_losses[i] - train_losses[i+window_size]
                             for i in range(len(train_losses) - window_size)]
        val_improvements = [val_losses[i] - val_losses[i+window_size]
                           for i in range(len(val_losses) - window_size)]

        # Plot improvement rates
        plt.plot(range(window_size + 1, len(train_losses) + 1),
                 train_improvements, 'b--', label='Training Improvement')
        plt.plot(range(window_size + 1, len(val_losses) + 1),
                 val_improvements, 'r--', label='Validation Improvement')
        plt.title('Loss Improvement Over Time (Higher is Better)')
        plt.xlabel('Epochs')
        plt.ylabel('Loss Reduction')
        plt.legend()
        plt.grid(True, alpha=0.3)
    else:
        # Not enough epochs for improvement analysis
        plt.text(0.5, 0.5, "Insufficient epochs for convergence analysis",
                 ha='center', va='center', fontsize=14)

    plt.tight_layout()
    pdf.savefig()
    plt.close()

    # Create additional training insights page if enough data
    if len(train_losses) > 10:
        plt.figure(figsize=(12, 8))

        # Early vs Late convergence
        plt.subplot(2, 2, 1)
        early_epochs = len(train_losses) // 3
        early_improvement = train_losses[0] - train_losses[early_epochs]
        late_improvement = train_losses[early_epochs] - train_losses[-1]

        bars = plt.bar(['Early Phase', 'Late Phase'],
                      [early_improvement, late_improvement])
        plt.title('Loss Improvement: Early vs Late Training')
        plt.ylabel('Loss Reduction')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                    f'{height:.4f}', ha='center', va='bottom')

        # Train-Val Loss Gap
        plt.subplot(2, 2, 2)
        loss_gaps = [val - train for train, val in zip(train_losses, val_losses)]
        plt.plot(epochs, loss_gaps)
        plt.title('Validation-Training Loss Gap')
        plt.xlabel('Epochs')
        plt.ylabel('Gap')
        plt.grid(True, alpha=0.3)

        # Loss distribution
        plt.subplot(2, 2, 3)
        plt.hist(train_losses, bins=10, alpha=0.5, label='Training')
        plt.hist(val_losses, bins=10, alpha=0.5, label='Validation')
        plt.title('Loss Distribution')
        plt.xlabel('Loss Value')
        plt.ylabel('Frequency')
        plt.legend()

        # Stability analysis (loss variance in last 1/3 of training)
        plt.subplot(2, 2, 4)
        stability_start = 2 * len(train_losses) // 3
        train_stability = np.std(train_losses[stability_start:])
        val_stability = np.std(val_losses[stability_start:])

        bars = plt.bar(['Training Stability', 'Validation Stability'],
                      [train_stability, val_stability])
        plt.title('Training Stability (Lower is Better)')
        plt.ylabel('Loss Standard Deviation')

        # Add values on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.0001,
                    f'{height:.6f}', ha='center', va='bottom')

        plt.tight_layout()
        pdf.savefig()
        plt.close()

def _create_performance_analysis(predictions, actuals, pdf):
    """
    Creates performance analysis page showing actual vs predicted values and error analysis.

    Args:
        predictions: Predicted values
        actuals: Actual values
        pdf: PDF object to save the page
    """
    # Ensure we have proper arrays
    predictions = np.array(predictions).ravel()
    actuals = np.array(actuals).ravel()

    plt.figure(figsize=(12, 10))
    gs = GridSpec(3, 1, figure=plt.gcf())

    # 1. Actual vs Predicted Scatter Plot
    ax1 = plt.subplot(gs[0, 0])
    ax1.scatter(actuals, predictions, alpha=0.5, s=10)

    # Add perfect prediction line
    min_val = min(np.min(actuals), np.min(predictions))
    max_val = max(np.max(actuals), np.max(predictions))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--')

    ax1.set_title('Actual vs Predicted Values')
    ax1.set_xlabel('Actual')
    ax1.set_ylabel('Predicted')
    ax1.grid(True, alpha=0.3)

    # 2. Error Distribution Histogram
    ax2 = plt.subplot(gs[0, 1])
    errors = predictions - actuals
    ax2.hist(errors, bins=30, alpha=0.7)
    ax2.set_title('Error Distribution')
    ax2.set_xlabel('Prediction Error')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)

    # Add mean and std as vertical lines
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    ax2.axvline(mean_error, color='r', linestyle='--', label=f'Mean: {mean_error:.4f}')
    ax2.axvline(mean_error + std_error, color='g', linestyle=':', label=f'Std: {std_error:.4f}')
    ax2.axvline(mean_error - std_error, color='g', linestyle=':')
    ax2.legend()

    # 3. Predicted vs Actual Time Series (sample)
    ax3 = plt.subplot(gs[1, :])
    sample_size = min(288, len(actuals))
    indices = range(sample_size)
    ax3.plot(indices, actuals[:sample_size], 'b-', label='Actual')
    ax3.plot(indices, predictions[:sample_size], 'r-', label='Predicted')
    ax3.set_title('Actual vs Predicted (Sample Time Series)')
    ax3.set_xlabel('Time Step')
    ax3.set_ylabel('Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Residual Plot
    ax4 = plt.subplot(gs[2, 0])
    ax4.scatter(actuals, errors, alpha=0.5, s=10)
    ax4.axhline(y=0, color='r', linestyle='--')
    ax4.set_title('Residual Plot')
    ax4.set_xlabel('Actual Value')
    ax4.set_ylabel('Residual (Error)')
    ax4.grid(True, alpha=0.3)

    # 5. Q-Q Plot for Error Normality
    # ax5 = plt.subplot(gs[2, 1])
    # stats.probplot(errors, dist="norm", plot=ax5)
    # ax5.set_title('Q-Q Plot of Residuals')
    # ax5.grid(True, alpha=0.3)

    # Overall metrics text
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    r2 = r2_score(actuals, predictions)
    mape = 100 * np.mean(np.abs((actuals - predictions) / (actuals + 1e-8)))

    metrics_text = (
        f"MAE: {mae:.4f}\n"
        f"RMSE: {rmse:.4f}\n"
        f"R²: {r2:.4f}\n"
        f"MAPE: {mape:.2f}%"
    )

    plt.figtext(0.5, 0.01, metrics_text, ha="center", fontsize=12,
               bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Adjust layout to make room for text
    pdf.savefig()
    plt.close()

def generate_traffic_report(
    model: nn.Module,
    test_loader: DataLoader,
    scaler: Any,
    config: TrainingConfig,
    device: str,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    results_dir: str,
    timestamp: Optional[str] = None
) -> str:
    """
    Generate comprehensive report for traffic prediction model analysis with improved error handling

    Args:
        model: Trained model
        test_loader: Test data loader
        scaler: Scaler used for normalization
        config: Training configuration
        device: Device used for model
        train_losses: Training loss history
        val_losses: Validation loss history
        fold_metrics: Metrics for each fold
        baseline_metrics: Metrics for baseline models
        results_dir: Directory to save report
        timestamp: Optional timestamp for the report

    Returns:
        Path to generated report
    """
    timestamp = get_maputo_timestamp() if timestamp is None else timestamp
    pdf_path = os.path.join(results_dir, f'traffic_prediction_report_{timestamp}.pdf')

    # Memory cleanup before generating report
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Set plotting style
    try:
        plt.style.use('seaborn-v0_8')
    except:
        try:
            plt.style.use('seaborn')
        except:
            plt.style.use('default')

    # Set publication-quality figure parameters
    plt.rcParams.update({
        'figure.figsize': (10, 6),
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'figure.titlesize': 20,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'lines.linewidth': 2,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })

    # Utility function for error pages
    def _create_error_page(error_message: str, pdf: PdfPages) -> None:
        """Creates an error page for the PDF when a section fails."""
        plt.figure(figsize=(12, 8))
        plt.axis('off')
        plt.text(0.5, 0.5, f"Error: {error_message}",
                ha='center', va='center', color='red', fontsize=14, wrap=True)
        pdf.savefig()
        plt.close()

    # Collect predictions and attention weights
    model.eval()
    predictions = []
    actuals = []
    attention_weights = []

    try:
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)

                # Store attention weights if available
                if hasattr(model, 'attention_weights') and model.attention_weights is not None:
                    # Handle different attention weight shapes
                    weights = model.attention_weights
                    if isinstance(weights, torch.Tensor):
                        weights = weights.cpu().numpy()
                    attention_weights.append(weights)

                # Store predictions and actuals
                pred = output.cpu().numpy()
                predictions.append(pred)
                actuals.append(target.cpu().numpy())

        predictions = np.concatenate(predictions)
        actuals = np.concatenate(actuals)

        # Reshape and inverse transform if needed
        if len(predictions.shape) > 2:
            predictions = predictions.reshape(-1, predictions.shape[-1])
            actuals = actuals.reshape(-1, actuals.shape[-1])
    except Exception as e:
        print(f"Error collecting predictions: {str(e)}")
        # Create minimal emergency report with available data
        _create_emergency_report(
            predictions or np.array([]),
            actuals or np.array([]),
            train_losses,
            val_losses,
            pdf_path
        )
        return pdf_path

    # Create PDF report with section-by-section error handling
    try:
        with PdfPages(pdf_path) as pdf:
            # 1. Title Page
            try:
                _create_title_page(model, config, timestamp, pdf)
            except Exception as e:
                _create_error_page(f"Error in title page: {str(e)}", pdf)
                print(f"Error in title page: {str(e)}")

            # 2. Training Analysis
            try:
                _create_training_analysis(train_losses, val_losses, pdf)
            except Exception as e:
                _create_error_page(f"Error in training analysis: {str(e)}", pdf)
                print(f"Error in training analysis: {str(e)}")

            # 3. Performance Analysis
            try:
                if len(predictions) > 0 and len(actuals) > 0:
                    _create_performance_analysis(predictions, actuals, pdf)
                else:
                    _create_error_page("Insufficient prediction data for performance analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in performance analysis: {str(e)}", pdf)
                print(f"Error in performance analysis: {str(e)}")

            # 4. Attention Analysis
            try:
                if attention_weights:
                    _create_attention_analysis(attention_weights, config, pdf)
                else:
                    _create_error_page("No attention weights available for analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in attention analysis: {str(e)}", pdf)
                print(f"Error in attention analysis: {str(e)}")

            # 5. Cross Validation Analysis
            try:
                if fold_metrics:
                    _create_cross_validation_analysis(fold_metrics, pdf)
                else:
                    _create_error_page("No fold metrics available for cross-validation analysis", pdf)
            except Exception as e:
                _create_error_page(f"Error in cross validation analysis: {str(e)}", pdf)
                print(f"Error in cross validation analysis: {str(e)}")

            # 6. Baseline Comparison
            try:
                if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
                    _create_baseline_comparison(baseline_metrics, fold_metrics, pdf)
                else:
                    _create_error_page("Insufficient data for baseline comparison", pdf)
            except Exception as e:
                _create_error_page(f"Error in baseline comparison: {str(e)}", pdf)
                print(f"Error in baseline comparison: {str(e)}")

            # 7. Model Configuration
            try:
                _create_config_summary(config, pdf)
            except Exception as e:
                _create_error_page(f"Error in config summary: {str(e)}", pdf)
                print(f"Error in config summary: {str(e)}")

            # 8. Summary Statistics
            try:
                _create_summary_statistics(
                    predictions, actuals, train_losses, val_losses,
                    fold_metrics, baseline_metrics, pdf
                )
            except Exception as e:
                _create_error_page(f"Error in summary statistics: {str(e)}", pdf)
                print(f"Error in summary statistics: {str(e)}")

    except Exception as e:
        print(f"Error generating report: {str(e)}")
        # Create minimal emergency report
        _create_emergency_report(predictions, actuals, train_losses, val_losses, pdf_path)

    return pdf_path


def _create_attention_analysis(
    attention_weights: List[np.ndarray],
    config: TrainingConfig,
    pdf: PdfPages
) -> None:
    """Creates attention analysis visualizations with improved error handling."""
    plt.figure(figsize=(12, 8))

    # Guard against empty attention weights
    if not attention_weights or all(w is None for w in attention_weights):
        plt.text(0.5, 0.5, "No attention weights available",
                 ha='center', va='center', fontsize=14)
        pdf.savefig()
        plt.close()
        return

    try:
        # Handle different possible shapes of attention weights
        sample_weights = attention_weights[0]
        attention_array = np.array(attention_weights)

        # Detect shape and process accordingly
        if len(attention_array.shape) == 4:  # [batch, heads, seq, seq]
            attention_mean = np.mean(attention_array, axis=(0, 1))  # Average across batch and heads
        elif len(attention_array.shape) == 3:  # [batch, seq, seq]
            attention_mean = np.mean(attention_array, axis=0)  # Average across batch
        elif len(attention_array.shape) == 2:  # Already [seq, seq]
            attention_mean = attention_array
        else:
            # If we have a list of tensors with different shapes
            reshaped_weights = []
            for weights in attention_weights:
                if hasattr(weights, 'shape'):
                    if len(weights.shape) == 3:  # [batch, seq, seq]
                        weights = np.mean(weights, axis=0)
                    elif len(weights.shape) > 3:  # More dimensions than expected
                        weights = np.mean(weights, axis=tuple(range(len(weights.shape)-2)))
                reshaped_weights.append(weights)

            attention_mean = np.mean(reshaped_weights, axis=0)

        # Create heatmap
        sns.heatmap(attention_mean, cmap="viridis", annot=False)
        plt.title('Average Attention Weights')
        plt.xlabel('Key Position')
        plt.ylabel('Query Position')

    except Exception as e:
        # Create a fallback visualization with error message
        plt.clf()  # Clear the figure
        plt.text(0.5, 0.5, f"Error visualizing attention weights: {str(e)}\n"
                           f"Shape info: {[w.shape if hasattr(w, 'shape') else type(w) for w in attention_weights[:3]]}...",
                 ha='center', va='center', wrap=True)
        plt.title('Attention Visualization Error')

    pdf.savefig()
    plt.close()


def _create_cross_validation_analysis(
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates cross validation analysis visualizations."""
    metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']
    metrics_values = np.array(fold_metrics)

    plt.figure(figsize=(12, 6))
    for i, metric in enumerate(metrics_names):
        plt.subplot(2, 2, i+1)
        plt.boxplot(metrics_values[:, i])
        plt.title(f'{metric} Across Folds')
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    pdf.savefig()
    plt.close()


def _create_baseline_comparison(
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    fold_metrics: List[Tuple[float, float, float, float]],
    pdf: PdfPages
) -> None:
    """Creates simplified baseline comparison visualization without ARIMA, LSTM, and ExpSmoothing."""

    plt.figure(figsize=(12, 8))
    plt.axis('off')  # Turn off axes for the message version

    # Check if there are any metrics in baseline_metrics
    if not any(metrics for metrics in baseline_metrics.values()):
        # Display message that baselines are disabled
        plt.text(0.5, 0.5,
                "Baseline models (ARIMA, LSTM, ExpSmoothing) have been disabled",
                ha='center', va='center', fontsize=14,
                bbox={'facecolor': 'lightgray', 'alpha': 0.5, 'pad': 10})
        plt.title('Baseline Comparison', fontsize=16)

        # Add a note about transformer performance
        if fold_metrics:
            avg_metrics = np.mean(fold_metrics, axis=0)
            metrics_text = (
                f"\n\nTransformer Model Metrics:\n"
                f"MAE: {avg_metrics[0]:.4f}\n"
                f"RMSE: {avg_metrics[1]:.4f}\n"
                f"R²: {avg_metrics[2]:.4f}\n"
                f"MAPE: {avg_metrics[3]:.2f}%"
            )
            plt.text(0.5, 0.3, metrics_text, ha='center', fontsize=12)
    else:
        # If there are any baseline metrics (like naive forecast), show comparison
        transformer_metrics = np.mean(fold_metrics, axis=0)
        metrics_names = ['MAE', 'RMSE', 'R²', 'MAPE']

        # Reset axis settings for plots
        plt.clf()

        # Setup subplots for metrics
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        plt.suptitle('Transformer vs Available Baselines', fontsize=16)

        # Flatten axes for easier access
        axes = axes.flatten()

        # Get available models
        models = ['Transformer'] + list(baseline_metrics.keys())

        # Prepare data for plotting
        metrics_data = [transformer_metrics]
        for model in baseline_metrics.keys():
            if baseline_metrics[model]:
                metrics_data.append(np.mean(baseline_metrics[model], axis=0))
            else:
                metrics_data.append([np.nan, np.nan, np.nan, np.nan])

        # Create plots for each metric
        for i, (metric, ax) in enumerate(zip(metrics_names, axes)):
            metric_values = [data[i] for data in metrics_data]
            ax.bar(models, metric_values)
            ax.set_title(f'{metric} Comparison')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(True, alpha=0.3)

            # Add values on bars
            for j, value in enumerate(metric_values):
                if not np.isnan(value):
                    ax.text(j, value + (max(metric_values) * 0.05),
                           f"{value:.3f}", ha='center')

    # Ensure proper layout
    plt.tight_layout()

    # Save to PDF
    pdf.savefig()
    plt.close()


def _create_config_summary(config: TrainingConfig, pdf: PdfPages) -> None:
    """Creates configuration summary page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    config_text = "Model Configuration:\n\n"
    for key, value in vars(config).items():
        if not key.startswith('_') and key not in ['input_dir', 'output_dir', 'model_dir', 'results_dir']:  # Skip directories for brevity
            config_text += f"{key}: {value}\n"

    plt.text(0.1, 0.9, config_text, fontsize=10, va='top')
    pdf.savefig()
    plt.close()


def _create_summary_statistics(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    fold_metrics: List[Tuple[float, float, float, float]],
    baseline_metrics: Dict[str, List[Tuple[float, float, float, float]]],
    pdf: PdfPages
) -> None:
    """Creates summary statistics page."""
    plt.figure(figsize=(12, 8))
    plt.axis('off')

    # Calculate overall metrics
    mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
    rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
    r2 = r2_score(actuals.ravel(), predictions.ravel())

    summary_text = f"""
    Overall Model Performance:

    Mean Absolute Error: {mae:.4f}
    Root Mean Squared Error: {rmse:.4f}
    R² Score: {r2:.4f}

    Training Summary:
    Initial Training Loss: {train_losses[0]:.6f}
    Final Training Loss: {train_losses[-1]:.6f}
    Loss Improvement: {train_losses[0] - train_losses[-1]:.6f}

    Cross-Validation Summary:
    Number of Folds: {len(fold_metrics)}
    Average MAE across folds: {np.mean([m[0] for m in fold_metrics]):.4f}
    Average RMSE across folds: {np.mean([m[1] for m in fold_metrics]):.4f}

    Baseline Comparison:
    """

    for model_name, metrics in baseline_metrics.items():
        if metrics:  # Check if metrics list is not empty
            avg_metrics = np.mean(metrics, axis=0)
            summary_text += f"\n{model_name.upper()} - MAE: {avg_metrics[0]:.4f}, RMSE: {avg_metrics[1]:.4f}"
        else:
            summary_text += f"\n{model_name.upper()} - No metrics available"

    plt.text(0.1, 0.9, summary_text, fontsize=12, va='top')
    pdf.savefig()
    plt.close()


def _create_emergency_report(
    predictions: np.ndarray,
    actuals: np.ndarray,
    train_losses: List[float],
    val_losses: List[float],
    pdf_path: str
) -> None:
    """Creates a minimal emergency report if the full report fails."""
    try:
        with PdfPages(pdf_path) as pdf:
            plt.figure(figsize=(12, 8))
            plt.axis('off')

            mae = mean_absolute_error(actuals.ravel(), predictions.ravel())
            rmse = np.sqrt(mean_squared_error(actuals.ravel(), predictions.ravel()))
            r2 = r2_score(actuals.ravel(), predictions.ravel())

            emergency_text = f"""
            Emergency Report (Error in full report generation)

            Basic Metrics:
            MAE: {mae:.4f}
            RMSE: {rmse:.4f}
            R² Score: {r2:.4f}

            Final Losses:
            Training: {train_losses[-1]:.6f}
            Validation: {val_losses[-1]:.6f}
            """
            plt.text(0.1, 0.9, emergency_text, fontsize=12, va='top')
            pdf.savefig()
            plt.close()
    except Exception as e2:
        warnings.warn(f"Emergency report also failed: {str(e2)}")



## Transfer Learning Module

In [ ]:
# =============================================================================
# Transfer Learning Module for Traffic Transformers
# =============================================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import gc

class TransferLearningModule:
    """
    Module for transfer learning with Traffic Transformer models.
    Enables fine-tuning models pre-trained on METR-LA for African traffic datasets.
    """

    def __init__(
        self,
        base_model: nn.Module,
        config: 'TrainingConfig',
        target_dataset_name: str = 'mozambique',
        freeze_encoder: bool = True,
        freeze_layers: int = 1,
        adapter_dim: int = 64
    ):
        """
        Initialize transfer learning module

        Args:
            base_model: Pre-trained Traffic Transformer model
            config: Configuration object
            target_dataset_name: Name of target dataset ('mozambique' or 'south_africa')
            freeze_encoder: Whether to freeze encoder layers
            freeze_layers: Number of transformer layers to freeze (if freeze_encoder is True)
            adapter_dim: Dimension of adapter layers (if used)
        """
        self.base_model = base_model
        self.config = config
        self.target_dataset_name = target_dataset_name
        self.freeze_encoder = freeze_encoder
        self.freeze_layers = freeze_layers
        self.adapter_dim = adapter_dim
        self.device = next(base_model.parameters()).device

        # Apply freezing based on parameters
        self._apply_parameter_freezing()

        # Add adapter layers if needed
        self.has_adapters = adapter_dim > 0
        if self.has_adapters:
            self._add_adapter_layers()

        # Store metrics for tracking
        self.transfer_history = {
            'train_loss': [],
            'val_loss': [],
            'source_metrics': None,  # Will store metrics on source dataset
            'target_metrics': None   # Will store metrics on target dataset
        }

    def _apply_parameter_freezing(self):
        """Apply freezing to specified parts of the model"""
        if self.freeze_encoder:
            # Freeze embedding layer
            for param in self.base_model.embedding.parameters():
                param.requires_grad = False

            # Freeze positional encoding (not trainable by default, but being explicit)
            if hasattr(self.base_model.pos_encoder, 'pe'):
                self.base_model.pos_encoder.pe.requires_grad = False

            # Freeze specified transformer layers
            for i, layer in enumerate(self.base_model.transformer):
                if i < self.freeze_layers:
                    for param in layer.parameters():
                        param.requires_grad = False

            print(f"Froze embedding layer and {self.freeze_layers} transformer layers")
        else:
            print("No parameter freezing applied - full fine-tuning")

    def _add_adapter_layers(self):
        """
        Add improved adapter layers to the model for more effective transfer learning.
        """
        device = self.device

        # Get the hidden dimension from the model
        if hasattr(self.base_model, 'hidden_dim'):
            hidden_dim = self.base_model.hidden_dim
        elif hasattr(self.base_model.transformer[0], 'linear1'):
            hidden_dim = self.base_model.transformer[0].linear1.out_features
        else:
            raise ValueError("Cannot determine hidden dimension for adapters")

        # Create adapter layers with proper initialization
        self.adapter_down = nn.ModuleList([
            nn.Linear(hidden_dim, self.adapter_dim)
            for _ in range(len(self.base_model.transformer) - self.freeze_layers)
        ]).to(device)

        self.adapter_up = nn.ModuleList([
            nn.Linear(self.adapter_dim, hidden_dim)
            for _ in range(len(self.base_model.transformer) - self.freeze_layers)
        ]).to(device)

        # Initialize adapters with small weights for stability (very important!)
        for down, up in zip(self.adapter_down, self.adapter_up):
            # Use specific initializations that work well for adapters
            # Down projection: Xavier normal with small gain
            nn.init.xavier_normal_(down.weight, gain=0.1)
            nn.init.zeros_(down.bias)

            # Up projection: zero init for stable start (common in adapter papers)
            nn.init.zeros_(up.weight)
            nn.init.zeros_(up.bias)

        # Layer normalization for each adapter for better stability
        self.adapter_norm = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in range(len(self.base_model.transformer) - self.freeze_layers)
        ]).to(device)

        # Store original forward methods
        self.original_forwards = []

        # Patch transformer layers with adapters
        for i in range(self.freeze_layers, len(self.base_model.transformer)):
            layer = self.base_model.transformer[i]
            adapter_idx = i - self.freeze_layers

            # Store original forward method
            orig_forward = layer.forward
            self.original_forwards.append(orig_forward)

            # Define new forward method with adapter
            def make_adapter_forward(layer_idx, orig_f):
                def forward_with_adapter(x):
                    # Call original layer
                    output = orig_f(x)

                    # Apply adapter
                    adapter_idx = layer_idx - self.freeze_layers

                    # Adapter forward path with scaling:
                    # 1. Save residual connection
                    residual = output

                    # 2. Layer norm before adapter
                    normalized = self.adapter_norm[adapter_idx](output)

                    # 3. Down projection
                    adapter_out = self.adapter_down[adapter_idx](normalized)

                    # 4. Activation - GeLU often works better than ReLU for adapters
                    adapter_out = F.gelu(adapter_out)

                    # 5. Up projection
                    adapter_out = self.adapter_up[adapter_idx](adapter_out)

                    # 6. Scaled residual connection (0.1 scaling factor)
                    output = residual + 0.1 * adapter_out

                    return output

                return forward_with_adapter

            # Set new forward method
            layer.forward = make_adapter_forward(i, orig_forward)

        print(f"Added {len(self.adapter_down)} improved adapter layers (dim={self.adapter_dim})")

    def load_african_dataset(
        self,
        file_path: str,
        sequence_length: Optional[int] = None,
        prediction_length: Optional[int] = None,
        test_split: float = 0.2
    ) -> Tuple[DataLoader, DataLoader, Any]:
        """
        Load and prepare an African traffic dataset for transfer learning

        Args:
            file_path: Path to the dataset CSV file
            sequence_length: Length of input sequence (defaults to config value)
            prediction_length: Length of prediction window (defaults to config value)
            test_split: Portion of data to use for testing

        Returns:
            Tuple of (train_loader, test_loader, scaler)
        """
        seq_length = sequence_length or self.config.seq_length
        pred_length = prediction_length or self.config.pred_length

        try:
            # Load dataset
            df = pd.read_csv(file_path, parse_dates=True, index_col=0)
            print(f"Loaded dataset with shape: {df.shape}")

            # Handle null values
            df.replace(0.0, np.nan, inplace=True)
            df.ffill(inplace=True)
            df.bfill(inplace=True)

            # Get timestamps and data
            timestamps = df.index
            sensor_data = df.values
            num_features = sensor_data.shape[1]

            # Scale data
            if self.config.data_scaler_type == 'minmax':
                data_scaler = MinMaxScaler()
            elif self.config.data_scaler_type == 'standard':
                data_scaler = StandardScaler()
            elif self.config.data_scaler_type == 'robust':
                data_scaler = RobustScaler()
            else:
                warnings.warn(f"Invalid scaler type: {self.config.data_scaler_type}, using MinMaxScaler")
                data_scaler = MinMaxScaler()

            data_normalized = data_scaler.fit_transform(sensor_data)

            # Split into train and test sets
            total_samples = len(data_normalized)
            test_size = int(total_samples * test_split)
            train_size = total_samples - test_size

            train_data = data_normalized[:train_size]
            test_data = data_normalized[train_size:]
            train_times = timestamps[:train_size]
            test_times = timestamps[train_size:]

            # Create datasets
            train_dataset = TrafficDataset(
                train_data, train_times, seq_length, pred_length, self.config
            )
            test_dataset = TrafficDataset(
                test_data, test_times, seq_length, pred_length, self.config
            )

            # Create data loaders
            batch_size = min(32, self.config.batch_size)  # Smaller batch size for smaller datasets

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=min(2, self.config.num_workers),  # Reduce workers for smaller datasets
                pin_memory=self.config.pin_memory
            )

            test_loader = DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=min(2, self.config.num_workers),
                pin_memory=self.config.pin_memory
            )

            return train_loader, test_loader, data_scaler

        except Exception as e:
            raise RuntimeError(f"Error loading African dataset: {str(e)}")

    def fine_tune(
        self,
        train_loader: DataLoader,
        val_loader: DataLoader,
        learning_rate: float = 1e-4,
        num_epochs: int = 30,
        patience: int = 5,
        output_dir: Optional[str] = None
    ) -> nn.Module:
        """
        Fine-tune the model on a new dataset

        Args:
            train_loader: DataLoader with training data
            val_loader: DataLoader with validation data
            learning_rate: Learning rate for fine-tuning
            num_epochs: Maximum number of epochs
            patience: Early stopping patience
            output_dir: Directory to save checkpoints and results

        Returns:
            Fine-tuned model
        """
        model = self.base_model
        model.train()

        # Only optimize parameters that require gradients
        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=learning_rate
        )

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', patience=patience//2, factor=0.5
        )

        criterion = nn.MSELoss()
        best_loss = float('inf')
        no_improve = 0
        train_losses = []
        val_losses = []

        # Training loop
        for epoch in range(num_epochs):
            model.train()
            epoch_loss = 0

            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(self.device), target.to(self.device)

                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()

                # Gradient clipping
                if self.config.gradient_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), self.config.gradient_clip)

                optimizer.step()
                epoch_loss += loss.item()

            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # Validation
            model.eval()
            val_loss = 0

            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(self.device), target.to(self.device)
                    output = model(data)
                    loss = criterion(output, target)
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            # Update learning rate scheduler
            scheduler.step(avg_val_loss)

            # Early stopping check
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                no_improve = 0

                # Save best model if output_dir is provided
                if output_dir:
                    os.makedirs(output_dir, exist_ok=True)
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': best_loss,
                        'target_dataset': self.target_dataset_name
                    }, os.path.join(output_dir, f'fine_tuned_{self.target_dataset_name}_best.pth'))
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f'Early stopping at epoch {epoch+1}')
                    break

            print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}')

        # Store training history
        self.transfer_history['train_loss'] = train_losses
        self.transfer_history['val_loss'] = val_losses

        # Plot training history if output_dir is provided
        if output_dir:
            self._plot_transfer_learning_curve(train_losses, val_losses, output_dir)

        return model

    def evaluate_transfer(
        self,
        source_loader: DataLoader,
        target_loader: DataLoader,
        source_scaler: Any,
        target_scaler: Any,
        output_dir: Optional[str] = None
    ) -> Dict[str, Dict[str, float]]:
        """
        Evaluate transfer learning performance on both source and target datasets

        Args:
            source_loader: DataLoader with source dataset
            target_loader: DataLoader with target dataset
            source_scaler: Scaler for source data
            target_scaler: Scaler for target data
            output_dir: Directory to save results

        Returns:
            Dictionary with evaluation metrics
        """
        model = self.base_model
        model.eval()

        # Evaluate on source dataset
        source_metrics = self._evaluate_on_dataset(model, source_loader, source_scaler, "METR-LA")

        # Evaluate on target dataset
        target_metrics = self._evaluate_on_dataset(model, target_loader, target_scaler, self.target_dataset_name)

        # Store metrics in history
        self.transfer_history['source_metrics'] = source_metrics
        self.transfer_history['target_metrics'] = target_metrics

        # Compare and visualize if output_dir is provided
        if output_dir:
            self._plot_transfer_comparison(source_metrics, target_metrics, output_dir)

        # Return both metrics
        return {
            'source': source_metrics,
            'target': target_metrics
        }

    def _evaluate_on_dataset(
        self,
        model: nn.Module,
        dataloader: DataLoader,
        scaler: Any,
        dataset_name: str
    ) -> Dict[str, float]:
        """Helper method to evaluate model on a dataset"""
        model.eval()
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(self.device), target.to(self.device)
                output = model(data)

                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        predictions = np.concatenate(all_preds)
        actuals = np.concatenate(all_targets)

        # Reshape for inverse transformation
        num_samples, pred_window, num_features = predictions.shape
        predictions_2d = predictions.reshape(-1, num_features)
        actuals_2d = actuals.reshape(-1, num_features)

        # Inverse transform
        predictions_inv = scaler.inverse_transform(predictions_2d)
        actuals_inv = scaler.inverse_transform(actuals_2d)

        # Calculate metrics
        mae = mean_absolute_error(actuals_inv.ravel(), predictions_inv.ravel())
        rmse = np.sqrt(mean_squared_error(actuals_inv.ravel(), predictions_inv.ravel()))
        r2 = r2_score(actuals_inv.ravel(), predictions_inv.ravel())

        # Calculate MAPE with handling for zeros
        mape = np.mean(np.abs((actuals_inv.ravel() - predictions_inv.ravel()) /
                               np.maximum(np.abs(actuals_inv.ravel()), 1e-10))) * 100

        print(f'Evaluation on {dataset_name}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, MAPE={mape:.2f}%')

        return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

    def _plot_transfer_learning_curve(
        self,
        train_losses: List[float],
        val_losses: List[float],
        output_dir: str
    ):
        """Plot transfer learning curves"""
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Training Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.title(f'Transfer Learning to {self.target_dataset_name.title()} Dataset')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Add details about freezing and adapters
        plt.annotate(
            f"Frozen layers: {self.freeze_layers if self.freeze_encoder else 'None'}\n"
            f"Adapters: {'Yes' if self.has_adapters else 'No'}",
            xy=(0.02, 0.02), xycoords='figure fraction'
        )

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'transfer_learning_curve_{self.target_dataset_name}.png'), dpi=300)
        plt.close()

    def _plot_transfer_comparison(
        self,
        source_metrics: Dict[str, float],
        target_metrics: Dict[str, float],
        output_dir: str
    ):
        """Plot comparison between source and target dataset performance"""
        metrics = ['mae', 'rmse', 'r2', 'mape']
        source_values = [source_metrics[m] for m in metrics]
        target_values = [target_metrics[m] for m in metrics]

        # For better visualization, normalize R² separately (higher is better)
        if metrics.index('r2') == 2:  # If r2 is the third metric
            # Invert R² so lower is better for visualization consistency
            source_values[2] = 1 - source_values[2]
            target_values[2] = 1 - target_values[2]
            metrics[2] = 'r2 (inverted)'

        plt.figure(figsize=(12, 8))

        x = range(len(metrics))
        width = 0.35

        plt.bar([i - width/2 for i in x], source_values, width, label='METR-LA (Source)')
        plt.bar([i + width/2 for i in x], target_values, width, label=f'{self.target_dataset_name.title()} (Target)')

        plt.xlabel('Metrics')
        plt.ylabel('Value (Lower is Better)')
        plt.title('Transfer Learning Performance Comparison')
        plt.xticks(x, metrics)
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Add text showing improvement/degradation percentages
        for i, (source, target) in enumerate(zip(source_values, target_values)):
            if metrics[i] != 'r2 (inverted)':
                change_pct = (target - source) / source * 100
                color = 'green' if change_pct < 0 else 'red'
                plt.annotate(
                    f"{change_pct:.1f}%",
                    xy=(i, max(source, target) * 1.05),
                    ha='center',
                    color=color
                )

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'transfer_comparison_{self.target_dataset_name}.png'), dpi=300)
        plt.close()

    @staticmethod
    def visualize_attention_transfer(
        base_model: nn.Module,
        source_loader: DataLoader,
        target_loader: DataLoader,
        output_dir: str
    ):
        """
        Visualize and compare attention patterns between source and target datasets

        Args:
            base_model: The fine-tuned model
            source_loader: DataLoader with source dataset
            target_loader: DataLoader with target dataset
            output_dir: Directory to save visualizations
        """
        device = next(base_model.parameters()).device
        base_model.eval()

        # Check if model has adapters
        has_adapters = hasattr(base_model, 'has_transfer_adapters') and base_model.has_transfer_adapters

        # Get sample batch from each dataset
        source_batch = next(iter(source_loader))
        target_batch = next(iter(target_loader))

        # Function to extract attention weights from a batch
        def get_attention_weights(batch, is_source=False):
            data, _ = batch
            data = data.to(device)

            # Store original state if we need to temporarily remove adapters
            original_forward = None
            original_decoder = None
            original_num_features = None
            if has_adapters and is_source:
                print("Temporarily removing adapters for source attention visualization...")
                # Store original methods and attributes
                original_forward = base_model.forward
                original_decoder = base_model.decoder
                original_num_features = base_model.num_features

                # Restore original methods for source data
                if hasattr(base_model, '_original_forward') and hasattr(base_model, 'source_output_dim'):
                    base_model.forward = base_model._original_forward
                    base_model.num_features = base_model.source_output_dim
                    base_model.decoder = base_model.original_decoder
                    print(f"Using original forward method with num_features={base_model.num_features}")

            # Clear any previous attention weights
            if hasattr(base_model, 'attention_weights'):
                base_model.attention_weights = None

            try:
                # Forward pass to capture attention weights
                with torch.no_grad():
                    _ = base_model(data)

                # Get attention weights
                if hasattr(base_model, 'attention_weights') and base_model.attention_weights is not None:
                    weights = base_model.attention_weights
                    if isinstance(weights, torch.Tensor):
                        weights = weights.cpu().numpy()
                    return weights
                return None
            finally:
                # Restore adapters if we temporarily removed them
                if has_adapters and is_source and original_forward is not None:
                    base_model.forward = original_forward
                    base_model.decoder = original_decoder
                    base_model.num_features = original_num_features
                    print("Restored adapters after source attention visualization")

        # Get attention weights for both datasets
        print("Getting attention weights for source dataset...")
        source_attention = get_attention_weights(source_batch, is_source=True)

        print("Getting attention weights for target dataset...")
        target_attention = get_attention_weights(target_batch, is_source=False)

        if source_attention is not None and target_attention is not None:
            print(f"Source attention shape: {source_attention.shape}, Target attention shape: {target_attention.shape}")

            # Make sure shapes match for comparison
            if source_attention.shape != target_attention.shape:
                print(f"Warning: Attention matrices have different shapes. Resizing for visualization.")
                # Resize to smallest common dimensions or use subsets
                min_dim = min(source_attention.shape[0], target_attention.shape[0])
                source_attention = source_attention[:min_dim, :min_dim]
                target_attention = target_attention[:min_dim, :min_dim]

            plt.figure(figsize=(15, 6))

            # Plot source attention
            plt.subplot(1, 3, 1)
            sns.heatmap(source_attention, cmap='viridis')
            plt.title('Source Dataset (METR-LA)\nAttention Pattern')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            # Plot target attention
            plt.subplot(1, 3, 2)
            sns.heatmap(target_attention, cmap='viridis')
            plt.title('Target Dataset\nAttention Pattern')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            # Plot difference
            plt.subplot(1, 3, 3)
            diff = target_attention - source_attention
            sns.heatmap(diff, cmap='coolwarm', center=0)
            plt.title('Attention Difference\n(Target - Source)')
            plt.xlabel('Key Position')
            plt.ylabel('Query Position')

            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'attention_transfer_comparison.png'), dpi=300)
            plt.close()
            print(f"Attention visualization saved to {os.path.join(output_dir, 'attention_transfer_comparison.png')}")
        else:
            print("Could not extract attention weights for visualization")

## Main Execution

In [ ]:
# =============================================================================
# Main Execution with Full Transfer Learning Support
# =============================================================================

def main():
    print(f"[ {generate_timestamp()} ] Starting main execution...")
    """Main execution function with enhanced transfer learning support"""
    # --- Load Configuration ---
    config_path = '' #'/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/config.yaml'
    if os.path.exists(config_path):
        config = load_config(config_path)
        print("Configuration loaded from config.yaml")
    else:
        print("config.yaml not found, using default parameters.")
        config = TrainingConfig(base_output_dir='/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15')

    # Check if transfer learning is enabled
    is_transfer_learning = hasattr(config, 'enable_transfer_learning') and config.enable_transfer_learning
    # Check if we should run ONLY transfer learning
    run_only_transfer = hasattr(config, 'run_only_transfer_learning') and config.run_only_transfer_learning

    # --- Setup Directories ---
    input_dir, output_dir, model_dir, results_dir = setup_directories(config)

    # --- Mount Google Drive if in Colab ---
    if IN_COLAB:
        try:
            drive.mount('/content/drive')
            print("Google Drive mounted successfully")
        except:
            warnings.warn("Failed to mount Google Drive, using local directories")

    # --- Weather Data Setup ---
    weather_file_path = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/clean_weather_data_v3.csv'
    if os.path.exists(weather_file_path):
        print(f"Weather file exists at: {weather_file_path}")
        with open(weather_file_path, 'r') as f:
            header = f.readline()
            print(f"Weather file header: {header}")
    else:
        print(f"Weather file NOT FOUND at: {weather_file_path}")

    # Enable weather features
    config.use_weather_feature = True
    config.weather_data_file = '/content/drive/Shareddrives/Almo-2002-R&D/1 - RD-Traffic-Prediction/Transformer_Versions/COLAB_NOBASELINE_V15/Transformers_Input/clean_weather_data_v3.csv'

    # Weather file path might be updated by setup_directories
    potential_weather_path_in_input = os.path.join(input_dir, os.path.basename(config.weather_data_file))
    if os.path.exists(potential_weather_path_in_input):
        config.weather_data_file = potential_weather_path_in_input
    else:
        print(f"Warning: Weather file {potential_weather_path_in_input} not found in input_dir.")

    # --- Set Device ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Start GPU memory monitoring
    monitor_stop_flag = start_gpu_memory_monitor(config, interval=15)

    try:
        if is_transfer_learning:
            # =====================================================
            # TRANSFER LEARNING PATH
            # =====================================================
            print("\n=== Running Transfer Learning Workflow ===\n")

            # 1. Load source (METR-LA) dataset for creating model and comparison
            try:
                source_df = pd.read_csv(os.path.join(input_dir, 'METR-LA.csv'), index_col=0, parse_dates=True)
                print(f"Loaded METR-LA data with shape: {source_df.shape}")

                # Store sensor IDs for later use in plotting
                source_sensor_ids = source_df.columns.tolist()

                # Prepare source data
                source_data, source_timestamps, source_scaler, source_num_features = prepare_data(source_df, config)

                # Create test dataset for source data (for evaluation comparison)
                source_test_size = int(len(source_data) * 0.2)
                source_test_data = source_data[-source_test_size:]
                source_test_times = source_timestamps[-source_test_size:]

                source_test_dataset = TrafficDataset(
                    source_test_data, source_test_times, config.seq_length, config.pred_length, config
                )

                source_test_loader = DataLoader(
                    source_test_dataset,
                    batch_size=config.batch_size,
                    shuffle=False,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                print(f"Prepared source test dataset with {len(source_test_dataset)} samples")

                # Set up source adjacency matrix if needed
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    min_data_len_adj = config.seq_length + config.pred_length
                    dataset_instance_adj = TrafficDataset(
                        source_data[:min_data_len_adj],
                        source_timestamps[:min_data_len_adj],
                        config.seq_length,
                        config.pred_length,
                        config
                    )
                    source_adjacency_matrix = dataset_instance_adj.get_adjacency_matrix()
                else:
                    source_adjacency_matrix = None

            except FileNotFoundError:
                raise FileNotFoundError(f"METR-LA.csv not found in {input_dir}, required for transfer learning")

            # 2. Load target dataset (Mozambique or South Africa)
            # target_dataset_name = getattr(config, 'target_dataset_name', 'clean_616_traffic station_data.csv')
            target_dataset_name = getattr(config, 'target_dataset_name', 'METR-CPT-Rolling-Mean-Expanded-v2-NoAugmentation.csv')
            target_data_path = getattr(config, 'target_data_path', f'{target_dataset_name}_traffic.csv')

            try:
                target_data_full_path = os.path.join(input_dir, target_data_path)
                if not os.path.exists(target_data_full_path):
                    raise FileNotFoundError(f"Target dataset file {target_data_path} not found in {input_dir}")

                target_df = pd.read_csv(target_data_full_path, index_col=0, parse_dates=True)
                print(f"Loaded {target_dataset_name} data with shape: {target_df.shape}")

                # Store target sensor IDs
                target_sensor_ids = target_df.columns.tolist()

                # Prepare target data
                target_data, target_timestamps, target_scaler, target_num_features = prepare_data(target_df, config)

                # Split into train and validation sets
                target_val_size = int(len(target_data) * 0.2)
                target_train_data = target_data[:-target_val_size]
                target_val_data = target_data[-target_val_size:]
                target_train_times = target_timestamps[:-target_val_size]
                target_val_times = target_timestamps[-target_val_size:]

                required_len = config.seq_length + config.pred_length -1 # Minimum length needed for __len__ >= 0
                if len(target_train_data) <= required_len:
                    raise ValueError(f"Insufficient training data samples in '{target_dataset_name}' for transfer learning. "
                                    f"Need at least {required_len + 1} samples after validation split "
                                    f"(seq_length={config.seq_length}, pred_length={config.pred_length}), "
                                    f"but got only {len(target_train_data)}. Check dataset size or config.")

                # Create datasets
                target_train_dataset = TrafficDataset(
                    target_train_data, target_train_times, config.seq_length, config.pred_length, config
                )

                target_val_dataset = TrafficDataset(
                    target_val_data, target_val_times, config.seq_length, config.pred_length, config
                )

                # Create data loaders
                target_train_loader = DataLoader(
                    target_train_dataset,
                    batch_size=config.batch_size,
                    shuffle=True,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                target_val_loader = DataLoader(
                    target_val_dataset,
                    batch_size=config.batch_size,
                    shuffle=False,
                    num_workers=config.num_workers,
                    pin_memory=config.pin_memory
                )

                print(f"Prepared target datasets - Train: {len(target_train_dataset)}, Val: {len(target_val_dataset)}")

                # Do NOT reuse source adjacency matrix when dimensions differ
                if config.use_spatial_features and config.use_gnn_pre_transformer:
                    print("Creating appropriate adjacency matrix for target dataset...")

                    # Get number of sensors in target dataset
                    num_target_sensors = target_df.shape[1]

                    # Option 1: Create k-nearest neighbor adjacency matrix based on temporal correlation
                    # This is better than fully connected as it captures actual structure
                    correlation_matrix = target_df.corr().values

                    # Convert to PyTorch tensor
                    correlation_tensor = torch.tensor(correlation_matrix, dtype=torch.float32)

                    # Create kNN graph - connect each node to its top k correlated neighbors
                    k = min(10, num_target_sensors - 1)  # Connect to 10 nearest or all if fewer

                    # Create an empty adjacency matrix
                    target_adjacency_matrix = torch.zeros((num_target_sensors, num_target_sensors))

                    # For each sensor, find the k most correlated sensors and connect them
                    for i in range(num_target_sensors):
                        # Get correlation values for this sensor (excluding self)
                        corr_values = correlation_tensor[i].clone()
                        corr_values[i] = -1  # Exclude self-connection temporarily

                        # Find the k highest correlations
                        _, top_indices = torch.topk(corr_values, k)

                        # Connect this sensor to its k most correlated neighbors
                        target_adjacency_matrix[i, top_indices] = 1

                    # Make the adjacency matrix symmetric (undirected graph)
                    target_adjacency_matrix = torch.max(target_adjacency_matrix, target_adjacency_matrix.t())

                    # Add self-loops for GCN
                    target_adjacency_matrix = target_adjacency_matrix + torch.eye(num_target_sensors)

                    # Normalize adjacency matrix for GCN
                    row_sum = target_adjacency_matrix.sum(1)
                    d_inv_sqrt = torch.pow(row_sum, -0.5)
                    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0
                    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
                    target_adjacency_matrix = d_mat_inv_sqrt @ target_adjacency_matrix @ d_mat_inv_sqrt

                    # Move to device
                    target_adjacency_matrix = target_adjacency_matrix.to(device)

                    print(f"Created correlation-based k-NN adjacency matrix with shape {target_adjacency_matrix.shape}")
                    print(f"Average connectivity: {(target_adjacency_matrix > 0).float().mean().item():.4f}")
                else:
                    target_adjacency_matrix = None

            except Exception as e:
                raise RuntimeError(f'Error loading target dataset: {str(e)}')

            # 3. Initialize model parameters
            model_params = {
                'input_dim': source_test_dataset[0][0].shape[1],  # Use sample from source dataset
                'hidden_dim': config.hidden_dim,
                'num_layers': config.num_layers,
                'num_heads': config.num_heads,
                'num_features': source_num_features,
                'dropout': config.dropout,
                'ff_dim_multiplier': config.ff_dim_multiplier,
                'activation': config.activation,
                'decoder_type': config.decoder_type,
                'use_gnn_pre_transformer': config.use_gnn_pre_transformer,
                'spatial_feature_dim': config.spatial_feature_dim,
                'gnn_type': config.gnn_type,
                'pred_len': config.pred_length,
            }

            # Ensure model dimensions are compatible
            model_params, config = ensure_compatible_dimensions(model_params, config)

            # 4. Load pre-trained model
            # source_model_path = getattr(config, 'source_model_path', None)
            # if source_model_path:
            #     pretrained_path = os.path.join(model_dir, source_model_path)
            # else:
            #     # Try to find the most recent best model file
            #     model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model_') and f.endswith('.pth')]
            #     if model_files:
            #         # Sort by modification time (newest first)
            #         model_files.sort(key=lambda x: os.path.getmtime(os.path.join(model_dir, x)), reverse=True)
            #         pretrained_path = os.path.join(model_dir, model_files[0])
            #     else:
            #         raise FileNotFoundError("No pre-trained model found. Please specify source_model_path or ensure a best_model_*.pth file exists.")

            # try:
            #     checkpoint = torch.load(pretrained_path, map_location=device)
            #     model = TrafficTransformer(**model_params).to(device)
            #     model.load_state_dict(checkpoint['model_state_dict'])
            #     print(f"Loaded pre-trained model from {pretrained_path}")
            # except Exception as e:
            #     raise RuntimeError(f"Error loading pre-trained model: {str(e)}")


            source_model_path = getattr(config, 'source_model_path', None)
            if source_model_path:
                pretrained_path = os.path.join(model_dir, source_model_path)
            else:
                # Try to find the most recent best model file
                model_files = [f for f in os.listdir(model_dir) if f.startswith('best_model_') and f.endswith('.pth')]
                if model_files:
                    # Sort by modification time (newest first)
                    model_files.sort(key=lambda x: os.path.getmtime(os.path.join(model_dir, x)), reverse=True)
                    pretrained_path = os.path.join(model_dir, model_files[0])
                else:
                    raise FileNotFoundError("No pre-trained model found. Please specify source_model_path or ensure a best_model_*.pth file exists.")

            # First, calculate actual dimensions for target dataset
            try:
                # Create a sample from target dataset to determine dimensions
                target_sample_x, _ = target_train_dataset[0]
                actual_input_dim = target_sample_x.shape[1]  # Input dimension including all features
                print(f"Target dataset input dimension: {actual_input_dim}")
            except Exception as dim_error:
                print(f"Warning: Could not determine target input dimension from dataset: {dim_error}")
                # Fallback - estimate input dimension
                actual_input_dim = target_num_features
                if config.use_time_features:
                    actual_input_dim += 4  # hour, day, week, month
                if config.use_holiday_feature:
                    actual_input_dim += 1
                if config.use_weather_feature:
                    actual_input_dim += 8  # Typical weather feature dimension
                print(f"Using estimated target input dimension: {actual_input_dim}")

            # Now load the model with the dimension information already determined
            try:
                # Load the checkpoint
                checkpoint = torch.load(pretrained_path, map_location=device)

                # Check if the checkpoint contains config
                source_pred_length = 1  # Default/fallback to 1 based on error message
                if 'config' in checkpoint:
                    # Use the pred_length from the checkpoint if available
                    checkpoint_config = checkpoint['config']
                    if 'pred_length' in checkpoint_config:
                        source_pred_length = checkpoint_config['pred_length']
                        print(f"Found pred_length={source_pred_length} in checkpoint config")
                else:
                    print(f"No config in checkpoint, using pred_length={source_pred_length}")

                # Initialize source dimensions with fallbacks
                source_input_dim = 219  # From previous log
                source_output_dim = 207  # From error message

                # Determine if the source model used GNN from state_dict keys
                has_gnn = any('gnn_encoder' in key for key in checkpoint['model_state_dict'].keys())

                # Source model params - maintain original architecture
                source_model_params = model_params.copy()
                source_model_params['input_dim'] = source_input_dim
                source_model_params['num_features'] = source_output_dim
                # CRITICAL FIX: Use the pred_length from the source model
                source_model_params['pred_len'] = source_pred_length
                source_model_params['use_gnn_pre_transformer'] = has_gnn

                print(f"\n=== Source Model Configuration ===")
                for key, value in source_model_params.items():
                    print(f"  {key}: {value}")

                # Create a model with the SOURCE dimensions and params
                source_model = TrafficTransformer(**source_model_params).to(device)

                try:
                    # Now loading should work with correct dimensions
                    source_model.load_state_dict(checkpoint['model_state_dict'])
                    print(f"Successfully loaded pre-trained model state dict")
                except Exception as load_error:
                    print(f"Error loading model state dict: {load_error}")
                    print("Creating manual mapping for compatible parameters...")

                    # Manual parameter loading for what can be copied
                    for name, param in source_model.named_parameters():
                        if name in checkpoint['model_state_dict']:
                            checkpoint_param = checkpoint['model_state_dict'][name]
                            if param.shape == checkpoint_param.shape:
                                param.data.copy_(checkpoint_param)
                                print(f"  Copied {name} with shape {param.shape}")
                            else:
                                print(f"  Shape mismatch for {name}: {param.shape} vs {checkpoint_param.shape}")

                # Get target dataset dimensions
                target_sample, _ = target_train_dataset[0]
                target_input_dim = target_sample.shape[1]  # Should be 136
                target_output_dim = target_num_features    # Should be 124

                # IMPORTANT: Create target model with SAME pred_length as source
                target_model_params = source_model_params.copy()

                # Create a new target model (identical to source initially)
                target_model = TrafficTransformer(**target_model_params).to(device)

                # Copy parameters from source to target
                target_model.load_state_dict(source_model.state_dict())

                # Now add adapters to handle the dimension mismatch
                target_model.add_transfer_adapters(
                    source_input_dim=source_input_dim,
                    source_output_dim=source_output_dim,
                    target_input_dim=target_input_dim,
                    target_output_dim=target_output_dim
                )

                # Set the target model as our model for further training
                model = target_model

                print(f"\n=== Transfer Learning Configuration ===")
                print(f"Source model: input_dim={source_input_dim}, output_dim={source_output_dim}, pred_length={source_pred_length}")
                print(f"Target data: input_dim={target_input_dim}, output_dim={target_output_dim}, pred_length={config.pred_length}")

                # Clean up source model to free memory
                del source_model
                torch.cuda.empty_cache()

            except Exception as e:
                import traceback
                traceback.print_exc()
                raise RuntimeError(f"Error loading pre-trained model: {str(e)}")


            # 5. Set up transfer learning configuration
            freeze_encoder = getattr(config, 'freeze_encoder', True)
            freeze_layers = getattr(config, 'freeze_layers', 1)
            adapter_dim = getattr(config, 'adapter_dim', 0)

            # Initialize transfer learning module
            transfer_module = TransferLearningModule(
                base_model=model,
                config=config,
                target_dataset_name=target_dataset_name,
                freeze_encoder=freeze_encoder,
                freeze_layers=freeze_layers,
                adapter_dim=adapter_dim
            )

            # 6. Fine-tune the model
            print("\n=== Starting Transfer Learning Fine-tuning ===")
            fine_tuned_model, history, metrics = train_transfer_model(
                model=model,
                train_loader=target_train_loader,
                val_loader=target_val_loader,
                source_loader=source_test_loader,
                optimizer=None,  # Will be created in the function
                scheduler=None,  # Will be created in the function
                criterion=None,  # Will be created in the function
                config=config,
                data_scaler=target_scaler,
                source_scaler=source_scaler,
                device=device,
                adjacency_matrix=target_adjacency_matrix
            )

            # 7. Generate visualizations for specific sensors
            print("\n=== Generating Sensor-specific Visualizations ===")

            # Make predictions on target validation set
            print("Generating predictions for target dataset...")
            target_predictions, target_actuals = predict(
                model=fine_tuned_model,
                dataloader=target_val_loader,
                scaler=target_scaler,
                device=device,
                adjacency_matrix=target_adjacency_matrix,
                config=config
            )

            # Plot predictions for a few target sensors
            for sensor_idx in range(min(100, target_actuals.shape[1])): # MODIFICATION
                if sensor_idx < len(target_sensor_ids):
                    actual_sensor_id = target_sensor_ids[sensor_idx]
                    plot_predictions_vs_actual(
                        actuals=target_actuals,
                        predictions=target_predictions,
                        sensor_index=sensor_idx,
                        sensor_id=actual_sensor_id,
                        fold=0,  # Not using folds for transfer learning
                        results_dir=results_dir,
                        pred_len=config.pred_length,
                        config=config
                    )

            # Make predictions on source test set
            print("Generating predictions for source dataset...")
            source_predictions, source_actuals = predict(
                model=fine_tuned_model,
                dataloader=source_test_loader,
                scaler=source_scaler,
                device=device,
                adjacency_matrix=source_adjacency_matrix,
                config=config,
                is_source_dataset=True
            )

            # Plot attention weights
            TransferLearningModule.visualize_attention_transfer(
                base_model=fine_tuned_model,
                source_loader=source_test_loader,
                target_loader=target_val_loader,
                output_dir=results_dir
            )

            # 8. Generate summary report
            print("\n=== Generating Transfer Learning Summary Report ===")

            # Create a comparison dataframe
            if metrics['source'] and metrics['target']:
                report_data = {
                    'Dataset': ['METR-LA (Source)', f'{target_dataset_name.title()} (Target)'],
                    'MAE': [metrics['source']['mae'], metrics['target']['mae']],
                    'RMSE': [metrics['source']['rmse'], metrics['target']['rmse']],
                    'R²': [metrics['source']['r2'], metrics['target']['r2']],
                    'MAPE (%)': [metrics['source']['mape'], metrics['target']['mape']]
                }

                # Calculate improvement percentages
                improvement = {
                    'MAE': (metrics['target']['mae'] - metrics['source']['mae']) / metrics['source']['mae'] * 100,
                    'RMSE': (metrics['target']['rmse'] - metrics['source']['rmse']) / metrics['source']['rmse'] * 100,
                    'R²': (metrics['target']['r2'] - metrics['source']['r2']) / max(0.001, abs(metrics['source']['r2'])) * 100,
                    'MAPE': (metrics['target']['mape'] - metrics['source']['mape']) / max(0.001, metrics['source']['mape']) * 100
                }

                # Add a row for improvement percentages
                report_data['Dataset'].append('Improvement (%)')
                report_data['MAE'].append(improvement['MAE'])
                report_data['RMSE'].append(improvement['RMSE'])
                report_data['R²'].append(improvement['R²'])
                report_data['MAPE (%)'].append(improvement['MAPE'])

                # Create dataframe and save to CSV
                df_report = pd.DataFrame(report_data)
                timestamp = get_maputo_timestamp()
                report_path = os.path.join(results_dir, f'transfer_learning_report_{target_dataset_name}_{timestamp}.csv')
                df_report.to_csv(report_path, index=False)

                print(f"Transfer learning summary report saved to {report_path}")

                # Print summary to console
                print("\n=== TRANSFER LEARNING SUMMARY ===")
                print(f"Source dataset: METR-LA, Target dataset: {target_dataset_name.title()}")
                print(f"MAE  - Source: {metrics['source']['mae']:.4f}, Target: {metrics['target']['mae']:.4f}, Change: {improvement['MAE']:.2f}%")
                print(f"RMSE - Source: {metrics['source']['rmse']:.4f}, Target: {metrics['target']['rmse']:.4f}, Change: {improvement['RMSE']:.2f}%")
                print(f"R²   - Source: {metrics['source']['r2']:.4f}, Target: {metrics['target']['r2']:.4f}, Change: {improvement['R²']:.2f}%")
                print(f"MAPE - Source: {metrics['source']['mape']:.2f}%, Target: {metrics['target']['mape']:.2f}%, Change: {improvement['MAPE']:.2f}%")

            else:
                print("Incomplete metrics for source or target dataset, cannot generate complete report")

        elif not run_only_transfer:  # Skip standard training if run_only_transfer is True
            # =====================================================
            # STANDARD TRAINING PATH
            # =====================================================
            print("\n=== Running Standard Training Workflow ===\n")

            # --- Load and Preprocess Data ---
            try:
                # Load the main dataset
                df = pd.read_csv(os.path.join(input_dir, 'METR-LA.csv'), index_col=0, parse_dates=True)
                print(f"Loaded METR-LA data with shape: {df.shape}")
                print(f"Sensor IDs (columns): {df.columns.tolist()[:5]}...") # Print first few sensor IDs

                # Copy weather data to input directory if needed
                weather_src = config.weather_data_file  # Source file path
                weather_dest = os.path.join(input_dir, os.path.basename(config.weather_data_file))

                if not os.path.exists(weather_dest):
                    # Try to find the weather file in the current directory
                    if os.path.exists(weather_src):
                        import shutil
                        shutil.copyfile(weather_src, weather_dest)
                        print(f"Weather data copied to {weather_dest}")
                    else:
                        # If file not found in current directory, try to find it elsewhere
                        possible_locations = [
                            '.',  # Current directory
                            './data',  # Data subdirectory
                            # os.path.dirname(os.path.abspath(__file__)),  # Script directory - may cause issues in notebooks
                            '/content'  # Colab root directory
                        ]
                        weather_found_and_copied = False
                        for location in possible_locations:
                            potential_path = os.path.join(location, os.path.basename(config.weather_data_file)) # Use basename here
                            if os.path.exists(potential_path):
                                import shutil
                                try:
                                    shutil.copyfile(potential_path, weather_dest)
                                    print(f"Weather data found at {potential_path} and copied to {weather_dest}")
                                    weather_found_and_copied = True
                                    break
                                except Exception as copy_e:
                                    print(f"Found weather file at {potential_path}, but failed to copy: {copy_e}")

                        if not weather_found_and_copied:
                            print(f"Weather data file not found after checking several locations. Expected source: {weather_src}")
                            print("Will proceed, but weather features might rely on simulated data if WeatherIntegration fails.")

            except FileNotFoundError:
                raise FileNotFoundError(f"METR-LA.csv not found in {input_dir}, please place the dataset file there")

            data_normalized, timestamps, data_scaler, num_features = prepare_data(df, config)

            # --- Get Sensor IDs ---
            # Store sensor IDs for later use in plotting filenames/titles
            sensor_ids = df.columns.tolist() # Get the list of sensor IDs from the DataFrame

            # --- Adjacency Matrix Preparation ---
            if config.use_spatial_features and config.use_gnn_pre_transformer:
                if not TORCH_GEOMETRIC_AVAILABLE:
                    raise ImportError("PyTorch Geometric is required for GNN pre-transformer but not available")

                # Use a small slice of data just for getting adjacency matrix if needed
                min_data_len_adj = config.seq_length + config.pred_length
                if len(data_normalized) >= min_data_len_adj:
                    dataset_instance_adj = TrafficDataset(data_normalized[:min_data_len_adj], timestamps[:min_data_len_adj], config.seq_length, config.pred_length, config)
                    adjacency_matrix = dataset_instance_adj.get_adjacency_matrix()
                else:
                    print("Warning: Not enough data to create dataset for adjacency matrix. Setting adjacency_matrix to None.")
                    adjacency_matrix = None
            else:
                adjacency_matrix = None

            # --- Model Parameters ---
            # Create a test dataset instance to get actual feature dimensions
            min_data_len = config.seq_length + config.pred_length
            if len(data_normalized) < min_data_len:
                raise ValueError(f"Not enough data ({len(data_normalized)}) to create a sample with seq_length={config.seq_length} and pred_length={config.pred_length}.")

            # Make sure weather file path in config points to the correct location in input_dir if needed for dataset creation
            # This path might have been updated by setup_directories, double check it points to input_dir
            potential_weather_path_in_input = os.path.join(input_dir, os.path.basename(config.weather_data_file))
            if os.path.exists(potential_weather_path_in_input):
                config.weather_data_file = potential_weather_path_in_input
            else:
                print(f"Warning: Weather file {potential_weather_path_in_input} not found in input_dir for test dataset creation.")


            print(f"Creating test dataset to determine input dim with weather file: {config.weather_data_file}")
            try:
                test_dataset = TrafficDataset(data_normalized[:min_data_len], timestamps[:min_data_len], config.seq_length, config.pred_length, config)
                if len(test_dataset) == 0:
                     raise ValueError("Test dataset is unexpectedly empty after creation. Check seq_length/pred_length vs data size.")

                sample_x, _ = test_dataset[0]
                actual_input_dim = sample_x.shape[1] # This *already* includes all concatenated features from TrafficDataset.__getitem__
            except Exception as dataset_init_error:
                 raise RuntimeError(f"Failed to create test TrafficDataset to determine input dimension: {dataset_init_error}")


            # Use the actual_input_dim derived from the dataset sample directly
            model_params = {
                'input_dim': actual_input_dim, # Use the dimension from the dataset sample
                'hidden_dim': config.hidden_dim,
                'num_layers': config.num_layers,
                'num_heads': config.num_heads,
                'num_features': num_features,  # Pass explicit number of features (output features)
                'dropout': config.dropout,
                'ff_dim_multiplier': config.ff_dim_multiplier,
                'activation': config.activation,
                'decoder_type': config.decoder_type,
                'use_gnn_pre_transformer': config.use_gnn_pre_transformer,
                'spatial_feature_dim': config.spatial_feature_dim, # Still needed if GNN is used
                'gnn_type': config.gnn_type,
                'pred_len': config.pred_length,
            }

            # Dimensions log update
            print(f"Actual input dimension determined from DataLoader sample: {actual_input_dim}")
            print(f"Using input_dim={model_params['input_dim']} for model creation") # Should now match actual_input_dim

            # Automatically find optimal batch size if enabled
            if config.find_optimal_batch_size and device == 'cuda' and torch.cuda.is_available():
                print("\n=== Finding Optimal Batch Size ===")
                # Create a dummy input/target on CPU first
                dummy_input_cpu = torch.randn(1, config.seq_length, model_params['input_dim'])
                dummy_target_cpu = torch.randn(1, config.pred_length, num_features)

                # Create a temporary model instance for testing on CPU first
                temp_model_params_bs, config_bs = ensure_compatible_dimensions(model_params.copy(), config) # Use copies
                try:
                    temp_model = TrafficTransformer(**temp_model_params_bs) # Create on CPU

                    # Find optimal batch size (function handles moving model/data to GPU)
                    optimal_batch_size = find_optimal_batch_size(
                        temp_model,
                        dummy_input_cpu, # Pass CPU tensor
                        dummy_target_cpu, # Pass CPU tensor
                        max_batch_size=2048,  # Upper limit to test
                        start_batch=32        # Starting test size
                    )
                    print(f"Optimal batch size for GPU memory: {optimal_batch_size}")
                     # Update configuration with optimal batch size
                    config.batch_size = optimal_batch_size

                except Exception as bs_error:
                     print(f"Error during optimal batch size search: {bs_error}. Using default batch size {config.batch_size}.")
                finally:
                     # Clean up
                    del temp_model, dummy_input_cpu, dummy_target_cpu, temp_model_params_bs, config_bs
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()


            # --- Optuna Hyperparameter Optimization ---
            if OPTUNA_AVAILABLE and config.optuna_trials is not None and config.optuna_trials > 0:
                # Define the objective function with proper config access
                def objective(trial):
                    # We need to declare config as nonlocal since it's from the outer scope
                    nonlocal config, model_params, data_normalized, timestamps, device, adjacency_matrix, num_features

                    # Define hyperparameter search space
                    num_heads = trial.suggest_categorical('num_heads', [4, 8, 16])

                    # Ensure hidden_dim is both divisible by num_heads and is even
                    hidden_dim_base = trial.suggest_int('hidden_dim', 64, 1024)
                    # Adjust to ensure divisibility by num_heads
                    hidden_dim = (hidden_dim_base // num_heads) * num_heads
                    if hidden_dim == 0: hidden_dim = num_heads # Ensure not zero
                    # Adjust to ensure it's even
                    if hidden_dim % 2 != 0:
                        hidden_dim += num_heads # Add num_heads to maintain divisibility and make it even

                    # Create a copy of the config to avoid modifying the original
                    trial_config = TrainingConfig( # Re-create with necessary defaults
                        base_output_dir=config.base_output_dir, # Keep base dir
                        # Trial-specific params
                        hidden_dim=hidden_dim,
                        num_layers=trial.suggest_int('num_layers', 2, 6),
                        num_heads=num_heads,
                        dropout=trial.suggest_float('dropout', 0.0, 0.5),
                        learning_rate=trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True),
                        decoder_type=trial.suggest_categorical('decoder_type', ['linear', 'mlp']),
                        # Inherited params
                        batch_size=config.batch_size, # Use found or default batch size
                        seq_length=config.seq_length,
                        pred_length=config.pred_length,
                        num_epochs=min(config.num_epochs, 10), # Limit epochs for Optuna trial
                        patience=config.patience // 2, # Shorter patience for trials
                        ff_dim_multiplier=config.ff_dim_multiplier,
                        activation=config.activation,
                        data_scaler_type=config.data_scaler_type,
                        optimizer_type=config.optimizer_type, # Can be tuned if needed
                        loss_function=config.loss_function, # Can be tuned if needed
                        use_time_features=config.use_time_features,
                        use_holiday_feature=config.use_holiday_feature,
                        holiday_country_code=config.holiday_country_code,
                        use_weather_feature=config.use_weather_feature,
                        weather_feature_type=config.weather_feature_type,
                        weather_data_file=config.weather_data_file, # Use updated path
                        gradient_clip=config.gradient_clip,
                        scheduler_type=config.scheduler_type, # Can be tuned if needed
                        scheduler_patience=config.scheduler_patience // 2,
                        scheduler_factor=config.scheduler_factor,
                        step_scheduler_step_size=config.step_scheduler_step_size,
                        step_scheduler_gamma=config.step_scheduler_gamma,
                        use_lagged_features=config.use_lagged_features,
                        num_lags=config.num_lags,
                        use_spatial_features=config.use_spatial_features,
                        spatial_feature_dim=config.spatial_feature_dim,
                        use_gnn_pre_transformer=config.use_gnn_pre_transformer,
                        gnn_type=config.gnn_type,
                        use_quantile_regression=config.use_quantile_regression,
                        quantiles=config.quantiles,
                        optuna_trials=0, # Disable nested Optuna
                        warmup_epochs=config.warmup_epochs // 2,
                        use_mixed_precision=config.use_mixed_precision,
                        accumulation_steps=config.accumulation_steps,
                        num_workers=min(config.num_workers, 2), # Reduce workers for trials
                        pin_memory=config.pin_memory,
                        find_optimal_batch_size=False, # Already done
                        monitor_gpu_usage=False, # Disable monitoring for trials
                    )

                    # Copy directory paths from main config
                    trial_config.input_dir = config.input_dir
                    trial_config.output_dir = config.output_dir
                    trial_config.model_dir = config.model_dir # Use main model dir (or could create subdirs)
                    trial_config.results_dir = config.results_dir # Use main results dir

                    print(f"\n--- Optuna Trial ---")
                    print(f"Params: hidden_dim={hidden_dim}, num_layers={trial_config.num_layers}, num_heads={num_heads}, lr={trial_config.learning_rate:.5f}, dropout={trial_config.dropout:.3f}, decoder={trial_config.decoder_type}")

                    # Copy model parameters but with trial values
                    trial_model_params = model_params.copy()
                    trial_model_params['hidden_dim'] = hidden_dim
                    trial_model_params['num_layers'] = trial_config.num_layers
                    trial_model_params['num_heads'] = num_heads
                    trial_model_params['dropout'] = trial_config.dropout
                    trial_model_params['decoder_type'] = trial_config.decoder_type

                    # Ensure dimensions are compatible FOR THE TRIAL
                    trial_model_params, trial_config = ensure_compatible_dimensions(trial_model_params, trial_config)

                    # Check constraints again after potential adjustments
                    assert trial_model_params['hidden_dim'] % trial_model_params['num_heads'] == 0, \
                        f"Trial Error: Hidden dimension {trial_model_params['hidden_dim']} must be divisible by number of heads {trial_model_params['num_heads']}"
                    assert trial_model_params['hidden_dim'] % 2 == 0, \
                        f"Trial Error: Hidden dimension {trial_model_params['hidden_dim']} must be even."

                    try:
                        # Create model with trial parameters
                        model_optuna = TrafficTransformer(**trial_model_params).to(device)

                        # Initialize optimizer and scheduler
                        optimizer_optuna = optim.AdamW(model_optuna.parameters(), lr=trial_config.learning_rate)
                        scheduler_optuna = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer_optuna, patience=trial_config.patience // 2, factor=0.5
                        ) if trial_config.scheduler_type == 'plateau' else None # Example scheduler

                        # Choose criterion based on loss function
                        if trial_config.loss_function == 'mse':
                            criterion_optuna = nn.MSELoss()
                        elif trial_config.loss_function == 'mae':
                            criterion_optuna = nn.L1Loss()
                        # Add other loss functions if needed for Optuna trials
                        else:
                            criterion_optuna = nn.MSELoss()

                        # Time Series Cross-Validation (Single Split for Optuna speed)
                        tscv_optuna = TimeSeriesSplit(n_splits=2) # Use 2 splits to get one validation set
                        all_indices = np.arange(len(data_normalized))
                        train_idx_optuna, val_idx_optuna = list(tscv_optuna.split(all_indices))[-1] # Take the last split

                        train_data_optuna = data_normalized[train_idx_optuna]
                        val_data_optuna = data_normalized[val_idx_optuna]
                        train_times_optuna = timestamps[train_idx_optuna]
                        val_times_optuna = timestamps[val_idx_optuna]

                        # Ensure dataset creation uses the trial_config
                        train_dataset_optuna = TrafficDataset(
                            train_data_optuna, train_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                        )
                        val_dataset_optuna = TrafficDataset(
                            val_data_optuna, val_times_optuna, trial_config.seq_length, trial_config.pred_length, trial_config
                        )

                        # Use smaller batch size for trials to avoid memory issues if needed
                        trial_batch_size = min(trial_config.batch_size, 64)  # Limiting batch size for trials

                        train_loader_optuna = DataLoader(
                            train_dataset_optuna, batch_size=trial_batch_size, shuffle=True,
                            num_workers=trial_config.num_workers, # Use reduced workers
                            pin_memory=trial_config.pin_memory
                        )
                        val_loader_optuna = DataLoader(
                            val_dataset_optuna, batch_size=trial_batch_size, shuffle=False,
                            num_workers=trial_config.num_workers, # Use reduced workers
                            pin_memory=trial_config.pin_memory
                        )

                        # Train the model (ensure train_model uses trial_config)
                        print(f"Starting Optuna trial training (Max epochs: {trial_config.num_epochs})...")
                        trained_model_optuna, _, _ = train_model(
                            model=model_optuna,
                            train_loader=train_loader_optuna,
                            val_loader=val_loader_optuna,
                            optimizer=optimizer_optuna,
                            scheduler=scheduler_optuna,
                            criterion=criterion_optuna,
                            config=trial_config,
                            device=device,
                            adjacency_matrix=adjacency_matrix, # Pass TRIAL config
                            data_scaler=data_scaler
                        )

                        # Evaluate the model (ensure evaluate_model uses trial_config)
                        avg_val_loss_optuna, _ = evaluate_model(
                            model=trained_model_optuna,
                            dataloader=val_loader_optuna,
                            criterion=criterion_optuna,
                            device=device,
                            config=trial_config,
                            adjacency_matrix=adjacency_matrix, # Pass TRIAL config
                            data_scaler=data_scaler,
                        )

                        print(f"--- Optuna Trial Completed --- Loss: {avg_val_loss_optuna:.6f}\n")
                        # Clean up trial resources
                        del model_optuna, optimizer_optuna, scheduler_optuna, criterion_optuna
                        del train_dataset_optuna, val_dataset_optuna, train_loader_optuna, val_loader_optuna
                        gc.collect()
                        if torch.cuda.is_available(): torch.cuda.empty_cache()

                        return avg_val_loss_optuna

                    except Exception as e:
                        print(f"--- Optuna Trial FAILED --- Error: {str(e)}\n")
                         # Clean up potential partial resources
                        gc.collect()
                        if torch.cuda.is_available(): torch.cuda.empty_cache()
                        # Return a high value to indicate failure
                        # Consider using optuna.TrialPruned() if appropriate
                        return float('inf')


                # Create and run Optuna study
                study = optuna.create_study(direction='minimize')
                print(f"\n=== Starting Optuna Hyperparameter Search ({config.optuna_trials} trials) ===")

                try:
                    # Pass necessary objects (like data) if objective function needs them indirectly
                    study.optimize(objective, n_trials=config.optuna_trials)

                    # Check if we have valid completed trials
                    completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

                    if completed_trials:
                         # Get best parameters from completed trials
                        best_params = study.best_params
                        print(f"\nOptuna finished. Best hyperparameters found: {best_params}")
                        print(f"Best validation loss achieved: {study.best_value:.6f}")

                        # Update MAIN config with best parameters found by Optuna
                        config.num_heads = best_params['num_heads']
                        config.hidden_dim = best_params['hidden_dim'] # Use the adjusted value from objective
                        config.num_layers = best_params['num_layers']
                        config.dropout = best_params['dropout']
                        config.learning_rate = best_params['learning_rate']
                        config.decoder_type = best_params['decoder_type']

                        # Update MAIN model_params dictionary as well
                        model_params['hidden_dim'] = config.hidden_dim
                        model_params['num_layers'] = config.num_layers
                        model_params['num_heads'] = config.num_heads
                        model_params['dropout'] = config.dropout
                        model_params['decoder_type'] = config.decoder_type

                        # CRITICAL: Re-ensure compatibility for the main config after Optuna update
                        print("Re-validating dimensions after Optuna...")
                        model_params, config = ensure_compatible_dimensions(model_params, config)
                        print(f"Final model params after Optuna & validation: Heads={model_params['num_heads']}, Hidden={model_params['hidden_dim']}")

                    else:
                        print("\nNo successful Optuna trials completed. Using original parameters.")

                except Exception as e:
                    print(f"Optuna optimization encountered an error: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    print("Continuing with original parameters...")
                    # Ensure dimensions are still compatible even if Optuna failed midway
                    model_params, config = ensure_compatible_dimensions(model_params, config)

                print("============================================================\n")

            else:
                if not OPTUNA_AVAILABLE:
                    warnings.warn("Optuna not available, skipping hyperparameter tuning")
                else:
                    print("Optuna hyperparameter tuning disabled (optuna_trials is None or 0)")

            # --- Time Series Cross-Validation ---
            tscv = TimeSeriesSplit(n_splits=5)
            fold_metrics = []
            baseline_metrics = {'naive_forecast': []} # Keep simplified baseline dict

            # Initialize variables to store results from the last successful fold for the final report
            last_trained_model = None
            last_test_loader = None
            last_fold_train_losses = []
            last_fold_val_losses = []

            try:
                # Iterate through time series folds
                for fold, (train_idx, test_idx) in enumerate(tscv.split(data_normalized)):
                    print(f"\n=== Fold {fold+1}/{tscv.get_n_splits()} ===")
                    print(f"Config: {config.data_scaler_type}, {config.optimizer_type}, {config.loss_function}, {config.pred_length}-step ahead")
                    print(f"Model Params: Heads={model_params['num_heads']}, Hidden={model_params['hidden_dim']}, Layers={model_params['num_layers']}, Dropout={model_params['dropout']:.3f}")
                    print(f"LR={config.learning_rate:.5f}, Batch={config.batch_size}, Accumulation={config.accumulation_steps}")
                    print(f"Weather features: {'Enabled' if config.use_weather_feature else 'Disabled'} ({config.weather_feature_type if config.use_weather_feature else 'N/A'})")

                    # Split data
                    train_data = data_normalized[train_idx]
                    test_data = data_normalized[test_idx]
                    train_times = timestamps[train_idx]
                    test_times = timestamps[test_idx]

                    # Create datasets for this fold using the main config
                    try:
                        train_dataset = TrafficDataset(train_data, train_times, config.seq_length, config.pred_length, config)
                        test_dataset = TrafficDataset(test_data, test_times, config.seq_length, config.pred_length, config)
                        print(f"Dataset created - Train: {len(train_dataset)} samples, Test: {len(test_dataset)} samples")
                        if len(train_dataset) == 0 or len(test_dataset) == 0:
                            print("Warning: Empty dataset created for this fold, skipping fold.")
                            continue # Skip to the next fold
                    except Exception as dataset_fold_error:
                        print(f"Error creating dataset for fold {fold+1}: {dataset_fold_error}. Skipping fold.")
                        continue

                    # Enhanced DataLoader configuration with workers and pinned memory
                    train_loader = DataLoader(
                        train_dataset,
                        batch_size=config.batch_size,
                        shuffle=True,
                        num_workers=config.num_workers,
                        pin_memory=config.pin_memory,
                        prefetch_factor=2 if config.num_workers > 0 else None, # Prefetch only if workers > 0
                        drop_last=True # Drop last incomplete batch for stability
                    )
                    test_loader = DataLoader(
                        test_dataset,
                        batch_size=config.batch_size,
                        shuffle=False,
                        num_workers=config.num_workers,
                        pin_memory=config.pin_memory
                    )

                    # Double-check dimension compatibility before creating model for THIS fold
                    model_params, config = ensure_compatible_dimensions(model_params, config)

                    # Initialize model
                    model = TrafficTransformer(**model_params).to(device)

                    # Initialize optimizer
                    if config.optimizer_type == 'adam':
                        optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
                    elif config.optimizer_type == 'adamw':
                        optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)
                    else:
                        warnings.warn(f"Invalid optimizer type: {config.optimizer_type}, using AdamW")
                        optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate)

                    # Initialize loss function
                    if config.loss_function == 'mse':
                        criterion = nn.MSELoss()
                    elif config.loss_function == 'mae':
                        criterion = nn.L1Loss()
                    elif config.loss_function == 'quantile':
                        # Ensure quantiles are available in config
                        if not hasattr(config, 'quantiles') or not config.quantiles:
                            config.quantiles = [0.1, 0.5, 0.9] # Default quantiles
                            print(f"Warning: Quantiles not found in config, using default: {config.quantiles}")
                        criterion = lambda output, target: quantile_loss(output, target, config.quantiles)
                    elif config.loss_function == 'hybrid':
                         criterion = hybrid_loss # Assumes hybrid_loss is defined
                    else:
                        warnings.warn(f"Invalid loss function: {config.loss_function}, using MSELoss")
                        criterion = nn.MSELoss()

                    # Initialize scheduler
                    scheduler = None # Default to None
                    if config.scheduler_type == 'plateau':
                        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, mode='min', patience=config.scheduler_patience,
                            factor=config.scheduler_factor, verbose=True
                        )
                    elif config.scheduler_type == 'cosine':
                        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs)
                    elif config.scheduler_type == 'step':
                        scheduler = optim.lr_scheduler.StepLR(
                            optimizer, step_size=config.step_scheduler_step_size, gamma=config.step_scheduler_gamma
                        )
                    elif config.scheduler_type == 'cosine_warmup':
                         # Ensure warmup_epochs is valid
                         if not hasattr(config, 'warmup_epochs') or config.warmup_epochs <= 0:
                            config.warmup_epochs = max(1, config.num_epochs // 10) # Default warmup
                            print(f"Warning: Invalid warmup_epochs, setting to {config.warmup_epochs}")
                         scheduler = CosineWarmupLR(
                            optimizer,
                            warmup_epochs=config.warmup_epochs,
                            total_epochs=config.num_epochs,
                            base_lr=config.learning_rate
                         )
                    elif config.scheduler_type is not None: # Catch invalid but not None types
                        warnings.warn(f"Invalid scheduler type: {config.scheduler_type}, no scheduler will be used.")


                    # Explicitly run garbage collection and clear CUDA cache before training
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    # Train model
                    print(f"Starting training for Fold {fold+1}...")
                    trained_model, fold_train_losses, fold_val_losses = train_model(
                        model=model,
                        train_loader=train_loader,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        criterion=criterion,
                        config=config,
                        device=device,
                        adjacency_matrix=adjacency_matrix, # Pass main config
                        val_loader=test_loader,
                        data_scaler=data_scaler
                    )

                    # Evaluate model
                    print(f"\nEvaluating Fold {fold+1}...")
                    test_loss, metrics = evaluate_model(
                        model=trained_model,
                        dataloader=test_loader,
                        criterion=criterion,
                        data_scaler=data_scaler,
                        device=device,
                        config=config,
                        adjacency_matrix=adjacency_matrix # Pass main config
                    )
                    fold_metrics.append(metrics)
                    mae, rmse, r2, mape = metrics

                    save_experiment_results(
                        config=config,
                        model_params=model_params,
                        metrics=metrics,
                        results_dir=results_dir,
                        fold=fold+1
                    )

                    print(f'Fold {fold+1} Test Metrics: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, MAPE={mape:.2f}%')

                    # --- Save predictions and plots using Actual Sensor IDs --- #
                    print(f"Generating predictions and plots for Fold {fold+1}...")
                    predictions, actuals = predict(
                        model=trained_model,
                        dataloader=test_loader,
                        scaler=data_scaler,
                        device=device,
                        adjacency_matrix=adjacency_matrix,
                        config=config
                    )

                    # MODIFIED Call: Pass actual sensor ID string
                    for sensor_idx in range(min(206, actuals.shape[1])):  # Plot first 206 sensors
                        if sensor_idx < len(sensor_ids): # Check index validity against sensor_ids list
                            actual_sensor_id = sensor_ids[sensor_idx] # Get the actual ID from the list
                            # print(f"  Plotting Sensor Index: {sensor_idx}, Sensor ID: {actual_sensor_id}") # Optional debug print
                            plot_predictions_vs_actual(
                                actuals=actuals,
                                predictions=predictions,
                                sensor_index=sensor_idx,
                                sensor_id=actual_sensor_id, # <<< Pass the actual sensor ID string
                                fold=fold,
                                results_dir=results_dir,
                                pred_len=config.pred_length,
                                config=config, # Pass main config
                            )
                        else:
                            print(f"  Warning: Sensor index {sensor_idx} out of bounds for sensor ID list (length {len(sensor_ids)}). Skipping plot.")

                    # Plot attention weights (using model from current fold)
                    plot_attention_weights(
                        trained_model, config.seq_length,
                        results_dir, config # Pass main config
                    )

                    # Plot training history (using losses from current fold)
                    plot_training_history(
                        fold_train_losses, fold_val_losses,
                        results_dir, config # Pass main config
                    )

                    # Train and evaluate baseline models (currently disabled in function)
                    print("\nHandling baseline models (currently disabled)...")
                    fold_baseline_metrics = train_baseline_models(
                        train_data,
                        test_data,
                        config,
                        device,
                        timestamps_train=train_times,
                        timestamps_test=test_times
                    )

                    # Add fold baseline metrics to overall baseline metrics
                    for model_name, metrics_list in fold_baseline_metrics.items():
                        if metrics_list:  # Only append if there are metrics
                            baseline_metrics.setdefault(model_name, []).extend(metrics_list)

                    # Store results from this successful fold for potential use in the final report
                    last_trained_model = trained_model
                    last_test_loader = test_loader
                    last_fold_train_losses = fold_train_losses
                    last_fold_val_losses = fold_val_losses

                    # Clean up fold-specific resources
                    del model, optimizer, criterion, scheduler, train_dataset, test_dataset, train_loader, test_loader
                    del predictions, actuals, train_data, test_data, train_times, test_times
                    gc.collect()
                    if torch.cuda.is_available(): torch.cuda.empty_cache()

            except Exception as e:
                print(f"\n--- Error during training loop (Fold {fold+1}) ---")
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()
                print("----------------------------------------------------")

            # --- Calculate and print average metrics ---
            if fold_metrics:
                avg_mae = np.mean([m[0] for m in fold_metrics])
                avg_rmse = np.mean([m[1] for m in fold_metrics])
                avg_r2 = np.mean([m[2] for m in fold_metrics])
                # Use nanmean for MAPE to handle potential NaNs if division by zero occurred
                avg_mape = np.nanmean([m[3] for m in fold_metrics])

                print(f"\n=== Average Metrics Across {len(fold_metrics)} Folds ===")
                print(f"Transformer MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, R²: {avg_r2:.4f}, MAPE: {avg_mape:.2f}%")

                # --- Print baseline metrics ---
                print("\n=== Average Baseline Metrics Across Folds ===")
                baseline_models_reported = False
                for model_name, metrics_list in baseline_metrics.items():
                    if metrics_list:  # Only calculate averages if there are metrics for this baseline
                        baseline_models_reported = True
                        avg_baseline_mae = np.mean([m[0] for m in metrics_list])
                        avg_baseline_rmse = np.mean([m[1] for m in metrics_list])
                        avg_baseline_r2 = np.mean([m[2] for m in metrics_list])
                        avg_baseline_mape = np.nanmean([m[3] for m in metrics_list]) # Use nanmean
                        print(f"Model: {model_name.upper()}")
                        print(f"  MAE: {avg_baseline_mae:.4f}, RMSE: {avg_baseline_rmse:.4f}, R²: {avg_baseline_r2:.4f}, MAPE: {avg_baseline_mape:.2f}%")
                if not baseline_models_reported:
                     print("No baseline model metrics were recorded.")

                # --- Generate final report ---
                # Ensure we have necessary components from the last successful fold
                if last_trained_model and last_test_loader:
                    try:
                        print("\nGenerating comprehensive traffic prediction report...")
                        # Use data from the last successful fold for the report context
                        report_path = generate_traffic_report(
                            model=last_trained_model,
                            test_loader=last_test_loader,
                            scaler=data_scaler,
                            config=config, # Pass the final (potentially Optuna-tuned) config
                            device=device,
                            train_losses=last_fold_train_losses, # Use losses from last fold
                            val_losses=last_fold_val_losses,     # Use losses from last fold
                            fold_metrics=fold_metrics,      # Use metrics from ALL folds
                            baseline_metrics=baseline_metrics, # Use ALL baseline metrics
                            results_dir=results_dir,
                            timestamp=get_maputo_timestamp()
                        )

                        if report_path and os.path.exists(report_path):
                            print(f"\nDetailed analysis report generated at: {report_path}")
                        else:
                            print("\nWarning: Failed to generate or find the detailed analysis report.")
                    except Exception as report_error:
                        print(f"\nError generating final report: {report_error}")
                        import traceback
                        traceback.print_exc()
                else:
                     print("\nSkipping final report generation: No successful fold completed or missing model/loader.")

            else:
                print("\nNo fold metrics available - training may have failed or all folds were skipped.")

            # Create summary plot with all model comparisons
            try:
                 # Check if there are transformer metrics and any baseline metrics to compare
                if fold_metrics and any(metrics for metrics in baseline_metrics.values()):
                    print("\nGenerating summary comparison plot...")
                    create_summary_comparison_plot(
                        fold_metrics,
                        baseline_metrics,
                        results_dir,
                        config # Pass final config
                    )
                    print("Summary comparison plot generated.")
                elif fold_metrics:
                    print("\nGenerating summary plot (Transformer only)...")
                    # If only transformer metrics are available, you might call a modified
                    # version of create_summary_comparison_plot or a dedicated function.
                    # For now, just note it wasn't created due to missing baselines.
                    create_summary_comparison_plot(fold_metrics, {}, results_dir, config) # Pass empty baseline dict
                    print("Summary plot generated (Transformer only).")
                else:
                     print("\nSkipping summary comparison plot: No metrics available.")
            except Exception as summary_plot_error:
                print(f"Error creating summary comparison plot: {summary_plot_error}")
        else:
            # Transfer learning is not enabled but run_only_transfer was requested
            print("\n=== Error: Transfer learning is not enabled but run_only_transfer_learning is set to True ===")
            print("Please enable transfer_learning or disable run_only_transfer_learning")

    except Exception as e:
        print(f"\n=== Main Execution Error: {str(e)} ===")
        import traceback
        traceback.print_exc()

    finally:
        # Stop GPU monitoring
        stop_gpu_memory_monitor(monitor_stop_flag)
        # Final cleanup
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        print("\n=== Execution Complete ===")
        if hasattr(config, 'enable_transfer_learning') and config.enable_transfer_learning:
            print(f"Transfer learning completed for {getattr(config, 'target_dataset_name', 'target')} dataset")
        else:
            print("Standard training workflow completed")


if __name__ == "__main__":
    # Configure PyTorch to optimize for performance
    torch.backends.cudnn.benchmark = True  # Enable cudnn auto-tuner

    try:
        # Run main function with error handling
        main()
    except Exception as e:
        print(f"\n--- Error in main execution ---")
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Details: {e}")
        import traceback
        traceback.print_exc()
        print("-----------------")

    # Final cleanup
    finally:
        gc.collect() # Ensure garbage collection runs
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
                print("\n=== Final GPU Memory Usage ===")
                gpu_info = get_gpu_memory_info()
                if isinstance(gpu_info, dict) and 'error' not in gpu_info:
                    for gpu_id, info in gpu_info.items():
                        if isinstance(info, dict) and 'error' not in info:
                            print(f"{gpu_id.upper()}: Used {info['allocated_memory_GB']:.2f}/{info['total_memory_GB']:.2f} GB "\
                                  f"({info['utilization_pct']:.1f}%)")
                elif isinstance(gpu_info, dict) and 'error' in gpu_info:
                    print(f"Could not get GPU info: {gpu_info['error']}")
                else:
                    print("Could not retrieve final GPU info.")
                print("===============================")
            except Exception as gpu_info_err:
                print(f"Error during final GPU memory check: {gpu_info_err}")
        print(f"[ {generate_timestamp()} ] Finnshing main execution...")
        print("\nExecution finished.")

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
<ipython-input-12-63e08774ace2>:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
<ipython-input-11-e594f68259eb>:292: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [ ]:
# # Once your main code is done
# from google.colab import runtime
# # Disconnect and delete the runtime
# runtime.unassign()

## (saltar para o fim)